# Digital Avionics Operations in Mission Simulation: an AI-assisted MBSE Modeling Approach
> **Pipeline of a multi-agent system for transforming a military mission modelled according to the Mission Engineering process into SysML v2, using the CoSMA framework.**  
> Contains 6 agents:

*   P1: PreprocessingAgent;
*   P2: MissionAgent;
*   P3: TemplateGeneratorClass;
*   P4: RefinementAgent;
*   P5: ValidatorAgent;
*   P6: FixerAgent.

The framework consolidates the agents into a single system orchestrated by a MasterOrchestrator via LangGraph.
  
> Phased construction: Blocks A through F:

*   A: Setup, PipelineState, helpers and RAG;
*   B: P1 (PreprocessingAgent) and P2 (MissionAgent);
*   C: P3 (TemplateGeneratorClass) and P4 (RefinementAgent);
*   D: P5 (ValidatorAgent) and P6 (FixerAgent);
*   E: LangGraph StateGraph
*   F: End-to-end execution and final report
---

---

# Block A: Setup, PipelineState, helpers and RAG base

### Cell 1 — Dependency installation

In [ ]:
# ============================================================
# Master Orchestrator (LangGraph StateGraph)
# Cell 1 — Dependency installation
# ============================================================
# All dependencies for P1–P6 and LangGraph for orchestration.
# langgraph-checkpoint: state persistence across nodes.
# ============================================================
# Using anthropic as the development kit for the agents

!pip install anthropic pymupdf jinja2 pydantic langgraph chromadb json_repair -q

# Verification of critical versions
import anthropic, jinja2, pydantic
print(f'anthropic  {anthropic.__version__}')
print(f'jinja2     {jinja2.__version__}')
print(f'pydantic   {pydantic.__version__}')

try:
    import langgraph
    print(f'langgraph  {langgraph.__version__}')
except AttributeError:
    print('langgraph  installed (no __version__)')
try:
    from importlib.metadata import version as pkg_version
    print(f'json_repair {pkg_version("json_repair")}')
except Exception:
    print('json_repair installed (no version)')

print('\nDependencies installed. ✅')

### Cell 2 — API Key, Google Drive and Global Constants

In [ ]:
# ============================================================
# Master Orchestrator
# Cell 2 — API Key, Google Drive saving and global constants
# ============================================================

import os, sys, json, time, re, math, copy
from pathlib import Path
from datetime import datetime
from typing import Optional, Any
from enum import Enum

import anthropic
from google.colab import userdata, drive, files

# ── 1. API Key ─────────────────────────────────────────────
try:
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    print('API key loaded successfully. ✅')
except Exception:
    print('ERROR: Secret ANTHROPIC_API_KEY not found.')
    print('Go to Secrets (lock icon) and add ANTHROPIC_API_KEY.')
    sys.exit(1)

# ── 2. Google Drive ────────────────────────────────────────
drive.mount('/content/drive')

# ── 3. Global constants ────────────────────────────────────

# LLM model, in this work use: claude-haiku-4-5-20251001 or claude-sonnet-4-5
LLM_MODEL = 'claude-sonnet-4-5'

# Loop P5↔P6 — maximum number of iterations
MAX_ITERATIONS = 2

# Default max tokens per agent
MAX_TOKENS_P1 = 8_000
MAX_TOKENS_P2 = 35_000
MAX_TOKENS_P4 = 16_000
MAX_TOKENS_P5 = 16_000
MAX_TOKENS_P6 = 16_000

# ── 4. Paths ───────────────────────────────────────────────

# Project base on Drive
BASE_DIR = Path('/content/drive/MyDrive/COSME_SysMLv2')
BASE_DIR.mkdir(parents=True, exist_ok=True)

# Subdirectories per agent
PREPROCESSING_DIR = BASE_DIR / 'preprocessing'
MISSION_AGENT_DIR = BASE_DIR / 'mission_agent'
TEMPLATE_DIR      = BASE_DIR / 'template_generator'
REFINEMENT_DIR    = BASE_DIR / 'refinement_agent'
VALIDATOR_DIR     = BASE_DIR / 'validator_agent'
FIXER_DIR         = BASE_DIR / 'fixer_agent'
ORCHESTRATOR_DIR  = BASE_DIR / 'orchestrator'

for d in [PREPROCESSING_DIR, MISSION_AGENT_DIR, TEMPLATE_DIR,
          REFINEMENT_DIR, VALIDATOR_DIR, FIXER_DIR, ORCHESTRATOR_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Timestamp of this run (used as run_id across all artefacts)
RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

# ── 5. Client ──────────────────────────────────────────────

_api_key = os.environ.get('ANTHROPIC_API_KEY')
CLIENT = anthropic.Anthropic(api_key=_api_key)

# Quick connection test
_test = CLIENT.messages.create(
    model=LLM_MODEL,
    max_tokens=5,
    messages=[{'role': 'user', 'content': 'Say OK'}],
)
print(f'Connection to {LLM_MODEL}: {_test.content[0].text} ✅')
# ── 6. Summary ─────────────────────────────────────────────
print(f'\n{"="*60}')
print(f'Master Orchestrator — Setup')
print(f'{"="*60}')
print(f'Model:           {LLM_MODEL}')
print(f'Max iterations:  {MAX_ITERATIONS}')
print(f'Base dir:        {BASE_DIR}')
print(f'Run timestamp:   {RUN_TIMESTAMP}')
print(f'Orchestrator:    {ORCHESTRATOR_DIR}')
print(f'{"="*60}')


### Cell 3 — PipelineState (TypedDict) and shared Helpers

In [6]:
# ============================================================
# Master Orchestrator
# Cell 3 — PipelineState (TypedDict) and shared Helpers
# ============================================================
# PipelineState is the central contract of the LangGraph StateGraph.
# Each node receives State and returns a dict with updated fields.
# ============================================================

from typing import TypedDict


# ── 1. PipelineState ───────────────────────────────────────

class PipelineState(TypedDict):
    """Central pipeline state — contract between all LangGraph nodes.

    Fields are populated progressively:
      P1 → mission_markdown, layer_markdowns
      P2 → mission_json
      P3 → skeletons
      P4 → refined_packages
      P5 → validation_report, validation_status
      P6 → fix_report (refined_packages is updated in-place)
    """
    # ── Inputs (populated before START) ──
    pdf_bytes: bytes
    pdf_filename: str

    # ── P1 outputs ──
    mission_markdown: str           # consolidated markdown (~110k chars)
    layer_markdowns: dict           # {layer_name: str} — 5 individual layers

    # ── P2 outputs ──
    mission_json: dict              # consolidated JSON (11 sections, ~214k bytes)

    # ── P3 outputs ──
    skeletons: dict                 # {pkg_name: sysml_str} — 21 skeletons

    # ── P4 outputs ──
    refined_packages: dict          # {pkg_name: sysml_str} — 21 refined packages

    # ── P5 outputs ──
    validation_report: dict         # complete report (issues, status, metrics)
    resume_from: str

    # ── P6 outputs ──
    fix_report: dict                # fix report

    # ── P5↔P6 loop control ──
    iteration: int                  # loop counter (starts at 0)
    max_iterations: int             # MAX_ITERATIONS (2)
    pipeline_status: str            # RUNNING | PASS | CONVERGED | MAX_ITERATIONS | ERROR

    # ── Observability ──
    cost_log: list                  # [{agent, cost_usd, time_s, tokens_in, tokens_out}]
    errors: list                    # [{agent, error, timestamp}]

    # ── Paths ──
    output_base_dir: str            # str(BASE_DIR)
    run_timestamp: str              # RUN_TIMESTAMP


# ── 2. Shared helpers ──────────────────────────────────────

def call_llm(system_prompt: str, user_prompt: str,
             max_tokens: int = 8000,
             client: anthropic.Anthropic = None) -> str:
    """Calls the LLM with exponential retry for rate limiting.

    Used by P1 and P2 (which are procedural).
    P4/P5/P6 use their own internal streaming methods.
    """
    c = client or CLIENT
    wait = 5
    for attempt in range(4):
        try:
            response = c.messages.create(
                model=LLM_MODEL,
                max_tokens=max_tokens,
                system=system_prompt,
                messages=[{'role': 'user', 'content': user_prompt}],
            )
            return response.content[0].text
        except anthropic.RateLimitError:
            print(f'  Rate limit. Waiting {wait}s...')
            time.sleep(wait)
            wait *= 2
        except Exception as e:
            raise RuntimeError(f'API error: {e}') from e
    raise RuntimeError('Failed after 4 attempts.')


def call_llm_streaming(system_prompt: str, user_prompt: str,
                       max_tokens: int = 16000,
                       client: anthropic.Anthropic = None,
                       prior_messages: Optional[list] = None) -> str:
    """Calls the LLM with streaming — used by P4/P5/P6.

    Supports multi-turn via prior_messages for continuation.
    Returns the full accumulated text.
    """
    c = client or CLIENT
    messages = list(prior_messages or [])
    messages.append({'role': 'user', 'content': user_prompt})

    wait = 5
    for attempt in range(4):
        try:
            collected = []
            with c.messages.stream(
                model=LLM_MODEL,
                max_tokens=max_tokens,
                system=system_prompt,
                messages=messages,
            ) as stream:
                for text in stream.text_stream:
                    collected.append(text)
            return ''.join(collected)
        except anthropic.RateLimitError:
            print(f'  Rate limit. Waiting {wait}s...')
            time.sleep(wait)
            wait *= 2
        except Exception as e:
            raise RuntimeError(f'API error (streaming): {e}') from e
    raise RuntimeError('Failed after 4 attempts (streaming).')


def log_cost(state: dict, agent: str, cost_usd: float,
             time_s: float, tokens_in: int = 0, tokens_out: int = 0):
    """Logs an agent's cost and time in state.cost_log."""
    entry = {
        'agent': agent,
        'cost_usd': round(cost_usd, 4),
        'time_s': round(time_s, 1),
        'tokens_in': tokens_in,
        'tokens_out': tokens_out,
        'timestamp': datetime.now().isoformat(),
    }
    state.setdefault('cost_log', []).append(entry)
    print(f'  💰 {agent}: ${cost_usd:.4f} | {time_s:.1f}s | {tokens_in:,} in / {tokens_out:,} out')


def log_error(state: dict, agent: str, error: str):
    """Logs an error into state."""
    entry = {
        'agent': agent,
        'error': str(error),
        'timestamp': datetime.now().isoformat(),
    }
    state.setdefault('errors', []).append(entry)
    print(f'  ❌ {agent}: {error}')


# Estimated price per model (USD/1M tokens)
_MODEL_PRICING = {
    'claude-sonnet-4-5':          (3.0, 15.0),
    'claude-haiku-4-5-20251001':  (0.80, 4.0),
    'claude-opus-4-6':            (15.0, 75.0),
}

def estimate_cost(tokens_in: int, tokens_out: int, model: str = None) -> float:
    """Estimates cost in USD for the configured model.

    Uses LLM_MODEL as default. Prices updatable in _MODEL_PRICING.
    """
    m = model or LLM_MODEL
    price_in, price_out = _MODEL_PRICING.get(m, (3.0, 15.0))
    return (tokens_in * price_in + tokens_out * price_out) / 1_000_000


def write_packages_to_dir(packages: dict, target_dir: Path, label: str = '') -> Path:
    """Writes a dict of .sysml packages to a directory, organised by CoSMA-layer folder.

    Uses PACKAGE_TO_FOLDER (defined in Cell 17) if available.
    Fallback: saves flat if the map is not defined.

    Used to create/update refined_dir before re-validation (D59).
    Returns the Path of the created directory.
    """
    target_dir.mkdir(parents=True, exist_ok=True)
    folder_map = globals().get('PACKAGE_TO_FOLDER', {})
    for pkg_name, content in packages.items():
        folder = folder_map.get(pkg_name, '')
        if folder:
            pkg_dir = target_dir / folder
        else:
            pkg_dir = target_dir
        pkg_dir.mkdir(parents=True, exist_ok=True)
        filepath = pkg_dir / f'{pkg_name}.sysml'
        filepath.write_text(content, encoding='utf-8')
    if label:
        print(f'  📁 {label}: {len(packages)} packages → {target_dir.name}/')
    return target_dir


def make_initial_state(pdf_bytes: bytes, pdf_filename: str) -> dict:
    """Creates the initial pipeline state with safe defaults."""
    return {
        # Inputs
        'pdf_bytes': pdf_bytes,
        'pdf_filename': pdf_filename,
        # Outputs (populated by nodes)
        'mission_markdown': '',
        'layer_markdowns': {},
        'mission_json': {},
        'skeletons': {},
        'refined_packages': {},
        'validation_report': {},
        'fix_report': {},
        # Control
        'iteration': 0,
        'max_iterations': MAX_ITERATIONS,
        'pipeline_status': 'RUNNING',
        'resume_from': None,
        'cost_log': [],
        'errors': [],
        # Paths
        'output_base_dir': str(BASE_DIR),
        'run_timestamp': RUN_TIMESTAMP,
    }


# ── 3. Verification ────────────────────────────────────────
print('PipelineState defined. ✅')
print(f'Fields: {len(PipelineState.__annotations__)}')
print(f'Helpers: call_llm, call_llm_streaming, log_cost, log_error,')
print(f'         estimate_cost, write_packages_to_dir, make_initial_state')
print(f'\nBlock A complete. Ready for Block B (P1 + P2). ✅')


PipelineState defined. ✅
Fields: 17
Helpers: call_llm, call_llm_streaming, log_cost, log_error,
         estimate_cost, write_packages_to_dir, make_initial_state

Block A complete. Ready for Block B (P1 + P2). ✅


---
## Cell 3B: RAG Knowledge Base

This cell indexes the 28 Apollo 11 packages in ChromaDB and creates the retrieval functions.


In [ ]:
# ============================================================
# Master Orchestrator
# Cell 3B — RAG Knowledge Base (SysML v2 Ground Truth)
# ============================================================
# Indexes the 28 Apollo 11 packages in ChromaDB
# for semantic retrieval in agents P4, P5 and P6.
#
# RAG architecture:
#   Indexing: Apollo 11 .sysml → chunks → embeddings → ChromaDB
#   Retrieval: query → top-k most similar chunks
#   Generation: prompt + retrieved chunks → LLM → output
#
# Stack: ChromaDB (in-memory) + all-MiniLM-L6-v2 (local)
# Cost: $0.00 (local embeddings, no API calls)
#
# Threshold τ added to retrieval (confidence threshold).
# Chunks with L2 distance above τ are discarded to
# avoid injecting irrelevant examples into the prompt:
# in this study, with a specific ground truth (entirely relevant examples),
#  τ=99.0 was set, permissive to all chunks
# ============================================================

# chromadb was already installed in Cell 1

import chromadb
from pathlib import Path


# ── 1. ChromaDB configuration ───────────────────────────────

# In-memory client (no server, for Colab)
_chroma_client = chromadb.Client()

# Collection for Apollo 11 packages
# Uses all-MiniLM-L6-v2 automatically (because of the free local embedding)
APOLLO_COLLECTION = _chroma_client.get_or_create_collection(
    name='apollo11_sysml_v2',
    metadata={'description': 'Apollo 11 SysML v2 ground truth packages (Helle & Schramm, 2026)'}
)

print(f'ChromaDB initialised (in-memory). ✅')


# ── 2. Indexing of Apollo 11 packages ───────────────────────

# Path of Apollo 11 packages on Drive
# IMPORTANT FOR THE USER! ADJUST this path if the packages are in a different location!
APOLLO_GT_DIR = BASE_DIR / 'apollo11_ground_truth'

# Mapping of package → CoSMA layer (for filtering metadata)
PACKAGE_TO_LAYER = {
    'StakeholderPackage': 'purpose',
    'StakeholderNeedsPackage': 'purpose',
    'CapabilitiesPackage': 'purpose',
    'MissionPackage': 'purpose',
    'MissionPhasesPackage': 'purpose',
    'MissionSpecificationPackage': 'purpose',
    'ContextPackage': 'purpose',
    'ProgramPackage': 'purpose',
    'OperationsPackage': 'operational',
    'FunctionsPackage': 'functional',
    'FunctionSpecificationPackage': 'functional',
    'FunctionalRequirementsPackage': 'requirements',
    'MissionRequirementsPackage': 'requirements',
    'TechnicalRequirementsPackage': 'requirements',
    'LogicalComponentsPackage': 'logical',
    'SystemPackage': 'technical',
    'SystemSpecificationPackage': 'technical',
    'TechnicalComponentsPackage': 'technical',
    'TechnicalIndividualsPackage': 'technical',
    'TechnicalPortsPackage': 'technical',
    'AstronautsPackage': 'technical',
    'AnalysisPackage': 'analysis',
    'CalculationsPackage': 'analysis',
    'CoSMAPackage': 'framework',
    'CoSMAQuantitiesAndUnitsPackage': 'framework',
    'CoSMAViewsPackage': 'framework',
    'Apollo11Model': 'root',
    'Apollo11MissionExecutionPackage': 'execution',
}


def _chunk_sysml_package(content: str, pkg_name: str,
                         max_chunk_chars: int = 3000) -> list[dict]:
    """Semantic chunking: splits SysML v2 packages by structural unit boundaries.

    Each chunk contains one or more COMPLETE structural units (never fragmented),
    with the package header (imports) prepended for context.

    Structural units by package type:
      - Requirements packages: requirement def
      - Component packages: part def
      - Operation/Function packages: action def
      - Ports packages: port def, interface def, item def
      - Analysis packages: analysis def, analysis
      - Calculations packages: calc def
      - Execution packages: timeslice
      - Phase packages: state def

    Principle: chunk boundaries align with SysML v2 grammar boundaries,
    ensuring each retrieved chunk is a semantically complete example.

    Reference: Domain-aware chunking (Gao et al., 2024, RAG Survey §Indexing).
    """
    if len(content) <= max_chunk_chars:
        return [{'text': content, 'chunk_id': f'{pkg_name}_full',
                 'chunk_type': 'full_package'}]

    # ── 1. Extract header (copyright, package declaration and imports) ──
    lines = content.split('\n')
    header_lines = []
    body_start = 0
    for i, line in enumerate(lines):
        stripped = line.strip()
        if (stripped.startswith('private import') or stripped.startswith('package ') or
            stripped.startswith('library package') or
            stripped.startswith('//') or stripped == '' or stripped == '{'):
            header_lines.append(line)
            body_start = i + 1
        else:
            break
    header = '\n'.join(header_lines)
    body = '\n'.join(lines[body_start:])

    # ── 2. Determining structural-unit pattern for each package ──
    STRUCTURAL_PATTERNS = {
        # Requirements: one chunk per requirement def
        'MissionRequirementsPackage': r'^\trequirement\s+def\s+',
        'FunctionalRequirementsPackage': r'^\trequirement\s+def\s+',
        'TechnicalRequirementsPackage': r'^\trequirement\s+def\s+',
        'StakeholderNeedsPackage': r'^\trequirement\s+def\s+',
        # Components: one chunk per part def
        'CapabilitiesPackage': r'^\t(?:abstract\s+)?part\s+def\s+',
        'TechnicalComponentsPackage': r'^\t(?:abstract\s+)?part\s+def\s+',
        'LogicalComponentsPackage': r'^\t(?:abstract\s+)?part\s+def\s+',
        'SystemPackage': r'^\t(?:abstract\s+)?part\s+def\s+',
        'ContextPackage': r'^\tpart\s+def\s+',
        'ProgramPackage': r'^\tpart\s+def\s+',
        'StakeholderPackage': r'^\tpart\s+(?:def\s+)?',
        # Operations/Functions: one chunk per action def
        'OperationsPackage': r'^\t(?:abstract\s+)?action\s+def\s+',
        'FunctionsPackage': r'^\t(?:abstract\s+)?(?:action\s+def|item\s+def)\s+',
        # Ports: port def, interface def, item def
        'TechnicalPortsPackage': r'^\t(?:port|interface|item)\s+def\s+',
        # Analysis: analysis def or analysis instance
        'AnalysisPackage': r'^\tanalysis\s+',
        # Calculations: calc def
        'CalculationsPackage': r'^\tcalc\s+def\s+',
        # Phases: state def
        'MissionPhasesPackage': r'^\tstate\s+def\s+',
    }

    pattern_str = STRUCTURAL_PATTERNS.get(pkg_name)

    # Handle dynamic execution package names (e.g., Apollo11MissionExecutionPackage, MissionExecutionPackage)
    if not pattern_str and 'Execution' in pkg_name:
        pattern_str = r'(?:^\t\t(?:then\s+)?timeslice\s+|^\tindividual\s+part\s+)'

    # Handle dynamic model/root packages
    if not pattern_str and pkg_name.endswith('Model'):
        return [{'text': content, 'chunk_id': f'{pkg_name}_full',
                 'chunk_type': 'full_package'}]

    # Fallback: generic top-level definitions
    if not pattern_str:
        pattern_str = r'^\t(?:abstract\s+)?(?:part|action|state|requirement|analysis|item|port|interface|calc|enum|individual)\s+'

    pattern = re.compile(pattern_str, re.MULTILINE)
    splits = list(pattern.finditer(body))

    if not splits:
        # Fallback: size-based chunking (preserves header)
        chunks = []
        usable = max_chunk_chars - len(header) - 100
        for i in range(0, len(body), max(usable, 500)):
            chunk_text = header + '\n\n' + body[i:i + usable]
            chunks.append({'text': chunk_text,
                          'chunk_id': f'{pkg_name}_size{len(chunks)}',
                          'chunk_type': 'size_split'})
        return chunks

    # ── 3. Extract individual structural blocks ──
    blocks = []
    # Include any content before the first match as preamble (section comments, etc.)
    if splits[0].start() > 0:
        preamble = body[:splits[0].start()].rstrip()
        if preamble:
            header = header + '\n' + preamble

    for i, match in enumerate(splits):
        end = splits[i + 1].start() if i + 1 < len(splits) else len(body)
        block = body[match.start():end].rstrip()
        blocks.append(block)

    # ── 4. Group small blocks; keep large blocks as individual chunks ──
    # Large blocks (>= max_chunk_chars/3) get their own chunk.
    # Small blocks are grouped with neighbors up to max_chunk_chars.
    chunks = []
    current_chunk = header + '\n'
    current_unit_count = 0

    for block in blocks:
        block_with_newline = block + '\n\n'
        would_exceed = len(current_chunk) + len(block_with_newline) > max_chunk_chars
        has_content = current_unit_count > 0

        if would_exceed and has_content:
            # Flush current chunk
            chunks.append({
                'text': current_chunk.rstrip(),
                'chunk_id': f'{pkg_name}_struct{len(chunks)}',
                'chunk_type': 'structural_unit',
            })
            current_chunk = header + '\n'
            current_unit_count = 0

        current_chunk += block_with_newline
        current_unit_count += 1

    # Flush remaining
    if current_unit_count > 0:
        chunks.append({
            'text': current_chunk.rstrip(),
            'chunk_id': f'{pkg_name}_struct{len(chunks)}',
            'chunk_type': 'structural_unit',
        })

    return chunks if chunks else [{'text': content, 'chunk_id': f'{pkg_name}_full',
                                    'chunk_type': 'full_package'}]


def index_apollo11_packages(gt_dir: Path = None) -> int:
    """Indexes all Apollo 11 packages into ChromaDB.

    Returns the number of indexed chunks.
    If the packages were already indexed, does not re-index.
    """
    gt_dir = gt_dir or APOLLO_GT_DIR

    # Check whether already indexed
    if APOLLO_COLLECTION.count() > 0:
        print(f'  Apollo 11 already indexed: {APOLLO_COLLECTION.count()} chunks. Skipping.')
        return APOLLO_COLLECTION.count()

    if not gt_dir.exists():
        print(f'  ⚠ Directory not found: {gt_dir}')
        print(f'  Copy the 28 Apollo 11 .sysml files to this directory.')
        print(f'  RAG will run without ground truth (empty retrieval).')
        return 0

    sysml_files = sorted(gt_dir.glob('*.sysml'))
    if not sysml_files:
        print(f'  ⚠ No .sysml found in {gt_dir}')
        return 0

    all_documents = []
    all_ids = []
    all_metadatas = []

    for filepath in sysml_files:
        pkg_name = filepath.stem
        content = filepath.read_text(encoding='utf-8')
        layer = PACKAGE_TO_LAYER.get(pkg_name, 'unknown')

        chunks = _chunk_sysml_package(content, pkg_name)

        for chunk in chunks:
            all_documents.append(chunk['text'])
            all_ids.append(chunk['chunk_id'])
            all_metadatas.append({
                'package_name': pkg_name,
                'cosma_layer': layer,
                'chunk_type': chunk['chunk_type'],
                'char_count': len(chunk['text']),
            })

    # Index in batch
    APOLLO_COLLECTION.add(
        documents=all_documents,
        ids=all_ids,
        metadatas=all_metadatas,
    )

    print(f'  Apollo 11 indexed: {len(all_documents)} chunks from {len(sysml_files)} packages ✅')

    # Summary by layer
    from collections import Counter
    layer_counts = Counter(m['cosma_layer'] for m in all_metadatas)
    for layer, count in sorted(layer_counts.items()):
        print(f'    {layer:15s}: {count} chunks')

    return len(all_documents)


# ── 3. Retrieval functions ──────────────────────────────────

# Threshold τ — retrieval confidence threshold (L2 distance).
# Chunks with distance above this value are discarded from the RAG context.
# Value 1.2: conservative for all-MiniLM-L6-v2 (384-dim).
# Value 99.0: permissive for all-MiniLM-L6-v2 (384-dim).
# SET the desired value, from 1.2 to 99.0.
# As stated above, this study used τ=99.0.
RAG_SIMILARITY_THRESHOLD = 99.0


def retrieve_sysml_examples(query: str, top_k: int = 2,
                            layer_filter: str = None,
                            pkg_name_filter: str = None,
                            threshold: float = None) -> list[dict]:
    """Retrieves SysML v2 chunks most similar to the query.

    Args:
        query: Search text (e.g., package name, task description)
        top_k: Number of chunks to retrieve
        layer_filter: Filter by CoSMA layer (e.g., 'technical')
        pkg_name_filter: Filter by a specific package name
        threshold: Maximum acceptable L2 distance (default: RAG_SIMILARITY_THRESHOLD).
                   Chunks above this threshold are discarded to avoid
                   injecting irrelevant examples into the P4 prompt.

    Returns:
        List of dicts with keys: text, package_name, cosma_layer, distance
        (only chunks with distance <= threshold)

    Reference: RAG survey (Gao et al., 2024), stage — Retrieval.
    """
    if APOLLO_COLLECTION.count() == 0:
        return []

    # Use global threshold if not specified
    if threshold is None:
        threshold = RAG_SIMILARITY_THRESHOLD

    # Build filter
    where_filter = None
    if layer_filter and pkg_name_filter:
        where_filter = {'$and': [
            {'cosma_layer': layer_filter},
            {'package_name': pkg_name_filter}
        ]}
    elif layer_filter:
        where_filter = {'cosma_layer': layer_filter}
    elif pkg_name_filter:
        where_filter = {'package_name': pkg_name_filter}

    try:
        results = APOLLO_COLLECTION.query(
            query_texts=[query],
            n_results=min(top_k, APOLLO_COLLECTION.count()),
            where=where_filter,
        )
    except Exception:
        # Fallback without filter if the filter does not match anything
        results = APOLLO_COLLECTION.query(
            query_texts=[query],
            n_results=min(top_k, APOLLO_COLLECTION.count()),
        )

    if not results or not results['documents'] or not results['documents'][0]:
        return []

    retrieved = []
    for i, doc in enumerate(results['documents'][0]):
        meta = results['metadatas'][0][i] if results['metadatas'] else {}
        dist = results['distances'][0][i] if results['distances'] else 0.0
        retrieved.append({
            'text': doc,
            'package_name': meta.get('package_name', ''),
            'cosma_layer': meta.get('cosma_layer', ''),
            'distance': dist,
        })

    # ── Threshold τ: discard chunks with distance above the threshold ──
    # If all chunks are above τ, returns an empty list →
    # P4 generates without RAG examples (safety net for queries with
    # no analogue in the Apollo 11 ground truth).
    n_before = len(retrieved)
    retrieved = [r for r in retrieved if r['distance'] <= threshold]
    n_filtered = n_before - len(retrieved)
    if n_filtered > 0:
        print(f'  ⚠ RAG threshold τ={threshold}: {n_filtered}/{n_before} '
              f'chunk(s) discarded for distance above the threshold.')

    return retrieved


def format_rag_context(retrieved: list[dict], label: str = 'GROUND TRUTH REFERENCE') -> str:
    """Formats retrieved chunks as context for injection into the prompt.

    Returns formatted string or '' if empty.
    """
    if not retrieved:
        return ''

    sections = [f'## {label} (Apollo 11 SysML v2 — Helle & Schramm, 2026)']
    sections.append('Use these examples as syntactic and structural reference.\n')

    for i, r in enumerate(retrieved):
        sections.append(f'### Example {i+1}: {r["package_name"]} ({r["cosma_layer"]} layer)')
        # Truncate if too long (max ~2000 chars per chunk in the prompt)
        text = r['text']
        if len(text) > 2000:
            text = text[:2000] + '\n// ... (truncated for context)'
        sections.append(f'```sysml\n{text}\n```\n')

    return '\n'.join(sections)

def index_extra_documents(docs: dict[str, str], layer: str = 'framework'):
    """Indexes additional SysML v2 content (e.g., adapted CoSMA packages) into ChromaDB.

    Args:
        docs: dict mapping package_name → SysML v2 content string
        layer: CoSMA layer label for metadata

    Skips documents whose chunk IDs already exist in the collection.
    """
    if not docs:
        return 0
    added = 0
    existing_ids = set(APOLLO_COLLECTION.get()['ids']) if APOLLO_COLLECTION.count() > 0 else set()
    for pkg_name, content in docs.items():
        chunks = _chunk_sysml_package(content, pkg_name)
        for chunk in chunks:
            if chunk['chunk_id'] in existing_ids:
                continue
            APOLLO_COLLECTION.add(
                documents=[chunk['text']],
                ids=[chunk['chunk_id']],
                metadatas=[{
                    'package_name': pkg_name,
                    'cosma_layer': layer,
                    'chunk_type': chunk['chunk_type'],
                    'char_count': len(chunk['text']),
                }],
            )
            added += 1
    if added:
        print(f'  RAG: +{added} extra chunks indexed ({", ".join(docs.keys())})')
    return added

# ── 4. Run indexing ─────────────────────────────────────────

# Force re-indexing with new semantic chunks (run once)
if APOLLO_COLLECTION.count() > 0:
    _chroma_client.delete_collection('apollo11_sysml_v2')
    APOLLO_COLLECTION = _chroma_client.get_or_create_collection(
        name='apollo11_sysml_v2',
        metadata={'description': 'Apollo 11 SysML v2 ground truth packages (Helle & Schramm, 2026)'}
    )
    print('  ChromaDB collection reset for semantic re-indexing. ✅')

n_chunks = index_apollo11_packages()

print(f'\n{"="*60}')
print(f'RAG Knowledge Base — Setup complete')
print(f'{"="*60}')
print(f'  ChromaDB: in-memory (all-MiniLM-L6-v2)')
print(f'  Collection: apollo11_sysml_v2')
print(f'  Chunks indexed: {n_chunks}')
print(f'  Threshold τ: {RAG_SIMILARITY_THRESHOLD} (maximum L2 distance)')
print(f'  Embedding cost: $0.00 (local)')
print(f'  Available functions:')
print(f'    retrieve_sysml_examples(query, top_k, layer_filter, pkg_name_filter, threshold)')
print(f'    format_rag_context(retrieved, label)')
print(f'{"="*60}')

---

## Block B — P1 (PreprocessingAgent) and P2 (MissionAgent)

### Cell 4 — Pydantic Schemas P2 (CoSMA Metamodel)

In [ ]:
# ============================================================
# Master Orchestrator
# Cell 4 — Pydantic Schemas P2 (CoSMA Metamodel)
# ============================================================

from __future__ import annotations
from pydantic import BaseModel, Field
from typing import Optional
from enum import Enum


# ── ENUMS ──────────────────────────────────────────────────

class InfluenceLevel(str, Enum):
    high = 'high'
    medium = 'medium'
    low = 'low'

class InfluenceKind(str, Enum):
    direct = 'direct'
    indirect = 'indirect'

class F2T2EAStep(str, Enum):
    find = 'Find'
    fix = 'Fix'
    track = 'Track'
    target = 'Target'
    engage = 'Engage'
    assess = 'Assess'
    not_applicable = 'N/A'

class Configuration(str, Enum):
    """Represents mission approaches per MEG §5.3:
    baseline = initial/reference approach
    modernized = alternative approach (primary)
    both = common to all approaches
    """
    both = 'both'
    baseline = 'baseline'
    modernized = 'modernized'

class PortDirection(str, Enum):
    in_ = 'in'
    out = 'out'
    inout = 'inout'

class Provenance(str, Enum):
    """M3 — Marks whether a value was explicitly stated in the source
    document or inferred/estimated by the pipeline."""
    explicit = 'explicit'
    inferred = 'inferred'


# ── PURPOSE LAYER — Stakeholders, Concerns, Needs ──────────

class Concern(BaseModel):
    text: str = Field(description='Concern description in English')

class StakeholderNeed(BaseModel):
    id: str = Field(description='Unique ID, e.g. SHN-001')
    name: str = Field(description='CamelCase name, e.g. PreserveNationalSovereignty')
    text: str = Field(description='Full requirement text in English')
    rationale: str = Field(default='', description='Rationale justifying this need — populates @Rationale metadata in SysML v2')
    stakeholder_ids: list[str] = Field(description='IDs of stakeholders owning this need')
    source: str = Field(description='Source section in PDF, e.g. §2.1.1.1')

class Stakeholder(BaseModel):
    id: str = Field(description='Unique ID, e.g. SH-001')
    name: str = Field(description='CamelCase name for SysML def')
    full_name: str = Field(description='Full name with original-language acronym')
    role: str = Field(description='Role description in English')
    type_name: str = Field(description='Stakeholder subtype')
    influence_level: InfluenceLevel
    influence_kind: InfluenceKind
    concerns: list[Concern] = Field(description='List of concerns')
    need_ids: list[str] = Field(description='IDs of StakeholderNeeds')
    source: str

class StakeholderData(BaseModel):
    stakeholders: list[Stakeholder]
    stakeholder_needs: list[StakeholderNeed]


# ── PURPOSE LAYER — Capabilities ───────────────────────────

class Capability(BaseModel):
    id: str = Field(description='Unique ID, e.g. CAP-001')
    name: str = Field(description='CamelCase name')
    description: str
    supports_goal_ids: list[str] = Field(description='GOAL IDs this capability supports')
    source: str

class CapabilityData(BaseModel):
    capabilities: list[Capability]


# ── PURPOSE + OPERATIONAL — Mission, Goals, Phases ─────────

class Goal(BaseModel):
    id: str = Field(description='Unique ID, e.g. GOAL-001')
    name: str = Field(description='CamelCase name')
    description: str
    refines_shn_ids: list[str] = Field(description='SHN IDs this goal refines')
    source: str

class MissionPhase(BaseModel):
    id: str = Field(description='Unique ID, e.g. PH-001')
    name: str = Field(description='CamelCase state name')
    display_name: str = Field(description='Human-readable name')
    entry_trigger: str
    do_actions: str
    exit_trigger: str
    next_phase_ids: list[str] = Field(description='Next phase ID(s)')
    operation_ids: list[str] = Field(description='OP IDs executed in this phase')
    source: str

class RelatedMission(BaseModel):
    name: str = Field(description='CamelCase mission/operation name')
    description: str = Field(description='One-sentence description of the mission/operation')
    is_current: bool = Field(default=False, description='True if this is the mission being modeled')

class Mission(BaseModel):
    name: str = Field(description='CamelCase mission name')
    display_name: str
    description: str
    program_name: str
    program_description: str = Field(default='', description='Description of the overarching program/campaign')
    related_missions: list[RelatedMission] = Field(default_factory=list,
        description='Other missions/operations within the same program, e.g. prior exercises, parallel operations, future phases')
    source: str

class MissionData(BaseModel):
    mission: Mission
    goals: list[Goal]
    phases: list[MissionPhase]


# ── PURPOSE — Context ──────

class GeographicPoint(BaseModel):
    name: str
    latitude: str
    longitude: str
    notes: str = ''

class ExternalSystem(BaseModel):
    id: str = Field(description='Unique ID, e.g. EXT-001')
    name: str
    description: str
    source: str

class NamedWaypointGroup(BaseModel):
    group_name: str = Field(
        description='Group identifier, e.g. "main_base", "patrol_points", "target_areas", "supply_routes"')
    description: str = Field(default='')
    points: list[GeographicPoint] = Field(default_factory=list)

class Context(BaseModel):
    name: str = Field(default='MissionContext',
                      description='CamelCase context name — derived from mission name')
    scenario_purpose: str
    epoch: str
    area_of_operations: str
    geopolitical_context: str = Field(default='',
                                     description='Geopolitical context if applicable')
    geographic_references: list[NamedWaypointGroup] = Field(default_factory=list,
        description='Named groups of geographic points — mission defines which types exist')
    operational_boundaries: list[str] = Field(default_factory=list,
        description='Boundary descriptions with values and units, e.g. "80 NM penetration limit"')
    external_systems: list[ExternalSystem] = Field(default_factory=list)
    source: str = ''

class ContextData(BaseModel):
    context: Context


# ── OPERATIONAL LAYER — Operations, Sequences, ROE ─────────

class Operation(BaseModel):
    id: str = Field(description='Unique ID, e.g. OP-001')
    name: str = Field(description='CamelCase name')
    description: str
    phase_ids: list[str]
    performer: str
    f2t2ea_step: F2T2EAStep = Field(default=F2T2EAStep.not_applicable,
        description='Kill chain step if applicable')
    configuration: Configuration = Field(default=Configuration.both)
    source: str

class EngagementStep(BaseModel):
    step_number: int
    name: str
    description: str
    operation_ids: list[str]

class EngagementSequence(BaseModel):
    configuration: Configuration
    steps: list[EngagementStep]
    key_constraints: list[str]
    source: str

class ROERule(BaseModel):
    rule_number: int
    title: str
    description: str

class ROE(BaseModel):
    is_simplified: bool = Field(default=True)
    rationale: str
    rules: list[ROERule]
    abort_criteria: list[str]
    source: str

class OperationalConstraint(BaseModel):
    category: str
    text: str
    source: str

class OperationData(BaseModel):
    operations: list[Operation]
    engagement_sequences: list[EngagementSequence] = Field(default_factory=list)
    roe: Optional[ROE] = Field(default=None,
        description='Rules of Engagement if stated in source')
    constraints: list[OperationalConstraint] = Field(default_factory=list)


# ── FUNCTIONAL LAYER — Functions, Decomposition ────────────

class ItemDef(BaseModel):
    name: str = Field(description='CamelCase name for SysML item def, e.g. MissionStatus, TargetTrack')
    description: str = Field(description='One-sentence description of what this data type represents')

class FunctionIO(BaseModel):
    name: str
    type_desc: str
    direction: str

class Function(BaseModel):
    id: str = Field(description='Unique ID, e.g. FN-001')
    name: str
    description: str
    inputs: list[FunctionIO] = Field(default_factory=list)
    outputs: list[FunctionIO] = Field(default_factory=list)
    parent_id: Optional[str] = Field(default=None)
    refines_op_ids: list[str] = Field(default_factory=list)
    f2t2ea_step: F2T2EAStep = Field(default=F2T2EAStep.not_applicable)
    configuration: Configuration = Field(default=Configuration.both)
    source: str

class FunctionalRequirement(BaseModel):
    id: str = Field(description='Unique ID, e.g. FR-001')
    text: str
    function_ids: list[str]
    traces_to: list[str]
    source: str

class FunctionData(BaseModel):
    item_defs: list[ItemDef] = Field(default_factory=list,
        description='Domain-specific item definitions used to type function inputs/outputs (5-10 concise types)')
    functions: list[Function]
    functional_requirements: list[FunctionalRequirement]

# ── LOGICAL LAYER — Components, Allocation, SoS ────────────

class LogicalInterface(BaseModel):
    from_lc_id: str
    to_lc_id: str
    information_flow: str
    flow_type: str
    description: str

class LogicalComponent(BaseModel):
    id: str = Field(description='Unique ID, e.g. LC-001')
    name: str
    description: str
    performs_fn_ids: list[str]
    owner_stakeholder_ids: list[str]
    is_threat: bool = Field(default=False)
    source: str

class SoSConstituent(BaseModel):
    lc_id: str
    role_in_sos: str

class SystemOfSystems(BaseModel):
    name: str = Field(default='MissionSystem',
                      description='CamelCase — derived from mission name')
    description: str
    constituents: list[SoSConstituent]
    threat_entities: list[SoSConstituent] = Field(default_factory=list)
    governance_model: str
    source: str

class LogicalData(BaseModel):
    logical_components: list[LogicalComponent]
    interfaces: list[LogicalInterface]
    sos: SystemOfSystems


# ── TECHNICAL LAYER — Components, Ports ───

class TechnicalSpec(BaseModel):
    name: str
    value: str
    unit: str
    source: str
    provenance: Provenance = Field(default=Provenance.explicit,
        description='M3: whether this value was explicitly stated or inferred')

class TechnicalComponent(BaseModel):
    id: str = Field(description='Unique ID, e.g. TC-001')
    name: str
    description: str
    realizes_lc_id: str
    configuration: Configuration
    specifications: list[TechnicalSpec] = Field(default_factory=list)
    source: str

class TechnicalPort(BaseModel):
    name: str
    owner_tc_id: str
    direction: PortDirection
    item_type: str
    connected_to: str
    port_def_name: str = Field(default='',
        description='Name of the PortDef this port is typed by, e.g. VoiceCommPort, WeaponFiringPort. Must match a name from port_defs.')
    notes: str = ''
    source: str

class PortFlowItemDef(BaseModel):
    name: str = Field(description='CamelCase name for item def, e.g. CommandSignal, TelemetryData, ElectricalPower')
    description: str = Field(description='What this data/energy/material flow represents')

class PortDefItem(BaseModel):
    direction: PortDirection = Field(description='in, out, or inout')
    name: str = Field(description='camelCase item name within the port, e.g. commands, telemetry, power')
    type_name: str = Field(description='Name of PortFlowItemDef this item is typed by, e.g. CommandSignal')

class PortDef(BaseModel):
    name: str = Field(description='CamelCase port def name, e.g. DatalinkPort, SensorDataPort')
    description: str
    items: list[PortDefItem] = Field(description='Directional items flowing through this port')
    source: str = ''

class InterfaceDef(BaseModel):
    name: str = Field(description='CamelCase, e.g. DatalinkInterface, SensorInterface')
    description: str
    end1_name: str = Field(description='First end role name, e.g. source, controller, supplier')
    end1_port_type: str = Field(description='PortDef name for first end')
    end1_conjugated: bool = Field(default=False, description='True if first end uses conjugated (~) port')
    end2_name: str = Field(description='Second end role name, e.g. target, controlled, consumer')
    end2_port_type: str = Field(description='PortDef name for second end')
    end2_conjugated: bool = Field(default=True, description='True if second end uses conjugated (~) port')
    source: str = ''

class TechnicalIndividual(BaseModel):
    id: str = Field(description='Unique ID, e.g. TI-001')
    name: str = Field(description='Display name for SysML quoted name, e.g. FARP-SBCZ, A29N-01')
    type_tc_id: str = Field(description='TC-ID of the component type this individual instantiates')
    description: str
    source: str = ''

class TechnicalData(BaseModel):
    technical_components: list[TechnicalComponent]
    ports: list[TechnicalPort]
    port_flow_item_defs: list[PortFlowItemDef] = Field(default_factory=list,
        description='Domain flow types used to type port items (5-10 concise types)')
    port_defs: list[PortDef] = Field(default_factory=list,
        description='Port type definitions with directional items')
    interface_defs: list[InterfaceDef] = Field(default_factory=list,
        description='Interface contracts defining how ports connect')
    technical_individuals: list[TechnicalIndividual] = Field(default_factory=list,
        description='Named individual instances of technical components (e.g., specific FARPs, specific aircraft)')


# ── REQUIREMENTS ───────────────────────────────────────────

class Requirement(BaseModel):
    id: str
    descriptive_name: str = Field(default='',
        description='CamelCase descriptive name, e.g. CrewReturnSafetyRequirement, UASDetectionLatencyRequirement. NOT generic like MR001Requirement.')
    text: str
    rationale: str = Field(default='',
        description='1-2 sentence justification explaining WHY this requirement exists — populates @Rationale in SysML v2')
    traces_to: list[str]
    source: str

class RequirementsData(BaseModel):
    mission_requirements: list[Requirement]
    functional_requirements: list[Requirement]
    technical_requirements: list[Requirement]


# ── MEASURES — MOSs, MOEs, MOPs ──

class FormulaVariable(BaseModel):
    """Variable used in a MOS/MOE/MOP formula (maps to calc def in/out)."""
    name: str = Field(description='Variable symbol, e.g. Nn, Nt, tef_i')
    description: str = Field(description='What this variable represents')
    role: str = Field(default='in', description='"in" for input, "out" for output/return')
    unit: str = Field(default='', description='Unit if applicable, e.g. s, km, dimensionless')

class MeasureOfSuccess(BaseModel):
    id: str
    name: str
    description: str
    unit: str
    higher_is_better: bool
    formula: str = Field(default='',
        description='Mathematical expression, e.g. "Nn / Nt"')
    variables: list[FormulaVariable] = Field(default_factory=list,
        description='Input/output variables of the formula')
    threshold: str = Field(default='TBD',
        description='Success criterion, e.g. ">= 0.80", "< 45s"')
    source: str

class MeasureOfEffectiveness(BaseModel):
    id: str
    name: str
    description: str
    unit: str
    formula: str = Field(default='',
        description='Mathematical expression, e.g. "1 - (1/(N*Tmax)) * sum(tef_i)"')
    variables: list[FormulaVariable] = Field(default_factory=list,
        description='Input/output variables of the formula')
    traces_to_mos_ids: list[str]
    source: str

class MeasureOfPerformance(BaseModel):
    id: str
    name: str
    description: str
    unit: str
    baseline_value: str
    modernized_value: str = Field(default='',
        description='Primary alternative value (backward compat with v1)')
    alternative_values: dict[str, str] = Field(default_factory=dict,
        description='For N>2 configs: {"alt_label": "value", ...}')
    provenance: Provenance = Field(default=Provenance.explicit,
        description='M3: whether values were explicitly stated or inferred')
    threshold: str = Field(default='TBD',
        description='Performance criterion per config, e.g. "[1, 10] s"')
    traces_to_moe_ids: list[str]
    source: str

class SimulationParameter(BaseModel):
    name: str
    description: str
    baseline_value: str
    modernized_value: str = Field(default='')
    alternative_values: dict[str, str] = Field(default_factory=dict)
    provenance: Provenance = Field(default=Provenance.explicit,
        description='M3: whether values were explicitly stated or inferred')
    unit: str
    source: str

class MeasuresData(BaseModel):
    measures_of_success: list[MeasureOfSuccess]
    measures_of_effectiveness: list[MeasureOfEffectiveness]
    measures_of_performance: list[MeasureOfPerformance]
    simulation_parameters: list[SimulationParameter]


# ── SPECIFICATION — satisfy/refine links ──────────────────

class SatisfyLink(BaseModel):
    requirement_id: str
    satisfied_by_id: str
    satisfied_by_type: str

class RefineLink(BaseModel):
    source_id: str
    target_id: str
    relationship: str

class SpecificationData(BaseModel):
    mission_specification_name: str = Field(default='MissionSpecification',
        description='CamelCase — derived from mission name by LLM')
    function_specification_name: str = Field(default='FunctionSpecification')
    system_specification_name: str = Field(default='SystemSpecification')
    satisfy_links: list[SatisfyLink]
    refine_links: list[RefineLink]


# ── CONSOLIDATED OUTPUT ────────────────────────────────────

class MissionAgentOutput(BaseModel):
    stakeholder_data: StakeholderData
    capability_data: CapabilityData
    mission_data: MissionData
    context_data: ContextData
    operation_data: OperationData
    function_data: FunctionData
    logical_data: LogicalData
    technical_data: TechnicalData
    requirements_data: RequirementsData
    measures_data: MeasuresData
    specification_data: SpecificationData


# ── Verification ───────────────────────────────────────────
_schemas = [
    StakeholderData, CapabilityData, MissionData, ContextData,
    OperationData, FunctionData, LogicalData, TechnicalData,
    RequirementsData, MeasuresData, SpecificationData, MissionAgentOutput,
]
print(f'Pydantic Schemas: {len(_schemas)} defined ✅')
for s in _schemas:
    print(f'  {s.__name__:30s} — {len(s.model_fields)} fields')

### Cell 5 — P1 Prompts + run_p1() | P2 Prompts + run_p2()

In [ ]:
# ============================================================
# Master Orchestrator
# Cell 5 — P1 (PreprocessingAgent) + P2 (MissionAgent)
# ============================================================
# Consolidates all procedural code of P1 and P2 into two wrapper
# functions: run_p1(state) → dict and run_p2(state) → dict.
# Both operate on PipelineState via LangGraph.
# ============================================================

import fitz  # PyMuPDF
import json as _json


# ================================================================
#                     P1 — PREPROCESSING AGENT
# ================================================================


# ── P1 System Prompt  ─────────────────

P1_SYSTEM_PROMPT = """You are a Senior Systems and Computer Engineer specialized in Mission Engineering
(U.S. DoD Mission Engineering Guide, 2023), the CoSMA framework (Helle & Schramm, 2026 — "Fly me to the Moon: Modeling Apollo 11 using SysML v2") and SysML v2 (Objet Management Group, 2025 - OMG Systems Modeling Language (SysML) Version 2.0: Part 1: Language Specification)

You are extracting structured information from a military mission case study
to populate a SysML v2 model organized by the 5 CoSMA layers:
Purpose → Operational → Functional → Logical → Technical.

EXTRACTION RULES:
1. Extract ONLY what is explicitly stated or directly derivable from the source text.
   NEVER invent data, actors, metrics, or requirements.
   If required information is absent, write exactly: "TBD — not stated in source"
2. Each extracted item MUST reference the source section where it was found
   (e.g., "Source: §2.1.1" or "Source: Section 3, paragraph 2").
3. Respond exclusively in English Markdown.
   Start directly with the first heading. No conversational preamble.
4. Use consistent ID prefixes:
   - SHN-XXX (stakeholder needs), CAP-XXX (capabilities), GOAL-XXX (goals)
   - MR-XXX (mission requirements), FR-XXX (functional requirements)
   - TR-XXX (technical requirements)
   - OP-XXX (operations), FN-XXX (functions), LC-XXX (logical components)
   - TC-XXX (technical components)
5. Maintain traceability: every lower-layer item must trace to at least one
   upper-layer item (e.g., Function traces to Operation, Operation traces to Phase).
6. For military-domain terms, preserve the original-language acronym in
   parentheses: e.g., "Aerospace Operations Command (COMAE)".
   Detect the source document's language automatically.
7. If the source document describes a kill chain or engagement sequence
   (e.g., F2T2EA: Find–Fix–Track–Target–Engage–Assess, or any other doctrinal
   kill chain), map operations and functions to its steps where applicable.
   If no kill chain is described, skip this mapping.
8. If the source document describes MULTIPLE system configurations
   (e.g., a baseline approach vs. one or more alternative/modernized approaches
   per MEG §5.3), identify and distinguish them throughout the extraction.
   Label them as: baseline, alternative_1, alternative_2, ..., or use the
   document's own labels. If only ONE configuration exists, note "single_config".
"""


# ── P1 prompts per CoSMA layer  ──────

def _p1_prompt_purpose(text):
    return f"""SOURCE TEXT (complete mission case study document):
---
{text}
---

Extract the CoSMA PURPOSE LAYER from the source text above.
This layer answers: "Why does this system exist?"

Scan the ENTIRE document for content related to: mission problem definition,
program identification, stakeholders and their concerns, mission engineering
purpose, mission context and scenario, and investigative questions.

Generate ALL of the following sections:

## 1. Program
- Name the overarching program or campaign this mission belongs to.
- Provide a 1-paragraph DESCRIPTION of the program's purpose and scope.
- List ALL related missions or operations mentioned in the document that
  belong to the same program/campaign (prior operations, exercises,
  parallel deployments, future phases). For each:
  Name | Description (1 sentence) | Is Current Mission? (yes/no)
- If no related missions are mentioned, list only the current mission.
- Identify the source section.

## 2. Stakeholders
For each stakeholder identified anywhere in the document:
- ID, Name (with original-language acronym if present), English description
- Role in the mission
- Influence level (high/medium/low) and kind (direct/indirect)
- Concerns (what they care about)
Table: | ID | Name | Role | Influence Level | Influence Kind | Concerns | Source |

## 3. Stakeholder Needs
Derive formal needs from stakeholder concerns.
Table: | ID (SHN-XXX) | Need Name | Text ("The mission shall...") | Stakeholder | Source |

## 4. Mission Goals
Refine stakeholder needs into mission goals.
Table: | ID (GOAL-XXX) | Goal Name | Description | Refines SHN-ID | Source |

## 5. Capabilities
What high-level abilities must the system possess to achieve its goals?
Table: | ID (CAP-XXX) | Capability Name | Description | Supports GOAL-ID | Source |

## 6. Mission Context
Extract ALL of the following that are present in the document:
- Scenario purpose and scope
- Epoch / time horizon
- Area of operations (coordinates, boundaries, geographic constraints)
- Geopolitical context (borders, treaties, agreements)
- Vignette scope (if applicable)
- Operating environment conditions:
  - Climate / weather (temperature ranges, visibility, precipitation)
  - Terrain type (jungle, desert, maritime, urban, mountainous)
  - Elevation / altitude ranges
  - Electromagnetic environment (if mentioned)
  Table: | Condition | Value/Description | Source |
If any of these are not stated, write "TBD — not stated in source".

## 7. External Systems and Participants
List all systems and actors that interact with the mission system of interest.
Classify each as: SystemOfInterest | ExternalSystem | ConstituentSystem | Threat
Table: | Name | Type | Role in Mission | Key Interactions | Source |

## 8. Mission Configurations (MEG §5.3)
If the document describes multiple system configurations (baseline vs. alternatives):
- Name each configuration and its key distinguishing features
- If only one configuration exists, state "Single configuration"
Table: | Config ID | Label | Key Features | Source |

## 9. Investigative Questions
List the research questions that drive the mission analysis (if stated).
Table: | IQ-ID | Question | Related MOS (if any) | Source |
"""


def _p1_prompt_operational(text, purpose_md):
    return f"""PURPOSE LAYER CONTEXT (already extracted — use for traceability):
---
{purpose_md}
---

SOURCE TEXT (complete mission case study document):
---
{text}
---

Extract the CoSMA OPERATIONAL LAYER from the source text.
This layer answers: "When do things happen and in what sequence?"

Scan the ENTIRE document for content related to: operational sequences,
mission phases, engagement procedures, rules of engagement, and
operational constraints.

Generate ALL of the following sections:

## 1. Mission Phases (State Machine)
Model the mission as a sequence of phases (state defs).
Table: | Phase ID (PH-XXX) | Phase Name | Entry Trigger | Do Actions | Exit Trigger | Next Phase | Estimated Duration | Source |
For "Estimated Duration": extract from document if stated (e.g., "2 hours",
"30 minutes"). If not stated, estimate based on context and mark as "~Xh (estimated)".

## 2. Operations
Define operational activities as CONCRETE SEQUENTIAL STEPS of the mission,
NOT as abstract doctrinal categories.

CRITICAL DISTINCTION — read carefully:
- WRONG approach: extracting generic capability categories like "Aerospace Control",
  "Airspace Policing", "Combat Sustainment". These are doctrinal LABELS, not operations.
- RIGHT approach: extracting the step-by-step actions that occur during the mission
  timeline. Each operation should describe a specific, observable action performed by
  a specific actor at a specific phase.

EXAMPLE of correct granularity (from Apollo 11 ground truth):
  OP-001 LoadConsumablesAndPropellants: "The process of fueling the rocket and loading all necessary consumables."
  OP-002 TransferCrewToVehicle: "The operation where the flight crew ingresses the Command Module."
  OP-003 PerformPreLaunchCountdown: "The synchronized sequence of checks in the final hours before liftoff."
  OP-004 ExecuteLaunchSequence: "The automated sequence of engine ignitions and staging events."
  OP-005 MonitorAscentTrajectory: "Continuous tracking of speed, altitude, and flight path during ascent."

Look for the OPERATIONAL SEQUENCE section in the source document (e.g., "Mission operational sequence", "Mission phases timeline"  or equivalent) and extract each step as an individual operation.

If a kill chain was identified in the Purpose layer, map each operation to its
corresponding step.
Table: | OP-ID | Operation Name | Description | Phase | Performer | Kill Chain Step (if applicable) | Source |
## 3. Engagement Sequences by Configuration
For EACH configuration identified in the Purpose layer (§8):
- Describe the step-by-step operational sequence
- If only one configuration exists, describe it once
Use sub-headings: ### Configuration: [label]

## 4. Rules of Engagement (ROE)
If ROE are described (even simplified or modeled), extract them.
If not stated, write "TBD — not stated in source".

## 5. Operational Constraints and Assumptions
Extract ALL constraints: fuel autonomy, range limits, timing, logistics,
idealized assumptions, spacing requirements, etc.
Table: | Category | Constraint | Source |
"""


def _p1_prompt_functional(text, purpose_md, operational_md):
    return f"""ACCUMULATED CONTEXT (Purpose + Operational layers):
---
### PURPOSE LAYER
{purpose_md[:12000]}

### OPERATIONAL LAYER
{operational_md[:12000]}
---

SOURCE TEXT (complete mission case study document):
---
{text}
---

Extract the CoSMA FUNCTIONAL LAYER from the source text.
This layer answers: "What does the system need to do?" — implementation-agnostic.

Generate ALL of the following sections:

## 1. Top-Level Function
Define the single top-level function (FN-001) that encompasses the entire mission.

## 2. Functional Decomposition
Decompose into a MULTI-LEVEL HIERARCHY of sub-functions.
Do NOT make all functions direct children of FN-001 — create intermediate
grouping levels that organize related functions.

REQUIRED STRUCTURE (follow this principle):
- Level 1: FN-001 (top-level mission function) — NO Refines OP-ID
- Level 2: Major functional areas that group related sub-functions
  (e.g., logistics, surveillance, engagement, assessment) — NO Refines OP-ID
- Level 3+: Further decomposition as needed until reaching leaf functions
  — intermediate levels have NO Refines OP-ID
- Leaf level: Concrete, atomic actions that are each allocatable to a single
  logical component and each refines exactly ONE operation — MUST have Refines OP-ID

STRUCTURAL RULES:
- Aim for 3-4 levels of depth. Flat hierarchies (all children of FN-001) are WRONG.
- Only LEAF functions (those with no children) should have Refines OP-ID filled.
- Orchestrating functions (those with children) must have Refines OP-ID = "" (empty).
- The number of functions per level depends on the mission complexity — let the
  source document drive the decomposition, not a fixed count.

EXAMPLE PATTERN (Apollo 11 ground truth):
  FN-001 PerformLunarMission (Level 1, parent=None)
    FN-002 ExecuteOutboundJourney (Level 2, parent=FN-001)
      FN-010 ProvideStage1Thrust (leaf, parent=FN-002, refines OP-004)
      FN-011 GuideAscentTrajectory (leaf, parent=FN-002, refines OP-005)
    FN-003 ConductLunarOperations (Level 2, parent=FN-001)
      FN-020 ExecuteDescentBurn (leaf, parent=FN-003, refines OP-012)

If a kill chain was identified, map each leaf function to its step.
Table: | FN-ID | Function Name | Description | Inputs | Outputs | Parent FN-ID | Refines OP-ID | Kill Chain Step | Source |

## 2.5 Data Item Definitions
List all distinct data items that flow between functions (inputs/outputs
from §2). These become `item def` in SysML v2.
Table: | Item Name (CamelCase) | Description | Produced by FN-ID | Consumed by FN-ID | Data Type (signal/message/stream/physical) | Source |
Examples: TargetCueingData, RadarTrack, EngagementAuthorization,
WeaponReleaseCommand, SurveillanceImagery, BDAReport

## 3. Functional Differences by Configuration
If multiple configurations exist, describe how each function differs.
Table: | FN-ID | Function Name | Baseline Behavior | Alternative Behavior | Notes |
If single configuration, write "Single configuration — no differences".

## 4. Functional Requirements
Table: | FR-ID | Requirement Text (\"The system shall...\") | Rationale (WHY) | Traces to FN-ID | Source |
For "Rationale": 1-2 sentences explaining WHY. Derive from context if not stated.
"""


def _p1_prompt_logical(text, purpose_md, operational_md, functional_md):
    return f"""ACCUMULATED CONTEXT (Purpose + Operational + Functional layers):
---
### PURPOSE LAYER
{purpose_md[:8000]}

### OPERATIONAL LAYER
{operational_md[:8000]}

### FUNCTIONAL LAYER
{functional_md[:12000]}
---

SOURCE TEXT (complete mission case study document):
---
{text}
---

Extract the CoSMA LOGICAL LAYER from the source text.
This layer answers: "What abstract parts are responsible for which functions?"

Generate ALL of the following sections:

## 1. Logical Components
List all PLATFORM-LEVEL abstract system components (implementation-independent).
A Logical Component represents a major mission role or actor — NOT a subsystem,
sensor, weapon, or communication device.

ABSTRACTION PRINCIPLE: Ask "what major platform or actor is responsible for this
group of functions?" — not "what specific equipment does it use?"
Subsystems, sensors, weapons, and communication links belong in the Technical
Layer (as Technical Components, ports, or interfaces), not here.

CORRECT examples: an interceptor aircraft platform, an airborne surveillance
platform, a ground control network, a threat entity class, a logistics system.
WRONG examples: a specific radar, a specific sensor, a specific rocket type,
a datalink radio, a weapon guidance kit — these are Technical Components.

The number of Logical Components depends entirely on the mission's structure.
Simple missions may have 3-5; complex Systems-of-Systems may have 10+.
Let the source document drive the count — do not inflate or deflate artificially.

Include friendly forces, external systems, and threat entities.
Table: | LC-ID | Component Name | Mission Role | Type (Friendly/External/Threat) | Source |

## 2. Function Allocation (Leaf Functions Only)
Allocate ONLY leaf-level functions (those with NO sub-functions in §2 of the
Functional Layer) to logical components. Each leaf function must be allocated
to EXACTLY ONE logical component — no duplication across LCs.
Do NOT allocate orchestrating/parent functions (those that compose sub-functions).
Table: | LC-ID | Component Name | Performs FN-ID (leaf only) | Function Name |

## 3. Logical Interfaces
Table: | From LC-ID | To LC-ID | Information Flow | Type (Data/Command/Status) | Source |

## 4. Mission System Composition (SoS)
Describe the System of Systems structure: which logical components are
constituents, what is the governance model, which are threat entities.
"""


def _p1_prompt_technical(text, purpose_md, operational_md, functional_md, logical_md):
    return f"""ACCUMULATED CONTEXT (all 4 previous layers):
---
### PURPOSE LAYER
{purpose_md[:6000]}

### OPERATIONAL LAYER
{operational_md[:6000]}

### FUNCTIONAL LAYER
{functional_md[:8000]}

### LOGICAL LAYER
{logical_md[:8000]}
---

SOURCE TEXT (complete mission case study document):
---
{text}
---

Extract the CoSMA TECHNICAL LAYER from the source text.
This layer answers: "How is the system physically built?"

Generate ALL of the following sections:

## 1. Technical Components by Platform/System
For EACH logical component identified in the Logical layer, list its
technical realization(s). Group by platform or system type.
Use sub-headings: ### Platform: [name] ([configuration label])
Table: | TC-ID | Component Name | Key Specs | Realizes LC-ID | Configuration | Source |

## 2. Technical Components — Threat Entities
If threat systems are described with technical detail:
Table: | TC-ID | Threat Type | Key Specs | Source |

## 3. Technical Ports
Table: | Port Name | Owner TC-ID | Direction (in/out/inout) | Item Type | Connected To | Source |

## 4. Mission Requirements
High-level requirements tracing to mission goals.
Table: | MR-ID | Requirement Text | Rationale (WHY — what drives this requirement) | Traces to GOAL-ID | Source |
For "Rationale": 1-2 sentences explaining WHY this requirement exists.
Derive from context if not explicitly stated.

## 5. Technical Requirements
Implementation-level requirements tracing to mission or functional requirements.
Table: | TR-ID | Requirement Text | Rationale (WHY — what drives this requirement) | Traces to MR-ID or FR-ID | Source |
For "Rationale": 1-2 sentences explaining WHY this requirement exists.
Derive from context if not explicitly stated.

## 6. Measures of Success (MOSs)
Table: | MOS-ID | Name | Description | Unit | Direction (higher=better?) | Threshold/Target (e.g., ">= 85%", "< 45s") | Formula | Variables | Source |
For "Threshold/Target": extract the success criterion if stated. If not explicit,
derive from context (e.g., "mission success requires all UAS neutralized" → ">= 100%").
If truly unknown, write "TBD".
For "Formula": extract the mathematical expression if defined in the document
(e.g., "Nn / Nt"). Write the formula as a single-line expression. If no formula, write "N/A".
For "Variables": list each variable as "symbol: description (unit)" separated by semicolons.
Example: "Nn: UAS neutralized before 80 nmi (count); Nt: total incursions (count)"

## 7. Measures of Effectiveness (MOEs)
Table: | MOE-ID | Name | Description | Unit | Formula | Variables | Traces to MOS-ID | Source |
For "Formula": extract the mathematical expression if defined (e.g.,
"1 - (1/(N * Tmax)) * sum(tef_i)"). Write as single-line expression. If no formula, write "N/A".
For "Variables": list each variable as "symbol: description (unit)" separated by semicolons.
Include sub-formulas if they define intermediate variables (e.g., "tef_i: effective
engagement time, see Eq.6 (s); Tmax: mean available time, see Eq.7 (s)").

## 8. Measures of Performance (MOPs)
For EACH configuration, list the MOP values. Use columns per configuration.
Table: | MOP-ID | Name | Description | Unit | Baseline Value | Alt Value(s) | Threshold/Range | Traces to MOE-ID | Source |
For "Threshold/Range": extract the performance criterion or valid range per configuration
(e.g., "[1, 10] s", "[5, 60] s", ">= 80%", "2 to 14 km"). If not stated, write "TBD".

## 9. Modeling Parameters and Sensitivity Ranges
Table: | Param | Description | Baseline Value | Alt Value(s) | Unit | Source |

## 10. Named Personnel and Operators
If the document names specific individuals (pilots, operators, commanders,
crew members) with roles or qualifications, list them.
Table: | Name | Role | Unit/Organization | Qualifications | Source |
If no named personnel are mentioned (common in military documents due to
operational security), write "No named personnel identified".

"""

# ── run_p1 ──────────────────────────────────────────

def run_p1(state: dict) -> dict:
    """PreprocessingAgent (P1): PDF → 5 CoSMA-layer Markdown documents.

    - max_tokens proportional to the section size (not hardcoded)
    - Mission-agnostic prompts (no references to a specific mission)

    State input: pdf_bytes, pdf_filename
    Output: mission_markdown, layer_markdowns
    """
    print(f'\n{"="*60}')
    print(f'  P1 — PreprocessingAgent')
    print(f'{"="*60}')
    t0 = time.time()

    # 1. Extract text from the PDF via PyMuPDF
    pdf_bytes = state['pdf_bytes']
    doc = fitz.open(stream=pdf_bytes, filetype='pdf')
    pages_text = [page.get_text() for page in doc]
    full_text = '\n\n'.join(pages_text)
    print(f'  PDF: {state["pdf_filename"]} | {len(pages_text)} pages | {len(full_text):,} chars')

    # 2. Adaptive max_tokens: based on document size
    #    Factor: ~1 token per 4 chars, 1.5x expansion for Markdown, minimum 6000
    doc_tokens_est = len(full_text) // 4
    def _adaptive_max_tokens(expansion_factor=1.5, minimum=6000, maximum=12000):
        """Computes max_tokens proportional to the document size."""
        estimated = int(doc_tokens_est * expansion_factor / 5)  # /5 because there are 5 layers
        return max(minimum, min(estimated, maximum))

    base_max_tokens = _adaptive_max_tokens()
    print(f'  adaptive max_tokens: {base_max_tokens} (doc ~{doc_tokens_est:,} tokens)')

    # 3. Sequential extraction of the 5 layers
    def _run_layer(name, prompt, max_tokens=None):
        mt = max_tokens or base_max_tokens
        print(f'  [{name}] Extracting (max_tokens={mt})...')
        t = time.time()
        result = call_llm(P1_SYSTEM_PROMPT, prompt, max_tokens=mt)
        print(f'  [{name}] {len(result):,} chars in {time.time()-t:.1f}s')
        return result

    md_purpose     = _run_layer('Purpose',     _p1_prompt_purpose(full_text))
    md_operational = _run_layer('Operational',  _p1_prompt_operational(full_text, md_purpose))
    md_functional  = _run_layer('Functional',   _p1_prompt_functional(full_text, md_purpose, md_operational))
    md_logical     = _run_layer('Logical',      _p1_prompt_logical(full_text, md_purpose, md_operational, md_functional))
    # Technical layer gets extra tokens (quantitative data is denser)
    tech_tokens = min(base_max_tokens + 4000, 16000)
    md_technical   = _run_layer('Technical',    _p1_prompt_technical(full_text, md_purpose, md_operational, md_functional, md_logical),
                                max_tokens=tech_tokens)

    # 4. Consolidate markdown
    SEPARATOR = '\n\n---\n\n'
    mission_markdown = (
        f'# PURPOSE LAYER\n> "Why does this system exist?"\n\n{md_purpose}'
        + SEPARATOR
        + f'# OPERATIONAL LAYER\n> "When do things happen and in what sequence?"\n\n{md_operational}'
        + SEPARATOR
        + f'# FUNCTIONAL LAYER\n> "What does the system need to do?"\n\n{md_functional}'
        + SEPARATOR
        + f'# LOGICAL LAYER\n> "What abstract parts are responsible for which functions?"\n\n{md_logical}'
        + SEPARATOR
        + f'# TECHNICAL LAYER\n> "How is the system physically built?"\n\n{md_technical}'
    )

    layer_markdowns = {
        '1_purpose': md_purpose,
        '2_operational': md_operational,
        '3_functional': md_functional,
        '4_logical': md_logical,
        '5_technical': md_technical,
    }

    # 5. Save to Drive
    preproc_dir = PREPROCESSING_DIR / f'run_{state["run_timestamp"]}'
    preproc_dir.mkdir(parents=True, exist_ok=True)
    md_path = preproc_dir / f'cosma_markdown_{state["run_timestamp"]}.md'
    md_path.write_text(mission_markdown, encoding='utf-8')
    for name, content in layer_markdowns.items():
        (preproc_dir / f'{name}.md').write_text(content, encoding='utf-8')
    print(f'  Saved to: {preproc_dir}')

    elapsed = time.time() - t0
    log_cost(state, 'P1_PreprocessingAgent', estimate_cost(len(full_text)//4*5, len(mission_markdown)//4), elapsed)

    print(f'  P1 finished: {len(mission_markdown):,} chars in {elapsed:.1f}s')
    return {
        'mission_markdown': mission_markdown,
        'layer_markdowns': layer_markdowns,
        'pdf_pages': len(pages_text),
    }


# ================================================================
#                     P2 — MISSION AGENT
# ================================================================

# ── P2 System Prompt  ─────

P2_SYSTEM_PROMPT = """(U.S. DoD Mission Engineering Guide, 2023), the CoSMA framework (Helle & Schramm, 2026 — "Fly me to the Moon: Modeling Apollo 11 using SysML v2") and SysML v2 (Objet Management Group, 2025 - OMG Systems Modeling Language (SysML) Version 2.0: Part 1: Language Specification).

You are converting structured Markdown (extracted from a military mission case study)
into JSON data conforming to Pydantic schemas. This JSON will be used downstream by:
  - TemplateGeneratorClass: to generate a syntactically correct SysML v2 skeleton
  - RefinementAgent: to fill the skeleton with complete data

EXTRACTION RULES:
1. Extract ONLY what is explicitly stated or directly derivable from the source text.
   NEVER invent data, actors, metrics, or requirements.
   If required information is absent, write exactly: "TBD — not stated in source"
2. Each extracted item MUST include the source reference (e.g., "§2.1.1").
3. Respond ONLY with a valid JSON object matching the schema provided.
   No conversational preamble, no markdown fencing, no explanation.
   Start with { and end with }.
4. Use consistent ID prefixes:
   - SH-XXX (stakeholders), SHN-XXX (stakeholder needs), CAP-XXX (capabilities)
   - GOAL-XXX (goals), MR-XXX (mission requirements)
   - FR-XXX (functional requirements), TR-XXX (technical requirements)
   - OP-XXX (operations), FN-XXX (functions), LC-XXX (logical components)
   - TC-XXX (technical components), PH-XXX (phases)
   - EXT-XXX (external systems), MOS-XXX, MOE-XXX, MOP-XXX
5. Use CamelCase for SysML definition names.
6. Use English for all field values.
7. For military-domain terms, preserve the original-language acronym in
   parentheses. Detect the source language automatically.
8. If the source Markdown describes multiple system configurations
   (baseline vs. alternatives), distinguish them using the Configuration
   enum values: 'baseline', 'modernized', or 'both'. If more than two
   configurations exist, use the document's own labels.
9. Capture ALL quantitative data (specs, parameters, values, units).
   For each numeric value, indicate provenance:
   - If the value appears explicitly in the source text → source field = section ref
   - If the value was inferred or estimated → source field = "inferred — [rationale]"
"""


# ── P2 helpers ─────────────────────────────────────────────

def _p2_extract_section(md: str, start_marker: str, end_marker: str | None) -> str:
    idx_start = md.find(start_marker)
    if idx_start == -1:
        return ''
    if end_marker:
        idx_end = md.find(end_marker, idx_start + len(start_marker))
        if idx_end == -1:
            return md[idx_start:]
        return md[idx_start:idx_end]
    return md[idx_start:]


def _p2_schema_for_prompt(model_class) -> str:
    schema = model_class.model_json_schema()
    def _clean(d):
        if isinstance(d, dict):
            d.pop('title', None)
            for v in d.values(): _clean(v)
        elif isinstance(d, list):
            for v in d: _clean(v)
        return d
    _clean(schema)
    return _json.dumps(schema, indent=2, ensure_ascii=False)


def _p2_parse_json(raw: str) -> dict:
    cleaned = re.sub(r'^```(?:json)?\s*\n?', '', raw.strip())
    cleaned = re.sub(r'\n?```\s*$', '', cleaned)
    cleaned = cleaned.strip()
    first = cleaned.find('{')
    last = cleaned.rfind('}')
    if first != -1 and last != -1:
        cleaned = cleaned[first:last + 1]
    try:
        return _json.loads(cleaned)
    except _json.JSONDecodeError as e:
        import json_repair
        print(f'    ⚠ json.loads failed ({e}), trying json_repair...')
        repaired = json_repair.loads(cleaned)
        print(f'    ✅ json_repair recovered the JSON successfully')
        return repaired


def _p2_normalize_enums(obj):
    """Normalize LLM-generated enum values that contain extra text or wrong casing.
    E.g., 'inferred — conversion from DMS' → 'inferred'
    E.g., 'find' → 'Find' (F2T2EAStep uses Title Case)
    Handles nested dicts and lists recursively.
    """
    # Maps field_name → {lowercase_prefix: exact_enum_value}
    ENUM_MAP = {
        'provenance': {'explicit': 'explicit', 'inferred': 'inferred'},
        'configuration': {'both': 'both', 'baseline': 'baseline', 'modernized': 'modernized'},
        'influence_level': {'high': 'high', 'medium': 'medium', 'low': 'low'},
        'influence_kind': {'direct': 'direct', 'indirect': 'indirect'},
        'f2t2ea_step': {'find': 'Find', 'fix': 'Fix', 'track': 'Track',
                        'target': 'Target', 'engage': 'Engage', 'assess': 'Assess',
                        'n/a': 'N/A', 'not_applicable': 'N/A'},
    }
    if isinstance(obj, dict):
        for key, val in obj.items():
            if key in ENUM_MAP and isinstance(val, str):
                val_lower = val.lower().strip()
                for prefix, correct_val in ENUM_MAP[key].items():
                    if val_lower.startswith(prefix):
                        obj[key] = correct_val
                        break
            else:
                _p2_normalize_enums(val)
    elif isinstance(obj, list):
        for item in obj:
            _p2_normalize_enums(item)
    return obj


def _p2_run_subagent(name, user_prompt, model_class, max_tokens=MAX_TOKENS_P2, max_retries=1):
    for attempt in range(max_retries + 1):
        if attempt > 0:
            print(f'  [{name}] Retry {attempt}/{max_retries}...')
        print(f'  [{name}] Extracting (max_tokens={max_tokens})...')
        t = time.time()
        raw = call_llm_streaming(P2_SYSTEM_PROMPT, user_prompt, max_tokens=max_tokens)
        elapsed = time.time() - t
        print(f'  [{name}] {len(raw):,} chars in {elapsed:.1f}s')
        try:
            data_dict = _p2_parse_json(raw)
        except _json.JSONDecodeError as e:
            print(f'  [{name}] JSON ERROR: {e}')
            if attempt < max_retries:
                continue
            return None, raw
        try:
            data_dict = _p2_normalize_enums(data_dict)
            validated = model_class.model_validate(data_dict)
            print(f'  [{name}] Pydantic OK ✓')
            return validated, raw
        except Exception as e:
            print(f'  [{name}] Pydantic ERROR: {e}')
            if attempt < max_retries:
                continue
            return None, raw


# ── P2 prompts (11 sub-agents) ────────────────────────────

def _p2_prompt_stakeholder(sec):
    s = _p2_schema_for_prompt(StakeholderData)
    return f"""SOURCE MARKDOWN (Purpose Layer — Stakeholders and Needs):
---
{sec}
---

Extract ALL stakeholders and their needs.
For each Stakeholder: SH-XXX ID, CamelCase name, full name, role, type, influence, concerns, need_ids.
For each StakeholderNeed: SHN-XXX ID, CamelCase name, SHALL text, stakeholder_ids, source.
  Additionally, for each StakeholderNeed provide a "rationale" field: a 1-2 sentence
  justification explaining WHY this need exists and what drives it (e.g., operational
  necessity, safety concern, national mandate). If the source text does not provide
  an explicit rationale, derive one from the stakeholder's concerns and mission context.

OUTPUT JSON SCHEMA:
{s}

Respond with a single valid JSON object matching this schema."""

def _p2_prompt_capability(sec):
    s = _p2_schema_for_prompt(CapabilityData)
    return f"""SOURCE MARKDOWN (Purpose Layer — Capabilities):
---
{sec}
---

Extract ALL capabilities. CAP-XXX IDs, CamelCase names, descriptions, GOAL IDs.

OUTPUT JSON SCHEMA:
{s}

Respond with a single valid JSON object matching this schema."""


def _p2_prompt_mission(sec_purpose, sec_operational):
    s = _p2_schema_for_prompt(MissionData)
    return f"""SOURCE MARKDOWN — Purpose Layer (Program, Goals):
---
{sec_purpose}
---

SOURCE MARKDOWN — Operational Layer (Mission Phases):
---
{sec_operational}
---

Extract Mission definition, Goals, and Mission Phases.
Mission is part def (structural), NOT action def.
Phases: capture branching (e.g., PH-005 can go to PH-006 or PH-008).

For the Mission, also extract:
  - "program_name": Name of the overarching program or campaign this mission belongs to.
  - "program_description": One paragraph describing the program's purpose and scope.
  - "related_missions": Other missions or operations mentioned in the source document that
    belong to the same program/campaign. These may include prior operations, parallel
    deployments, exercises, or future phases. Mark the current mission with is_current=true.
    If no related missions are mentioned, include only the current mission.

Example (Apollo 11 ground truth):
  program_name: "ApolloProgram"
  program_description: "The United States human spaceflight program led by NASA..."
  related_missions: [
    {{"name": "Apollo1", "description": "Not flown. Crew died in launch pad fire.", "is_current": false}},
    {{"name": "Apollo7", "description": "First crewed Earth orbital CSM test.", "is_current": false}},
    {{"name": "Apollo11", "description": "First crewed lunar landing.", "is_current": true}},
    {{"name": "Apollo12", "description": "Second lunar landing.", "is_current": false}}
  ]

OUTPUT JSON SCHEMA:
{s}

Respond with a single valid JSON object matching this schema."""


def _p2_prompt_context(sec):
    s = _p2_schema_for_prompt(ContextData)
    return f"""SOURCE MARKDOWN (Purpose Layer — Context):
---
{sec}
---

Extract Mission Context: scenario, epoch, area of operations, geopolitical context,
bases of operation, patrol/waypoints, logistics points, operational boundaries, external systems.
Use exact coordinates from source tables when available.

OUTPUT JSON SCHEMA:
{s}

Respond with a single valid JSON object matching this schema."""


def _p2_prompt_operation(sec):
    s = _p2_schema_for_prompt(OperationData)
    return f"""SOURCE MARKDOWN (Operational Layer):
---
{sec}
---

Extract ALL operations, engagement sequences (baseline + modernized), ROE, constraints.

CRITICAL — Operation Granularity:
Each operation MUST represent a CONCRETE, SEQUENTIAL STEP in the mission timeline —
NOT a generic doctrinal category. Think of operations as "what happens next" in the
mission execution, from preparation through mission complete.

CORRECT examples (concrete steps):
  "PrepareAircraftForDeployment" — "Configure aircraft systems, load armament, and brief crew before departure"
  "TransitToForwardBase" — "Fly aircraft from main base to designated FARP for refueling"
  "RefuelAndRearmAtFarp" — "Replenish fuel and rocket armament at forward operating point"
  "EstablishPatrolOrbit" — "Navigate to assigned PAC point and enter holding pattern"
  "DetectAndCueTarget" — "Radar system detects UAS incursion and transmits cueing data to interceptor"
  "NavigateToIntercept" — "Adjust flight path from patrol orbit to intercept geometry"
  "AcquireTargetVisually" — "Pilot establishes visual or sensor contact with UAS"
  "EngageWithKineticWeapon" — "Fire rocket armament to neutralize UAS"
  "AssessBattleDamage" — "Determine if target neutralized or evaded"

INCORRECT examples (doctrinal categories — do NOT use):
  "AerospaceControl" — too abstract, describes a doctrinal concept not a step
  "AirspacePolicing" — a mission TYPE, not an operational step
  "CombatSustainment" — a logistics CATEGORY, not a concrete action

OUTPUT JSON SCHEMA:
{s}

Respond with a single valid JSON object matching this schema."""

def _p2_prompt_function(sec):
    s = _p2_schema_for_prompt(FunctionData)
    return f"""SOURCE MARKDOWN (Functional Layer):
---
{sec}
---

Extract item definitions, functions (in a hierarchy), and functional requirements.

=== STEP 1: Define Domain Item Definitions (item_defs) ===
Define 5-10 concise item types representing the key DATA FLOWS in this mission.
These will type ALL function inputs and outputs. Choose generic, reusable names.

Example item defs (from Apollo 11 ground truth):
  VehicleStatus: "Represents the complete state of a vehicle system, including configuration, resources, and trajectory."
  CrewStatus: "Represents the state of the crew, including health and readiness."
  MissionPlan: "Contains the overall objectives and parameters for the mission."
  MissionReport: "A comprehensive report detailing the outcomes of the mission."
  LunarSamples: "Represents the collected geological materials from the Moon."

CRITICAL: Do NOT create a unique type for every input/output. Reuse types.
  WRONG: "DetectionAlertsFromDECEAGroundRadarsAndE99MAirborneRadar" (too specific)
  RIGHT: "SurveillanceAlert" or "TargetTrack" (reusable across functions)

=== STEP 2: Define Functions in a HIERARCHY ===
Functions MUST form a tree:
  Level 1: ONE top-level function (FN-001) representing the entire mission.
           parent_id = null. No refines_op_ids.
  Level 2: 3-6 orchestrating functions grouping major mission phases.
           parent_id = FN-001. No refines_op_ids.
  Level 3+: Leaf functions performing concrete actions.
            parent_id = their orchestrating parent. refines_op_ids = [OP-XXX].

Rules:
  - Every function's inputs/outputs type_desc MUST be one of the item_defs from Step 1.
  - parent_id is MANDATORY for all functions except FN-001.
  - ONLY leaf functions (no children) have refines_op_ids.
  - Include variant functions for different configurations as separate entries.

EXAMPLE hierarchy (Apollo 11 ground truth):
  FN-001 PerformLunarMission (top, parent=null)
    FN-002 ExecuteOutboundJourney (orchestrating, parent=FN-001)
      FN-010 PrepareForLaunch (orchestrating, parent=FN-002)
        FN-020 PerformPropellantLoading (leaf, parent=FN-010, refines OP-001)
        FN-021 PerformCrewIngress (leaf, parent=FN-010, refines OP-002)
      FN-011 LaunchToOrbit (orchestrating, parent=FN-002)
        FN-022 ProvideStage1Thrust (leaf, parent=FN-011, refines OP-004)
        FN-023 GuideAscentTrajectory (leaf, parent=FN-011, refines OP-005)

=== STEP 3: Functional Requirements ===
Extract ALL functional requirements with traces to function IDs.

OUTPUT JSON SCHEMA:
{s}

Respond with a single valid JSON object matching this schema."""

def _p2_prompt_logical(sec):
    s = _p2_schema_for_prompt(LogicalData)
    return f"""SOURCE MARKDOWN (Logical Layer):
---
{sec}
---

Extract ALL logical components, interfaces, SoS structure.
Set is_threat=true for adversary entities.

CRITICAL — performs_fn_ids:
- Include ONLY leaf-level function IDs (those with no sub-functions).
- Each leaf FN-ID must appear in EXACTLY ONE logical component.
- Do NOT include orchestrating/parent function IDs.

OUTPUT JSON SCHEMA:
{s}

Respond with a single valid JSON object matching this schema."""


def _p2_prompt_technical(sec):
    s = _p2_schema_for_prompt(TechnicalData)
    return f"""SOURCE MARKDOWN (Technical Layer — Components and Ports):
---
{sec}
---

Extract technical components, port architecture, interface definitions, and named individuals.

=== STEP 1: Technical Components ===
Extract ALL technical components with their specifications.
Capture EVERY numerical parameter from source tables.

=== STEP 2: Port Flow Item Definitions (port_flow_item_defs) ===
Define 5-10 concise domain flow types representing data/energy/material exchanged between
systems. These type ALL items inside port definitions.

Example flow item defs (Apollo 11 ground truth):
  CommandSignal: "Represents a discrete command or electrical signal."
  TelemetryData: "Represents a flow of vehicle health and status data."
  ElectricalPower: "Represents a flow of electrical energy."

For a C-UAS mission, appropriate flow types might include:
  RadarTrackData, TargetCueingData, VoiceCommData, DatalinkData,
  SensorImagery, WeaponCommand, FuelFlowData, NavigationData

CRITICAL: Reuse types across ports. Do NOT create a unique type per port.

=== STEP 3: Port Definitions (port_defs) ===
Define port types as connection points on components. Each port has multiple
directional items typed by the flow item defs from Step 2.

Example (Apollo 11 ground truth):
  PayloadInterfacePort:
    inout loads: StructuralLoad
    out commands: CommandSignal
    in telemetry: TelemetryData
    out power: ElectricalPower

Each port represents a LOGICAL CONNECTION POINT, not a physical connector.
Name ports by their function: DatalinkPort, SensorDataPort, VoiceCommPort, etc.

=== STEP 4: Interface Definitions (interface_defs) ===
Define contracts for how ports connect. Each interface has two ends.
The second end is typically conjugated (~), meaning flows reverse direction.

Example (Apollo 11 ground truth):
  StagingInterface: end source: StagingPort, end target: ~StagingPort
  DockingInterface: end vehicle1: DockingPort, end vehicle2: DockingPort (symmetric)

=== STEP 5: Ports on Components ===
For each technical component, list which ports it owns in the existing "ports" field.
Use owner_tc_id to associate ports with components.
CRITICAL: For each port, set port_def_name to the name of the PortDef from Step 3
that best matches this port's function. This creates the type reference.

Ground truth pattern (Apollo 11):
  TechnicalPortsPackage defines: StagingPort, ControlPort, UmbilicalPort, DockingPort
  TechnicalComponentsPackage references them:
    SaturnVInstrumentUnit has port stageControlPort → port_def_name: "ControlPort"
    ApolloServiceModule has port umbilicalPort → port_def_name: "UmbilicalPort"
    ApolloCommandModule has port dockingPort → port_def_name: "DockingPort"

The port instance name is descriptive (e.g., "stageControlPort"), but the port_def_name
MUST match exactly one of the port defs defined in Step 3.
Every port MUST have a port_def_name referencing a port_def from Step 3.

=== STEP 6: Technical Individuals (technical_individuals) ===
Extract specifically NAMED instances of components from the source document.
These are individual, identifiable items (not types/classes).

Examples: specific bases (SBCZ, SWKU), specific aircraft (if tail numbers given),
specific radar installations. Each references the TC-ID of its type.

OUTPUT JSON SCHEMA:
{s}

Respond with a single valid JSON object matching this schema."""

def _p2_prompt_requirements(sec_tech, sec_func):
    s = _p2_schema_for_prompt(RequirementsData)
    return f"""SOURCE MARKDOWN — Technical Layer (Requirements):
---
{sec_tech}
---

SOURCE MARKDOWN — Functional Layer:
---
{sec_func}
---

Extract ALL requirements: MR-XXX, FR-XXX, TR-XXX.

For EACH requirement, provide:
  1. "id": Standard prefix (MR-001, FR-001, TR-001)
  2. "descriptive_name": A CamelCase name that describes what the requirement is about.
     WRONG: "MR001Requirement", "FR002Requirement", "TR003Requirement"
     RIGHT: "UASDetectionCoverageRequirement", "TargetCueingLatencyRequirement",
            "APKWSEngagementEnvelopeRequirement"
     The name should convey the requirement's subject without reading the full text.
  3. "text": Full SHALL statement in English.
  4. "rationale": 1-2 sentences explaining WHY this requirement exists — what drives it,
     what risk it mitigates, or what capability it enables. Derive from context if not
     explicitly stated.
  5. "traces_to": IDs of upper-layer elements this requirement traces to.
     MR traces to GOAL-XXX or CAP-XXX.
     FR traces to MR-XXX.
     TR traces to FR-XXX.

Example (Apollo 11 ground truth):
  MR: id="MR-001", descriptive_name="CrewReturnSafetyRequirement",
      text="The mission shall safely return all crew members to Earth.",
      rationale="This is paramount for fulfilling the core mission objective and addresses the primary concern of all stakeholders regarding human life safety.",
      traces_to=["CAP-006", "CAP-005"]

  FR: id="FR-001", descriptive_name="PropellantLoadingRequirement",
      text="The system shall enable the loading of all required propellants.",
      rationale="All propulsive stages must be fully supplied to perform their functions.",
      traces_to=["MR-001"]

  TR: id="TR-001", descriptive_name="DatalinkUpdateRateRequirement",
      text="The A-29N shall receive target coordinates via datalink at [1,10]s latency.",
      rationale="Instantaneous cueing data is critical for reducing engagement timeline vs voice baseline.",
      traces_to=["FR-003"]

OUTPUT JSON SCHEMA:
{s}

Respond with a single valid JSON object matching this schema."""

def _p2_prompt_measures(sec):
    s = _p2_schema_for_prompt(MeasuresData)
    return f"""SOURCE MARKDOWN (Technical Layer — Measures and Parameters):
---
{sec}
---

Extract ALL MOSs, MOEs, MOPs, and simulation parameters.
Capture EVERY row from the parameters table.

CRITICAL — Formula extraction:
- For each MOS and MOE, extract the "formula" field as a single-line math expression
  (e.g., "Nn / Nt", "1 - (1/(N * Tmax)) * sum(tef_i)").
- For "variables", extract EACH symbol used in the formula as a FormulaVariable object:
  name = symbol (e.g., "Nn"), description = what it represents,
  role = "in" for inputs or "out" for the result, unit = physical unit if applicable.
- For MOS "threshold", extract the success criterion (e.g., ">= 0.80", "< 45 s").
- For MOP "threshold", extract the valid range or criterion per configuration
  (e.g., "[1, 10] s", "[5, 60] s", "2 to 14 km").
- If a formula or threshold is not stated, use "" or "TBD" respectively.

OUTPUT JSON SCHEMA:
{s}

Respond with a single valid JSON object matching this schema."""


def _p2_prompt_specification(traceability_ctx):
    s = _p2_schema_for_prompt(SpecificationData)
    return f"""EXTRACTED DATA SUMMARY (traceability from all sub-agents):
---
{traceability_ctx}
---

Generate cross-cutting satisfy and refine links.

SATISFY links (requirement_id satisfied_by component/function):
  MR-XXX satisfied by OP-XXX ONLY (operations that fulfill mission requirements)
  FR-XXX satisfied by FN-XXX ONLY (leaf functions that fulfill functional requirements)
  TR-XXX satisfied by TC-XXX or LC-XXX (components that fulfill technical requirements)

SEMANTIC CORRELATION RULES for satisfy links:
  - Each requirement MUST have at least 1 satisfy link.
  - A requirement MAY have MULTIPLE satisfy links if multiple operations/functions/components
    jointly fulfill it (add one satisfy_links entry per correlation).
  - Example: MR-001 "Neutralize UAS before 80 nm boundary" may be satisfied by:
      OP-XXX "DetectIllicitUAS", OP-XXX "IdentifyUASThreat", OP-XXX "ExecuteFiring".
  - Use the descriptions and names to match semantics — do NOT correlate by ID patterns.
  - For methodological or stochastic requirements (e.g., "shall use Monte Carlo methodology",
    "shall model stochastically") that describe HOW the system is analyzed rather than a
    component/function behavior, OMIT the satisfy link entirely. These requirements are
    fulfilled by the modeling approach itself, not by a specific system feature.


REFINE links — use EXACTLY these relationship strings:
  "goal_refines_stakeholder_need": source_id=GOAL-XXX, target_id=SHN-XXX
  "capability_refines_goal": source_id=CAP-XXX, target_id=GOAL-XXX
  "mission_requirement_refines_capability": source_id=MR-XXX, target_id=CAP-XXX
  "functional_requirement_refines_mission_requirement": source_id=FR-XXX, target_id=MR-XXX
  "technical_requirement_refines_functional_requirement": source_id=TR-XXX, target_id=FR-XXX
  "function_refines_operation": source_id=FN-XXX, target_id=OP-XXX

Generate ALL applicable links. Every Goal MUST refine at least one SHN.
Every Capability MUST refine at least one Goal.

OUTPUT JSON SCHEMA:
{s}

Respond with a single valid JSON object matching this schema."""

def _p2_build_traceability_context(results):
    lines = []
    if results.get('stakeholder'):
        lines.append('## StakeholderNeeds')
        for shn in results['stakeholder'].stakeholder_needs:
            lines.append(f'  {shn.id} ({shn.name}): stakeholders={shn.stakeholder_ids}')
    if results.get('mission'):
        lines.append('## Goals')
        for g in results['mission'].goals:
            lines.append(f'  {g.id} ({g.name}): refines={g.refines_shn_ids}')
    if results.get('capability'):
        lines.append('## Capabilities')
        for c in results['capability'].capabilities:
            lines.append(f'  {c.id} ({c.name}): supports={c.supports_goal_ids}')
    if results.get('operation'):
        lines.append('## Operations (leaf-level mission activities)')
        for op in results['operation'].operations:
            phase_info = f' phases={op.phase_ids}' if op.phase_ids else ''
            lines.append(f'  {op.id} ({op.name}){phase_info}: {op.description[:400]}')
    if results.get('function'):
        lines.append('## Functions')
        for fn in results['function'].functions:
            if fn.refines_op_ids:
                lines.append(f'  {fn.id} ({fn.name}): refines_ops={fn.refines_op_ids}')
    if results.get('logical'):
        lines.append('## LogicalComponents')
        for lc in results['logical'].logical_components:
            lines.append(f'  {lc.id} ({lc.name}): performs={lc.performs_fn_ids}')
    if results.get('technical'):
        lines.append('## TechnicalComponents')
        for tc in results['technical'].technical_components:
            lines.append(f'  {tc.id} ({tc.name}): realizes={tc.realizes_lc_id}')
    if results.get('requirements'):
        lines.append('## MissionRequirements')
        for r in results['requirements'].mission_requirements:
            lines.append(f'  {r.id} ({r.descriptive_name}): {r.text[:500]}')
        lines.append('## FunctionalRequirements')
        for r in results['requirements'].functional_requirements:
            lines.append(f'  {r.id} ({r.descriptive_name}): {r.text[:500]}')
        lines.append('## TechnicalRequirements')
        for r in results['requirements'].technical_requirements:
            lines.append(f'  {r.id} ({r.descriptive_name}): {r.text[:500]}')
    return '\n'.join(lines)


# ── run_p2: wrapper State → State ──────────────────────────

def _p2_build_accumulated_context(results: dict) -> str:
    """Builds a summary of IDs already extracted by previous sub-agents.

    Used to inject cumulative context into every downstream sub-agent,
    ensuring that cross-layer traceability IDs are visible.

    Addresses Q6/L2: RequirementsExtractor now sees SHN-XXX, GOAL-XXX, etc.
    """
    lines = ['## Previously Extracted IDs (use for traceability cross-references):']

    if results.get('stakeholder'):
        lines.append('### Stakeholders')
        for s in results['stakeholder'].stakeholders:
            lines.append(f'  {s.id}: {s.name}')
        lines.append('### StakeholderNeeds')
        for sn in results['stakeholder'].stakeholder_needs:
            lines.append(f'  {sn.id}: {sn.name}')

    if results.get('capability'):
        lines.append('### Capabilities')
        for c in results['capability'].capabilities:
            lines.append(f'  {c.id}: {c.name} (supports {c.supports_goal_ids})')

    if results.get('mission'):
        lines.append('### Goals')
        for g in results['mission'].goals:
            lines.append(f'  {g.id}: {g.name} (refines {g.refines_shn_ids})')
        lines.append('### Phases')
        for p in results['mission'].phases:
            lines.append(f'  {p.id}: {p.name} (ops: {p.operation_ids})')

    if results.get('operation'):
        lines.append('### Operations')
        for o in results['operation'].operations:
            lines.append(f'  {o.id}: {o.name} (phase: {o.phase_ids}, F2T2EA: {o.f2t2ea_step.value})')

    if results.get('function'):
        lines.append('### Functions')
        for fn in results['function'].functions:
            lines.append(f'  {fn.id}: {fn.name} (refines_ops: {fn.refines_op_ids})')
        lines.append('### FunctionalRequirements')
        for fr in results['function'].functional_requirements:
            lines.append(f'  {fr.id}: traces_to={fr.traces_to}')

    if results.get('logical'):
        lines.append('### LogicalComponents')
        for lc in results['logical'].logical_components:
            lines.append(f'  {lc.id}: {lc.name} (performs: {lc.performs_fn_ids})')

    if results.get('technical'):
        lines.append('### TechnicalComponents')
        for tc in results['technical'].technical_components:
            lines.append(f'  {tc.id}: {tc.name} (realizes: {tc.realizes_lc_id}, config: {tc.configuration.value})')

    if results.get('requirements'):
        lines.append('### MissionRequirements')
        for r in results['requirements'].mission_requirements:
            lines.append(f'  {r.id}: traces_to={r.traces_to}')
        lines.append('### TechnicalRequirements')
        for r in results['requirements'].technical_requirements:
            lines.append(f'  {r.id}: traces_to={r.traces_to}')

    if results.get('measures'):
        lines.append('### MeasuresOfSuccess')
        for m in results['measures'].measures_of_success:
            lines.append(f'  {m.id}: {m.name}')
        lines.append('### MeasuresOfEffectiveness')
        for m in results['measures'].measures_of_effectiveness:
            lines.append(f'  {m.id}: {m.name} (traces_to_mos: {m.traces_to_mos_ids})')

    return '\n'.join(lines)


def _p2_with_context(base_prompt: str, accumulated_ctx: str) -> str:
    """Injects accumulated context of previous IDs into a sub-agent prompt.

    If there is no accumulated context (first sub-agent), returns the original prompt.
    The context is added BEFORE the prompt, so the LLM sees it as reference.
    """
    if not accumulated_ctx or accumulated_ctx.count('\n') < 3:
        return base_prompt
    return f"""{accumulated_ctx}

---

{base_prompt}"""


# ── run_p2  ──────────────────────

def run_p2(state: dict) -> dict:
    """MissionAgent (P2): Markdown → 11 consolidated Pydantic JSONs.

    - Each sub-agent receives the IDs from previous sub-agents (Q6/L2)
    - Context is built progressively via _p2_build_accumulated_context
    - Downstream sub-agents may reference upstream IDs for traceability

    State input: mission_markdown
    Output: mission_json (consolidated dict)
    """
    print(f'\n{"="*60}')
    print(f'  P2 — MissionAgent v2 (11 sub-agents + cumulative context)')
    print(f'{"="*60}')
    t0 = time.time()

    md = state['mission_markdown']

    # Split sections by layer
    sec_purpose = _p2_extract_section(md, '# PURPOSE LAYER', '# OPERATIONAL LAYER')
    sec_operational = _p2_extract_section(md, '# OPERATIONAL LAYER', '# FUNCTIONAL LAYER')
    sec_functional = _p2_extract_section(md, '# FUNCTIONAL LAYER', '# LOGICAL LAYER')
    sec_logical = _p2_extract_section(md, '# LOGICAL LAYER', '# TECHNICAL LAYER')
    sec_technical = _p2_extract_section(md, '# TECHNICAL LAYER', None)

    print(f'  Sections: P={len(sec_purpose):,} O={len(sec_operational):,} '
          f'F={len(sec_functional):,} L={len(sec_logical):,} T={len(sec_technical):,}')

    # ── Execution of the 11 sub-agents with cumulative context ──
    results = {}

    # 1. StakeholderExtractor (no prior context)
    results['stakeholder'], _ = _p2_run_subagent(
        'StakeholderExtractor', _p2_prompt_stakeholder(sec_purpose), StakeholderData)

    # 2. CapabilityExtractor (sees: stakeholders, needs)
    acc = _p2_build_accumulated_context(results)
    results['capability'], _ = _p2_run_subagent(
        'CapabilityExtractor',
        _p2_with_context(_p2_prompt_capability(sec_purpose), acc),
        CapabilityData)

    # 3. MissionExtractor (sees: stakeholders, needs, capabilities)
    acc = _p2_build_accumulated_context(results)
    results['mission'], _ = _p2_run_subagent(
        'MissionExtractor',
        _p2_with_context(_p2_prompt_mission(sec_purpose, sec_operational), acc),
        MissionData)

    # 4. ContextExtractor (sees: stakeholders, needs, capabilities, goals)
    acc = _p2_build_accumulated_context(results)
    results['context'], _ = _p2_run_subagent(
        'ContextExtractor',
        _p2_with_context(_p2_prompt_context(sec_purpose), acc),
        ContextData)

    # 5. OperationExtractor (sees: all Purpose + goals/phases)
    acc = _p2_build_accumulated_context(results)
    results['operation'], _ = _p2_run_subagent(
        'OperationExtractor',
        _p2_with_context(_p2_prompt_operation(sec_operational), acc),
        OperationData)

    # 6. FunctionExtractor (sees: all Purpose + Operational)
    acc = _p2_build_accumulated_context(results)
    results['function'], _ = _p2_run_subagent(
        'FunctionExtractor',
        _p2_with_context(_p2_prompt_function(sec_functional), acc),
        FunctionData)

    # 7. LogicalExtractor (sees: everything up to Functions)
    acc = _p2_build_accumulated_context(results)
    results['logical'], _ = _p2_run_subagent(
        'LogicalExtractor',
        _p2_with_context(_p2_prompt_logical(sec_logical), acc),
        LogicalData)

    # ── CoSMA validation: leaf-function allocation uniqueness ──
    if results.get('logical') and results.get('function'):
        fn_data = results['function']
        parent_ids = {fn.parent_id for fn in fn_data.functions if fn.parent_id}
        leaf_fn_ids = {fn.id for fn in fn_data.functions if fn.id not in parent_ids}

        allocated = []
        for lc in results['logical'].logical_components:
            allocated.extend(lc.performs_fn_ids)

        from collections import Counter
        counts = Counter(allocated)
        dupes = {fid: cnt for fid, cnt in counts.items() if cnt > 1 and fid in leaf_fn_ids}
        if dupes:
            print(f'  ⚠ CoSMA: leaf functions allocated to multiple LCs: {dupes}')

        orch_allocated = [fid for fid in allocated if fid not in leaf_fn_ids and fid in {fn.id for fn in fn_data.functions}]
        if orch_allocated:
            print(f'  ⚠ CoSMA: orchestrating functions in performs_fn_ids: {orch_allocated}')

    # 8. TechnicalExtractor (sees: everything up to LogicalComponents)
    acc = _p2_build_accumulated_context(results)
    results['technical'], _ = _p2_run_subagent(
        'TechnicalExtractor',
        _p2_with_context(_p2_prompt_technical(sec_technical), acc),
        TechnicalData)

    # 9. RequirementsExtractor (sees: EVERYTHING — critical for traceability)
    acc = _p2_build_accumulated_context(results)
    print(f'  [RequirementsExtractor] Contexto acumulado: {len(acc):,} chars ({acc.count(chr(10))} linhas de IDs)')
    results['requirements'], _ = _p2_run_subagent(
        'RequirementsExtractor',
        _p2_with_context(_p2_prompt_requirements(sec_technical, sec_functional), acc),
        RequirementsData)

    # 10. MeasuresExtractor (sees: everything)
    acc = _p2_build_accumulated_context(results)
    results['measures'], _ = _p2_run_subagent(
        'MeasuresExtractor',
        _p2_with_context(_p2_prompt_measures(sec_technical), acc),
        MeasuresData)

    # 11. SpecificationExtractor (sees: everything — cross-cutting)
    trace_ctx = _p2_build_traceability_context(results)
    results['specification'], _ = _p2_run_subagent(
        'SpecificationExtractor', _p2_prompt_specification(trace_ctx), SpecificationData)

    # ── Check failures ──
    failed = [k for k, v in results.items() if v is None]
    if failed:
        msg = f'Failed sub-agents: {failed}'
        print(f'  ⚠ {msg}')
        log_error(state, 'P2_MissionAgent', msg)
        return {'pipeline_status': 'ERROR_AT_P2', 'errors': [{'agent': 'P2', 'error': msg}]}

    # ── Backfill: use refine_links from SpecificationExtractor to fill direct fields ──
    # Fixes a timing gap: CapabilityExtractor runs before Goals exist,
    # so supports_goal_ids ends up empty. The SpecificationExtractor (last sub-agent)
    # has the full view and captures the links correctly.
    spec_obj = results['specification']
    if spec_obj:
        refine_links = spec_obj.refine_links if hasattr(spec_obj, 'refine_links') else []
        # 1. Backfill Capability.supports_goal_ids from capability_refines_goal links
        cap_to_goals = {}
        for rl in refine_links:
            if rl.relationship == 'capability_refines_goal':
                cap_to_goals.setdefault(rl.source_id, []).append(rl.target_id)
        cap_obj = results['capability']
        if cap_obj and cap_to_goals:
            backfilled_caps = 0
            for cap in cap_obj.capabilities:
                if not cap.supports_goal_ids and cap.id in cap_to_goals:
                    cap.supports_goal_ids = cap_to_goals[cap.id]
                    backfilled_caps += 1
            if backfilled_caps:
                print(f'  ✅ Backfill: {backfilled_caps} capabilities got supports_goal_ids from SpecificationExtractor')

        # 2. Backfill Goal.refines_shn_ids from goal_refines_stakeholder_need links
        goal_to_shns = {}
        for rl in refine_links:
            if rl.relationship == 'goal_refines_stakeholder_need':
                goal_to_shns.setdefault(rl.source_id, []).append(rl.target_id)
        mission_obj = results['mission']
        if mission_obj and goal_to_shns:
            backfilled_goals = 0
            for goal in mission_obj.goals:
                if not goal.refines_shn_ids and goal.id in goal_to_shns:
                    goal.refines_shn_ids = goal_to_shns[goal.id]
                    backfilled_goals += 1
            if backfilled_goals:
                print(f'  ✅ Backfill: {backfilled_goals} goals got refines_shn_ids from SpecificationExtractor')

    # ── Consolidate JSON ──
    consolidated = MissionAgentOutput(
        stakeholder_data=results['stakeholder'],
        capability_data=results['capability'],
        mission_data=results['mission'],
        context_data=results['context'],
        operation_data=results['operation'],
        function_data=results['function'],
        logical_data=results['logical'],
        technical_data=results['technical'],
        requirements_data=results['requirements'],
        measures_data=results['measures'],
        specification_data=results['specification'],
    )
    mission_json = consolidated.model_dump()

    # Save to Drive
    out_dir = MISSION_AGENT_DIR / f'run_{state["run_timestamp"]}'
    out_dir.mkdir(parents=True, exist_ok=True)
    json_path = out_dir / f'mission_agent_output_{state["run_timestamp"]}.json'
    with open(json_path, 'w', encoding='utf-8') as f:
        _json.dump(mission_json, f, indent=2, ensure_ascii=False)
    print(f'  Consolidated saved: {json_path} ({json_path.stat().st_size:,} bytes)')

    # Item count
    total_items = sum(
        len(v) if isinstance(v, list) else 1
        for section in mission_json.values()
        if isinstance(section, dict)
        for v in section.values()
        if isinstance(v, (list, dict))
    )

    elapsed = time.time() - t0
    log_cost(state, 'P2_MissionAgent', estimate_cost(len(md)//4*11, len(_json.dumps(mission_json))//4), elapsed)

    print(f'  P2 v2 finished: {total_items} items, {len(_json.dumps(mission_json)):,} chars in {elapsed:.1f}s')
    print(f'  Failures: {len(failed)}')

    return {'mission_json': mission_json}

# ── Verification ───────────────────────────────────────────
print(f'P1 defined: run_p1(state) → mission_markdown, layer_markdowns')
print(f'  max_tokens: adaptive to document size')
print(f'P2 defined: run_p2(state) → mission_json')
print(f'  Prompts: 11 sub-agents')
print(f'Block B complete. Ready for Block C (P3 + P4). ✅')




---

## Block C — P3 (TemplateGeneratorClass) and P4 (RefinementAgent)


### Cell 6 — P3 Jinja2 Templates and TemplateGeneratorClass + run_p3()

In [ ]:
# ============================================================
# Master Orchestrator
# Cell 6 — P3 (TemplateGeneratorClass)
# ============================================================

from collections import defaultdict
from jinja2 import Environment, BaseLoader, StrictUndefined


# ── Utility functions for SysML v2 names ───────────────────

def to_sysml_id(name: str) -> str:
    """Converts a name into a valid SysML v2 identifier (PascalCase)."""
    if re.match(r'^[A-Za-z][A-Za-z0-9]*$', name):
        return name[0].upper() + name[1:] if name else 'Unnamed'
    clean = re.sub(r'[^a-zA-Z0-9\s\-_]', '', name)
    words = re.split(r'[\s\-_]+', clean)
    parts = [w[0].upper() + w[1:] for w in words if w]
    result = ''.join(parts)
    if result and not result[0].isalpha():
        result = 'X' + result
    return result or 'Unnamed'


def to_camel_case(name: str) -> str:
    """Converts a name to camelCase. Handles leading acronyms (D20)."""
    pascal = to_sysml_id(name)
    if not pascal:
        return 'unnamed'
    match = re.match(r'^([A-Z]+)', pascal)
    if match:
        prefix = match.group(1)
        rest = pascal[len(prefix):]
        if rest and len(prefix) > 1:
            return prefix[:-1].lower() + prefix[-1] + rest
        elif rest:
            return prefix.lower() + rest
        else:
            return prefix.lower()
    return pascal[0].lower() + pascal[1:]


def safe_doc(text: str, max_len: int = 500) -> str:
    """Sanitises text for doc /* ... */ blocks."""
    if not text:
        return 'TBD'
    clean = text.replace('*/', '* /').replace('/*', '/ *')
    if len(clean) > max_len:
        clean = clean[:max_len-3] + '...'
    return clean


# ── Traceability maps ──────────────────────────────────────

def build_id_to_name_map(data: dict) -> dict:
    """Builds the global ID -> SysML name map."""
    id_map = {}
    for s in data['stakeholder_data']['stakeholders']:
        id_map[s['id']] = to_sysml_id(s['name'])
    for sn in data['stakeholder_data']['stakeholder_needs']:
        id_map[sn['id']] = to_sysml_id(sn['name'])
    for c in data['capability_data']['capabilities']:
        id_map[c['id']] = to_sysml_id(c['name'])
    for g in data['mission_data']['goals']:
        id_map[g['id']] = to_sysml_id(g['name'])
    for p in data['mission_data']['phases']:
        id_map[p['id']] = to_sysml_id(p['name'])
    for o in data['operation_data']['operations']:
        id_map[o['id']] = to_sysml_id(o['name'])
    for fn in data['function_data']['functions']:
        id_map[fn['id']] = to_sysml_id(fn['name'])
    for lc in data['logical_data']['logical_components']:
        id_map[lc['id']] = to_sysml_id(lc['name'])
    for tc in data['technical_data']['technical_components']:
        id_map[tc['id']] = to_sysml_id(tc['name'])
    for mr in data['requirements_data']['mission_requirements']:
        desc = mr.get('descriptive_name', '').strip()
        if desc:
            name = to_sysml_id(desc)
            if not name.endswith('Requirement'):
                name += 'Requirement'
        else:
            name = to_sysml_id(mr['id'].replace('-', '')) + 'Requirement'
        id_map[mr['id']] = name
    for fr in data['requirements_data']['functional_requirements']:
        desc = fr.get('descriptive_name', '').strip()
        if desc:
            name = to_sysml_id(desc)
            if not name.endswith('Requirement'):
                name += 'Requirement'
        else:
            name = to_sysml_id(fr['id'].replace('-', '')) + 'Requirement'
        id_map[fr['id']] = name
    for tr in data['requirements_data']['technical_requirements']:
        desc = tr.get('descriptive_name', '').strip()
        if desc:
            name = to_sysml_id(desc)
            if not name.endswith('Requirement'):
                name += 'Requirement'
        else:
            name = to_sysml_id(tr['id'].replace('-', '')) + 'Requirement'
        id_map[tr['id']] = name
    for m in data['measures_data'].get('measures_of_success', []):
        id_map[m['id']] = to_sysml_id(m['name'])
    for m in data['measures_data'].get('measures_of_effectiveness', []):
        id_map[m['id']] = to_sysml_id(m['name'])
    for m in data['measures_data'].get('measures_of_performance', []):
        id_map[m['id']] = to_sysml_id(m['name'])
    return id_map


def build_refine_map(data: dict) -> dict:
    rmap = defaultdict(list)
    for rl in data['specification_data']['refine_links']:
        rmap[rl['relationship']].append((rl['source_id'], rl['target_id']))
    return dict(rmap)


def build_cap_to_shn_transitive(data: dict) -> dict:
    cap_to_goal = defaultdict(list)
    goal_to_shn = defaultdict(list)
    for rl in data['specification_data']['refine_links']:
        if rl['relationship'] == 'capability_refines_goal':
            cap_to_goal[rl['source_id']].append(rl['target_id'])
        elif rl['relationship'] == 'goal_refines_stakeholder_need':
            goal_to_shn[rl['source_id']].append(rl['target_id'])
    result = {}
    for cap_id, goal_ids in cap_to_goal.items():
        shns = set()
        for gid in goal_ids:
            for shn_id in goal_to_shn.get(gid, []):
                shns.add(shn_id)
        result[cap_id] = sorted(shns)
    return result


# ── Jinja2 templates (calibrated against Apollo 11: syntactic patterns extracted from the packages) ─────

# ── CoSMA Framework Packages (adapted from Helle & Schramm, 2026) ────
# Original: github.com/airbus/apollo-11-sysml-v2
# License: Mozilla Public License v2.0
# Modifications: Domain-specific elements generalized for mission-agnostic use.
#   - Astronaut :> Person → Operator :> Person
#   - MannedSpaceMission → CrewedMission
#   - Added aviation/maritime/ISR units (e.g. knot, nmi, arcmin, arcsec, fathom)

COSMA_PACKAGE = """\
//
// Original Copyright (c) 2026 AIRBUS and its affiliates.
// This Source Code Form is subject to the terms of the Mozilla Public
// License, v. 2.0. If a copy of the MPL was not distributed with this
// file, You can obtain one at http://mozilla.org/MPL/2.0/.
//
// Modified for mission-agnostic multi-agent pipeline.
// Changes: Astronaut→Operator, MannedSpaceMission→CrewedMission.
//

library package CoSMAPackage {

\tprivate import NumericalFunctions::*;
\tprivate import States::*;
\tprivate import ISQ::*;
\tprivate import SI::*;
\tprivate import MeasurementReferences::*;
\tprivate import Quantities::*;
\tprivate import Base::DataValue;
\tprivate import Metaobjects::*;
\tprivate import CoSMAQuantitiesAndUnitsPackage::*;

\tpart def MassedThing {
\t\tattribute mass :> ISQ::mass;
\t\tattribute totalMass :> ISQ::mass;
\t}

\tpart def SimpleThing :> MassedThing {
\t\tattribute redefines totalMass = mass;
\t}

\tpart def CompositeThing :> MassedThing {
\t\tpart subcomponents: MassedThing[*];
\t\t:>> totalMass = mass + sum(subcomponents.totalMass);
\t}

\tabstract action def Operation;

\tabstract action def Function {
\t\taction subfunctions[*] : Function :>> subactions;
\t}

\tabstract part def SystemOfInterest;
\tabstract part def LogicalComponent;

\tabstract part def TechnicalComponent {
\t\tattribute failureRate :> ISQ::frequency;
\t}

\tabstract part def SoftwareComponent :> TechnicalComponent;
\tabstract part def HardwareComponent :> TechnicalComponent, SimpleThing;

\tabstract part def PowerProvider :> HardwareComponent {
\t\tattribute powerGenerated :> ISQ::power default 0 [W];
\t}

\tabstract part def PowerConsumer :> HardwareComponent {
\t\tattribute powerLoad :> ISQ::power;
\t}

\tpart def Program {
\t\tpart missions[1..*] : Mission;
\t}

\tstate def Phase :> StateAction;

\tpart def Capability;

\tpart def Operator :> Person;

\tabstract part def CrewedMission :> Mission {
\t\tpart crew[1..*] : Operator;
\t}

\tabstract part def Mission {
\t\trequirement goals[1..*] : Goal;
\t\tpart system : System;
\t\tpart context : Context;
\t\texhibit state phases : StateAction;
\t\tpart requiredCapabilites[0..*];
\t\tabstract connection capabilityToGoals[*] : CapabilityToGoalDerivation;
\t\tattribute plannedDuration :> ISQ::duration;

\t\tattribute researchAndDevelopmentCost :> currency;
\t\tattribute manufacturingCost :> currency;
\t\tattribute operationsCost :> currency;
\t\tattribute personnelCost :> currency;
\t}

\tabstract connection def CapabilityToGoalDerivation {
\t\tref part capability[1] : Capability :> participant;
\t\tref requirement goals[1..*] : Goal :> participant;
\t}

\trequirement def Goal;

\titem def Concern;

\tenum def StakeholderInfluenceKind {
\t\tenum direct;
\t\tenum indirect;
\t}

\tenum def StakeholderInfluenceLevel {
\t\tenum high;
\t\tenum medium;
\t\tenum low;
\t}

\tpart def Person;

\trequirement def StakeholderNeed;

\tpart def Stakeholder {
\t\tattribute influenceLevel: StakeholderInfluenceLevel;
\t\tattribute influenceKind: StakeholderInfluenceKind;
\t\titem concerns[*]: Concern;
\t\trequirement needs[*]: StakeholderNeed;
\t}

\tpart def Context :> System {
\t\tpart externalSystems[*] : System;
\t}

\tpart def System :> CompositeThing {

\t}

\tpart def SystemOfSystems :> System {
\t\tpart constituentSystems[*] : ConstituentSystem;
\t}

\tpart def ConstituentSystem :> System {

\t}

}
"""

COSMA_QUANTITIES_AND_UNITS_PACKAGE = """\
//
// Original Copyright (c) 2026 AIRBUS and its affiliates.
// This Source Code Form is subject to the terms of the Mozilla Public
// License, v. 2.0. If a copy of the MPL was not distributed with this
// file, You can obtain one at http://mozilla.org/MPL/2.0/.
//
// Modified for mission-agnostic multi-agent pipeline.
// Changes: Added aviation/maritime/ISR domain units (knot, nmi, arcmin, arcsec, fathom).
//

library package CoSMAQuantitiesAndUnitsPackage {

\tprivate import NumericalFunctions::*;
\tprivate import ISQ::*;
\tprivate import SI::*;
\tprivate import MeasurementReferences::*;
\tprivate import Quantities::*;
\tprivate import Base::DataValue;

\tattribute def RatioValue :> MeasurementReferences::DimensionOneValue {
\t\tdoc /* Quantity of dimension one value that represents a ratio */
\t\tattribute :>> num: ScalarValues::Real;
\t\tattribute :>> mRef: RatioUnit;
\t}

\tattribute ratio : RatioValue nonunique :> Quantities::scalarQuantities;
\tattribute def RatioUnit :> MeasurementReferences::DimensionOneUnit;

\tattribute <'%'> 'per cent' : RatioUnit {
\t\tattribute :>> unitConversion {
\t\t\tattribute :>> conversionFactor = RationalFunctions::rat(1, 100);
\t\t}
\t}

\tattribute <'m³⋅s⁻²'> 'metre to the power 3 second to the power minus 2' : SimpleUnit;

\t// --- SI-derived units (from original CoSMA) ---
\tattribute <kN> kilonewton : ForceUnit { :>> unitConversion: ConversionByPrefix { :>> prefix = kilo; :>> referenceUnit = N; } }
\tattribute <kPa> kilopascal : PressureUnit { :>> unitConversion: ConversionByPrefix { :>> prefix = kilo; :>> referenceUnit = Pa; } }
\tattribute <MN> meganewton : ForceUnit { :>> unitConversion: ConversionByPrefix { :>> prefix = mega; :>> referenceUnit = N; } }
\tattribute <gn> standardgravity : AccelerationUnit {:>> unitConversion: ConversionByConvention { :>> referenceUnit = 'm⋅s⁻²'; :>> conversionFactor = 9.80665; :>> isExact = false; }}
\tattribute <fps> framespersecond : FrequencyUnit;
\tattribute <kB> kilobyte : StorageCapacityUnit { :>> unitConversion: ConversionByPrefix { :>> prefix = kilo; :>> referenceUnit = B; } }
\tattribute <yr> year : TimeUnit {:>> unitConversion: ConversionByConvention { :>> referenceUnit = 's'; :>> conversionFactor = 31536000; :>> isExact = false; }}

\t// --- Aviation and maritime domain units ---
\tattribute <kn> knot : SpeedUnit {:>> unitConversion: ConversionByConvention { :>> referenceUnit = 'm/s'; :>> conversionFactor = 0.514444; :>> isExact = false; }}
\tattribute <nmi> nauticalmile : LengthUnit {:>> unitConversion: ConversionByConvention { :>> referenceUnit = m; :>> conversionFactor = 1852; :>> isExact = true; }}

\t// --- Sensor and ISR domain units ---
\tattribute <'arcmin'> arcminute : SimpleUnit {:>> unitConversion: ConversionByConvention { :>> referenceUnit = rad; :>> conversionFactor = 2.908882E-4; :>> isExact = false; }}
\tattribute <'arcsec'> arcsecond : SimpleUnit {:>> unitConversion: ConversionByConvention { :>> referenceUnit = rad; :>> conversionFactor = 4.848137E-6; :>> isExact = false; }}

\t// --- Maritime domain units ---
\tattribute <ftm> fathom : LengthUnit {:>> unitConversion: ConversionByConvention { :>> referenceUnit = m; :>> conversionFactor = 1.8288; :>> isExact = true; }}

\t// --- Currency ---
\tattribute def CurrencyUnit :> SimpleUnit {
\t\tprivate attribute currencyPF: QuantityPowerFactor[1] { :>> quantity = Currency; :>> exponent = 1; }
\t\tattribute :>> quantityDimension { :>> quantityPowerFactors = currencyPF; }
\t}

\tattribute def CurrencyValue :> ScalarQuantityValue {
\t\tdoc
\t\t/*
\t\t* application domain: generic
\t\t* name: Currency
\t\t* measurement unit(s): $
\t\t* tensor order: 0
\t\t*/
\t\tattribute :>> num: Real;
\t\tattribute :>> mRef: CurrencyUnit[1];
\t}

\tattribute Currency: CurrencyValue[1];
\tattribute currency: CurrencyValue[*] nonunique :> scalarQuantities;
\tattribute <'$'> dollar : CurrencyUnit;
\tattribute <'€'> euro : CurrencyUnit;

\t// --- Rocket engineering (retained from original CoSMA) ---
\tattribute def SpecificImpulseValue :> ScalarQuantityValue {
\t\tdoc
\t\t/*
\t\t* application domain: rocket engineering
\t\t* name: Specific Impulse
\t\t* measurement unit(s): s
\t\t* tensor order: 0
\t\t*/
\t\tattribute :>> num: Real;
\t\tattribute :>> mRef: DurationUnit[1];
\t}

\tattribute SpecificImpulse: SpecificImpulseValue[1];
\tattribute specificImpulse: SpecificImpulseValue[*] nonunique :> scalarQuantities;

\tcalc <ln> naturalLogarithm { in x: DataValue[1]; in y: DataValue[1]; return : DataValue[1]; }
}
"""

COSMA_VIEWS_PACKAGE = """\
//
// Original Copyright (c) 2026 AIRBUS and its affiliates.
// This Source Code Form is subject to the terms of the Mozilla Public
// License, v. 2.0. If a copy of the MPL was not distributed with this
// file, You can obtain one at http://mozilla.org/MPL/2.0/.
//

library package CoSMAViewsPackage {

\tview RequirementsTableView {
\t\t//expose Requirements::MissionRequirementsPackage::*;
\t\t//filter hastype SysML::Systems::RequirementDefinition;
\t}
}
"""

# Index adapted CoSMA packages in RAG corpus
index_extra_documents({
    'CoSMAPackage_adapted': COSMA_PACKAGE,
    'CoSMAQuantitiesAndUnitsPackage_adapted': COSMA_QUANTITIES_AND_UNITS_PACKAGE,
    'CoSMAViewsPackage_adapted': COSMA_VIEWS_PACKAGE,
}, layer='framework')

TEMPLATES = {}

TEMPLATES['StakeholderPackage'] = '''
package StakeholderPackage {

\tprivate import CoSMAPackage::*;
\tprivate import StakeholderNeedsPackage::*;
{% for td in type_defs %}
\tpart def {{ td }} :> Stakeholder;
{% endfor %}
{% for sh in stakeholders %}
\tpart {{ sh.instance_name }} : {{ sh.type_name }} {
\t\t:> influenceLevel = StakeholderInfluenceLevel::{{ sh.influence_level }};
\t\t:> influenceKind = StakeholderInfluenceKind::{{ sh.influence_kind }};
{% for c in sh.concerns %}
\t\titem concern{{ loop.index }} : Concern {
\t\t\tdoc /* {{ c.text }} */
\t\t} :> concerns;
{% endfor %}
{% for need_id in sh.need_ids %}
\t\trequirement {{ need_id | resolve_id | camel }} : {{ need_id | resolve_id }} :> needs;
{% endfor %}
\t}
{% endfor %}
}
'''

TEMPLATES['StakeholderNeedsPackage'] = '''
package StakeholderNeedsPackage {

\tprivate import ModelingMetadata::*;
\tprivate import CoSMAPackage::*;
{% for sn in stakeholder_needs %}
\trequirement def <'{{ sn.id }}'> {{ sn.name }} :> StakeholderNeed {
\t\tdoc /* {{ sn.text }} */
{% if sn.rationale %}
\t\t@Rationale {
\t\t\ttext = "{{ sn.rationale }}";
\t\t}
{% endif %}
\t}
{% endfor %}
}
'''

TEMPLATES['CapabilitiesPackage'] = '''
package CapabilitiesPackage {

\tprivate import CoSMAPackage::*;
\tprivate import StakeholderNeedsPackage::*;
\tprivate import ModelingMetadata::*;
{% for cap in capabilities %}
\tpart def {{ cap.name }} :> Capability {
\t\tdoc /* {{ cap.description }} */
{% for shn_id in cap.refines_shn_ids %}
\t\t#refinement dependency {{ cap.name }} to StakeholderNeedsPackage::'{{ shn_id }}';
{% endfor %}
\t}
{% endfor %}
}
'''

TEMPLATES['MissionPackage'] = '''
package MissionPackage {

\tprivate import CoSMAPackage::*;
\tprivate import MissionPhasesPackage::*;
\tprivate import CapabilitiesPackage::*;
\tprivate import SystemPackage::*;
\tprivate import FunctionsPackage::*;
\tprivate import ContextPackage::*;
\tprivate import CoSMAQuantitiesAndUnitsPackage::*;
\tprivate import SI::*;
\tprivate import ISQ::*;

\tpart def {{ mission.name }} :> Mission {
\t\tdoc /* {{ mission.description }} */

\t\tpart {{ mission.system_instance }} : {{ mission.system_type }} :> system;
\t\tpart {{ mission.context_instance }} : {{ mission.context_type }} :> context;

\t\tperform action {{ mission.top_function_instance }} : {{ mission.top_function_type }};
{% for goal in goals %}
\t\trequirement {{ goal.instance_name }} : Goal {
\t\t\tdoc /* {{ goal.description }} */
\t\t} :> goals;
{% endfor %}
{% for cap in capability_instances %}
\t\tpart {{ cap.instance_name }} : {{ cap.type_name }} :> requiredCapabilites;
{% endfor %}
{% for conn in capability_goal_connections %}
\t\tconnection : CapabilityToGoalDerivation {
\t\t\tend capa :>> {{ conn.cap_instance }};
{% for goal_end in conn.goal_ends %}
\t\t\tend {{ goal_end.end_name }} :>> {{ goal_end.goal_instance }};
{% endfor %}
\t\t} :> capabilityToGoals;
{% endfor %}

\t\texhibit state {{ mission.phases_name }} {

\t\t\tstate initial : Initial;
{% for phase in phases %}
\t\t\tstate {{ phase.instance_name }} : {{ phase.type_name }};
{% endfor %}
\t\t\tstate final : MissionComplete;

\t\t\ttransition first initial then {{ phases[0].instance_name }};
{% for phase in phases %}
{% if phase.next_phase_ids %}
\t\t\ttransition first {{ phase.instance_name }} accept {{ phase.type_name }}CompletedNotification then {{ phase.next_instance_name }};
{% else %}
\t\t\ttransition first {{ phase.instance_name }} accept {{ phase.type_name }}CompletedNotification then final;
{% endif %}
{% endfor %}
\t\t} :>> phases;
\t}
}
'''

TEMPLATES['ContextPackage'] = '''
package ContextPackage {

\tprivate import CoSMAPackage::*;
\tprivate import LogicalComponentsPackage::*;
\tprivate import SI::*;
\tprivate import ISQ::*;
\tprivate import ScalarValues::*;
\tprivate import CoSMAQuantitiesAndUnitsPackage::*;
{% for ext in external_systems %}
\tpart def {{ ext.type_name }} :> System {
\t\tdoc /* {{ ext.description }} */
\t\t/* TODO[P4]: add quantitative attributes (coordinates, ranges, frequencies, capacities) from doc */
\t}
{% endfor %}

\tpart def {{ context_name }} :> Context {
\t\tdoc /* {{ context_description }} */
{% for ext in external_systems %}
\t\tpart {{ ext.instance_name }} : {{ ext.type_name }} :> externalSystems;
{% endfor %}
\t}
}
'''

TEMPLATES['MissionPhasesPackage'] = '''
package MissionPhasesPackage {

\tprivate import CoSMAPackage::*;
\tprivate import OperationsPackage::*;

\t// --- Item Definitions (Completion Notifications) ---
{% for phase in phases %}
\titem def {{ phase.type_name }}CompletedNotification;
{% endfor %}

\t// --- State Definitions ---
{% for phase in phases %}
\tstate def {{ phase.type_name }} :> Phase {
{% if phase.operations %}
\t\tdo action {{ phase.ops_block_name }} {
\t\t\tfirst start;
{% for op in phase.operations %}
\t\t\tthen action {{ op.instance_name }} : {{ op.type_name }};
{% endfor %}
\t\t\tthen done;
\t\t}
{% endif %}
\t}
{% endfor %}

\t// Special states
\tstate def Initial;
\tstate def MissionComplete;
}
'''

TEMPLATES['OperationsPackage'] = '''
package OperationsPackage {

\tprivate import CoSMAPackage::*;
{% for op in operations %}
\taction def {{ op.type_name }} :> Operation {
\t\tdoc /* {{ op.description }} */
\t}
{% endfor %}
}
'''

TEMPLATES['FunctionsPackage'] = '''
package FunctionsPackage {

\tprivate import MissionPhasesPackage::*;
\tprivate import CoSMAPackage::*;
\tprivate import ScalarValues::*;
\tprivate import ModelingMetadata::*;
\tprivate import MissionRequirementsPackage::*;

\t// --- Domain Item Definitions ---
{% for item in item_defs %}
\titem def {{ item.name }} { doc /* {{ item.description }} */ }
{% endfor %}

\t// ===================================================================
\t//  Level 4: Leaf-Level Function Definitions
\t//  NOTE: These functions refine the abstract Operations from OperationsPackage.
\t// ===================================================================
{% for fn in leaf_functions %}
\taction def {{ fn.type_name }} :> Function {
{% for inp in fn.inputs %}
\t\tin {{ inp.name }} : {{ inp.type_name }};
{% endfor %}
{% for out in fn.outputs %}
\t\tout {{ out.name }} : {{ out.type_name }};
{% endfor %}
\t\tdoc /* {{ fn.description }} */
{% for ref_op in fn.refines_ops %}
\t\t#refinement dependency {{ fn.type_name }} to OperationsPackage::{{ ref_op }};
{% endfor %}
\t}
{% endfor %}

\t// ===================================================================
\t//  Level 1-3: Orchestrating Functions (compose Leaf-Level Functions)
\t// ===================================================================
{% for fn in orchestrating_functions %}
\taction def {{ fn.type_name }} :> Function {
{% for inp in fn.inputs %}
\t\tin {{ inp.name }} : {{ inp.type_name }};
{% endfor %}
{% for out in fn.outputs %}
\t\tout {{ out.name }} : {{ out.type_name }};
{% endfor %}
\t\tdoc /* {{ fn.description }} */
{% for sub in fn.sub_actions %}
\t\taction {{ sub.instance_name }} : {{ sub.type_name }};
{% endfor %}
\t}
{% endfor %}
}
'''

TEMPLATES['LogicalComponentsPackage'] = '''
package LogicalComponentsPackage {

\tprivate import CoSMAPackage::*;
\tprivate import FunctionsPackage::*;
{% for lc in logical_components %}
\tpart def {{ lc.type_name }} :> LogicalComponent {
\t\tdoc /* {{ lc.description }} */
{% for fn in lc.performs %}
\t\tperform action {{ fn.instance_name }} : {{ fn.type_name }};
{% endfor %}
\t}
{% endfor %}
}
'''

TEMPLATES['SystemPackage'] = '''
package SystemPackage {

\tprivate import CoSMAPackage::*;
\tprivate import LogicalComponentsPackage::*;
\tprivate import TechnicalComponentsPackage::*;
\tprivate import TechnicalPortsPackage::*;
\tprivate import TechnicalIndividualsPackage::*;

\tpart def {{ sos.type_name }} :> SystemOfSystems {
\t\tdoc /* {{ sos.description }} */
{% for cs in sos.constituents %}
\t\tpart {{ cs.instance_name }} : {{ cs.type_name }} :> constituentSystems;
{% endfor %}
\t}
}
'''

TEMPLATES['TechnicalComponentsPackage'] = '''
package TechnicalComponentsPackage {

\tprivate import CoSMAPackage::*;
\tprivate import CoSMAQuantitiesAndUnitsPackage::*;
\tprivate import ShapeItems::*;
\tprivate import SI::*;
\tprivate import ISQ::*;
\tprivate import LogicalComponentsPackage::*;
\tprivate import TechnicalPortsPackage::*;
{% for tc in technical_components %}
\tpart def {{ tc.type_name }} :> {{ tc.basetype }} {
\t\tdoc /* {{ tc.description }} */
{% for spec in tc.specifications %}
\t\t/* TODO[P4]: attribute {{ spec.name }} = {{ spec.value }} [{{ spec.unit }}]; */
{% endfor %}
{% for port in tc.ports %}
\t\tport {{ port.instance_name }} : {{ port.type_name }};
{% endfor %}
\t}
{% endfor %}
}
'''

TEMPLATES['TechnicalPortsPackage'] = '''
package TechnicalPortsPackage {

\tprivate import SI::*;
\tprivate import ISQ::*;

\t// --- Flow Item Definitions ---
{% for fi in flow_item_defs %}
\titem def {{ fi.name }} { doc /* {{ fi.description }} */ }
{% endfor %}

\t// --- Port Definitions ---
{% for pd in port_defs %}
\tport def {{ pd.name }} {
\t\tdoc /* {{ pd.description }} */
{% for item in pd.port_items %}
\t\t{{ item.direction }} item {{ item.name }} : {{ item.type_name }};
{% endfor %}
\t}
{% endfor %}

\t// --- Interface Definitions ---
{% for idef in interface_defs %}
\tinterface def {{ idef.name }} {
\t\tdoc /* {{ idef.description }} */
\t\tend {{ idef.end1_name }} : {{ idef.end1_port }};
\t\tend {{ idef.end2_name }} : {{ idef.end2_port }};
\t}
{% endfor %}
}
'''

TEMPLATES['TechnicalIndividualsPackage'] = '''
package TechnicalIndividualsPackage {

\tprivate import CoSMAPackage::*;
\tprivate import TechnicalComponentsPackage::*;
\tprivate import CoSMAQuantitiesAndUnitsPackage::*;
\tprivate import SI::*;
\tprivate import ISQ::*;
{% for ti in individuals %}
\tindividual part def '{{ ti.display_name }}' :> {{ ti.type_name }} {
\t\tdoc /* {{ ti.description }} */
\t}
{% endfor %}
}
'''

TEMPLATES['MissionRequirementsPackage'] = '''
package MissionRequirementsPackage {

\tprivate import SI::*;
\tprivate import ISQ::*;
\tprivate import ScalarValues::*;
\tprivate import CoSMAPackage::*;
\tprivate import CoSMAQuantitiesAndUnitsPackage::*;
\tprivate import ModelingMetadata::*;
\tprivate import StakeholderNeedsPackage::*;
\tprivate import CapabilitiesPackage::*;
{% for mr in mission_requirements %}
\trequirement def <'{{ mr.id }}'> {{ mr.name }} {
\t\tdoc /* {{ mr.text }} */
{% if mr.rationale %}
\t\t@Rationale {
\t\t\ttext = "{{ mr.rationale }}";
\t\t}
{% endif %}
\t\t/* TODO[P4]: attributes and require constraint */
{% for ref in mr.refines_cap %}
\t\t#refinement dependency '{{ mr.id }}' to CapabilitiesPackage::{{ ref }};
{% endfor %}
\t}
{% endfor %}
}
'''

TEMPLATES['FunctionalRequirementsPackage'] = '''
package FunctionalRequirementsPackage {

\tprivate import SI::*;
\tprivate import ISQ::*;
\tprivate import ScalarValues::*;
\tprivate import CoSMAPackage::*;
\tprivate import CoSMAQuantitiesAndUnitsPackage::*;
\tprivate import ModelingMetadata::*;
\tprivate import MissionRequirementsPackage::*;
{% for fr in functional_requirements %}
\trequirement def <'{{ fr.id }}'> {{ fr.name }} {
\t\tdoc /* {{ fr.text }} */
{% if fr.rationale %}
\t\t@Rationale {
\t\t\ttext = "{{ fr.rationale }}";
\t\t}
{% endif %}
\t\t/* TODO[P4]: attributes and require constraint */
{% for ref in fr.refines_mr %}
\t\t#refinement dependency '{{ fr.id }}' to MissionRequirementsPackage::'{{ ref }}';
{% endfor %}
\t}
{% endfor %}
}
'''

TEMPLATES['TechnicalRequirementsPackage'] = '''
package TechnicalRequirementsPackage {

\tprivate import SI::*;
\tprivate import ISQ::*;
\tprivate import ScalarValues::*;
\tprivate import CoSMAPackage::*;
\tprivate import CoSMAQuantitiesAndUnitsPackage::*;
\tprivate import ModelingMetadata::*;
\tprivate import FunctionalRequirementsPackage::*;
{% for tr in technical_requirements %}
\trequirement def <'{{ tr.id }}'> {{ tr.name }} {
\t\tdoc /* {{ tr.text }} */
{% if tr.rationale %}
\t\t@Rationale {
\t\t\ttext = "{{ tr.rationale }}";
\t\t}
{% endif %}
\t\t/* TODO[P4]: attributes and require constraint */
{% for ref in tr.refines_fr %}
\t\t#refinement dependency '{{ tr.id }}' to FunctionalRequirementsPackage::'{{ ref }}';
{% endfor %}
\t}
{% endfor %}
}
'''

TEMPLATES['MissionSpecificationPackage'] = '''
package MissionSpecificationPackage {

\tprivate import MissionRequirementsPackage::*;
\tprivate import MissionPackage::*;
\tprivate import MissionPhasesPackage::*;

\trequirement {{ spec_name }} {
\t\tdoc /* Mission requirements specification for {{ mission_name }} */
\t\tsubject {{ mission_instance }} : {{ mission_type }};
{% for mr in mission_requirements %}
\t\trequirement '{{ mr.id }}' : {{ mr.name }};
{% endfor %}
{% for sat in satisfy_links %}
\t\tsatisfy '{{ sat.req_id }}' by {{ sat.satisfied_by }};
{% endfor %}
\t}
}
'''

TEMPLATES['FunctionSpecificationPackage'] = '''
package FunctionSpecificationPackage {

\tprivate import FunctionalRequirementsPackage::*;
\tprivate import FunctionsPackage::*;

\trequirement {{ spec_name }} {
\t\tdoc /* Functional requirements specification */
\t\tsubject {{ function_instance }} : {{ function_type }};
{% for fr in functional_requirements %}
\t\trequirement '{{ fr.id }}' : {{ fr.name }};
{% endfor %}
{% for sat in satisfy_links %}
\t\tsatisfy '{{ sat.req_id }}' by {{ sat.satisfied_by }};
{% endfor %}
\t}
}
'''

TEMPLATES['SystemSpecificationPackage'] = '''
package SystemSpecificationPackage {

\tprivate import TechnicalRequirementsPackage::*;
\tprivate import SystemPackage::*;

\trequirement {{ spec_name }} {
\t\tdoc /* Technical requirements specification */
\t\tsubject {{ system_instance }} : {{ system_type }};
{% for tr in technical_requirements %}
\t\trequirement '{{ tr.id }}' : {{ tr.name }};
{% endfor %}
{% for sat in satisfy_links %}
\t\tsatisfy '{{ sat.req_id }}' by {{ sat.satisfied_by }};
{% endfor %}
\t}
}
'''

TEMPLATES['ProgramPackage'] = '''
package ProgramPackage {

\tprivate import MissionPackage::*;
\tprivate import CoSMAPackage::*;

\tpart def {{ program_name }} :> Program {
\t\tdoc /* {{ program_description }} */

\t\tpart {{ current_mission.instance }} : {{ current_mission.type }} :> missions;
{% for rm in related_missions %}
\t\tpart {{ rm.instance }} : Mission {
\t\t\tdoc /* {{ rm.description }} */
\t\t} :> missions;
{% endfor %}
\t}
}
'''

TEMPLATES['AnalysisPackage'] = '''
package AnalysisPackage {

\tprivate import MissionPackage::*;
\tprivate import CoSMAPackage::*;
\tprivate import CoSMAQuantitiesAndUnitsPackage::*;
\tprivate import CalculationsPackage::*;
\tprivate import ISQ::*;
\tprivate import SI::*;
\tprivate import ScalarValues::*;
\tprivate import TechnicalComponentsPackage::*;
\tprivate import MissionExecutionPackage::*;

\t// --- Measures of Success (MOS) — Analysis Definitions ---
{% for mos in measures_of_success %}
\tanalysis def {{ mos.name }}Analysis {
\t\tdoc /* {{ mos.description }} */
\t\tsubject missionSystem : {{ mos.subject_type }};
\t\tout {{ mos.out_name }} :> ScalarValues::Real;
\t\t/* TODO[P4]: bind out to calc def, e.g. out x :> type = calculateX(param1, param2); add assert constraint */
\t}

\tanalysis {{ mos.subject_instance }}{{ mos.name }}Analysis : {{ mos.name }}Analysis {
\t\tdoc /* Runs {{ mos.name }} analysis on the {{ mos.subject_type }} mission. */
\t\tsubject {{ mos.subject_instance }} : {{ mos.subject_type }};
\t}
{% endfor %}

\t// --- Measures of Effectiveness (MOE) — Analysis Definitions ---
{% for moe in measures_of_effectiveness %}
\tanalysis def {{ moe.name }}Analysis {
\t\tdoc /* {{ moe.description }} */
\t\tsubject missionSystem : {{ moe.subject_type }};
\t\tout {{ moe.out_name }} :> ScalarValues::Real;
\t\t/* TODO[P4]: bind out to calc def, e.g. out x :> type = calculateX(param1, param2); add assert constraint */
\t}

\tanalysis {{ moe.subject_instance }}{{ moe.name }}Analysis : {{ moe.name }}Analysis {
\t\tdoc /* Runs {{ moe.name }} analysis on the {{ moe.subject_type }} mission. */
\t\tsubject {{ moe.subject_instance }} : {{ moe.subject_type }};
\t}
{% endfor %}

\t// --- Measures of Performance (MOP) — Analysis Definitions ---
{% for mop in measures_of_performance %}
\tanalysis def {{ mop.name }}Analysis {
\t\tdoc /* {{ mop.description }} */
\t\tsubject missionSystem : {{ mop.subject_type }};
\t\tout {{ mop.out_name }} :> ScalarValues::Real;
\t\t/* TODO[P4]: bind out to calc def, e.g. out x :> type = calculateX(param1, param2); add assert constraint */
\t}

\tanalysis {{ mop.subject_instance }}{{ mop.name }}Analysis : {{ mop.name }}Analysis {
\t\tdoc /* Runs {{ mop.name }} analysis on the {{ mop.subject_type }} mission. */
\t\tsubject {{ mop.subject_instance }} : {{ mop.subject_type }};
\t}
{% endfor %}
}
'''

TEMPLATES['CalculationsPackage'] = '''
package CalculationsPackage {

\tprivate import ISQ::*;
\tprivate import SI::*;
\tprivate import ScalarValues::*;
\tprivate import CoSMAPackage::*;
\tprivate import ScalarFunctions::*;
\tprivate import CoSMAQuantitiesAndUnitsPackage::*;
\tprivate import TechnicalComponentsPackage::*;
\tprivate import NumericalFunctions::*;
\tprivate import CollectionFunctions::*;
\tprivate import ControlFunctions::*;
\tprivate import Parts::*;

\t// --- Calculation Definitions for Measures of Success ---
{% for mos in measures_of_success %}
\tcalc def calculate{{ mos.name }} {
\t\tdoc /* Calculates {{ mos.description }} */
{% if mos.formula %}
\t\t/* TODO[P4]: formula = {{ mos.formula }} */
{% for v in mos.variables %}
\t\t/* TODO[P4]: {{ v.role }} {{ v.name }} — {{ v.description }} ({{ v.unit }}) */
{% endfor %}
{% else %}
\t\t/* TODO[P4]: define in parameters, return type :> ISQ or ScalarValues, and formula */
{% endif %}
\t\treturn result :> ScalarValues::Real;
\t}
{% endfor %}

\t// --- Calculation Definitions for Measures of Effectiveness ---
{% for moe in measures_of_effectiveness %}
\tcalc def calculate{{ moe.name }} {
\t\tdoc /* Calculates {{ moe.description }} */
{% if moe.formula %}
\t\t/* TODO[P4]: formula = {{ moe.formula }} */
{% for v in moe.variables %}
\t\t/* TODO[P4]: {{ v.role }} {{ v.name }} — {{ v.description }} ({{ v.unit }}) */
{% endfor %}
{% else %}
\t\t/* TODO[P4]: define in parameters, return type :> ISQ or ScalarValues, and formula */
{% endif %}
\t\treturn result :> ScalarValues::Real;
\t}
{% endfor %}

\t// --- Calculation Definitions for Measures of Performance ---
{% for mop in measures_of_performance %}
\tcalc def calculate{{ mop.name }} {
\t\tdoc /* Calculates {{ mop.description }} */
{% if mop.formula %}
\t\t/* TODO[P4]: formula = {{ mop.formula }} */
{% for v in mop.variables %}
\t\t/* TODO[P4]: {{ v.role }} {{ v.name }} — {{ v.description }} ({{ v.unit }}) */
{% endfor %}
{% else %}
\t\t/* TODO[P4]: define in parameters, return type :> ISQ or ScalarValues, and formula */
{% endif %}
\t\treturn result :> ScalarValues::Real;
\t}
{% endfor %}
}
'''

TEMPLATES['MissionExecutionPackage'] = '''
package MissionExecutionPackage {

\tprivate import SI::*;
\tprivate import ISQ::*;
\tprivate import Time::*;
\tprivate import ScalarValues::*;
\tprivate import MissionPackage::*;
\tprivate import FunctionsPackage::*;
\tprivate import SystemPackage::*;
\tprivate import LogicalComponentsPackage::*;
\tprivate import ContextPackage::*;
\tprivate import OccurrenceFunctions::*;
\tprivate import TechnicalComponentsPackage::*;
\tprivate import CoSMAQuantitiesAndUnitsPackage::*;

\tindividual part def {{ mission.individual_type }} :> {{ mission.name }} {
{% for outcome in outcomes %}
\t\tattribute {{ outcome.attr_name }} : {{ outcome.attr_type }};
{% endfor %}
\t}

\tindividual part {{ mission.individual_instance }} : {{ mission.individual_type }} {
\t\tdoc /* A timeline representing the key events of the {{ mission.display_name }} mission. */
{% for ts in timeslices %}
{% if not loop.first %}
\t\tthen timeslice {{ ts.instance_name }} {
{% else %}
\t\ttimeslice {{ ts.instance_name }} {
{% endif %}
\t\t\tdoc /* {{ ts.description }} */
\t\t\tassert constraint { isDuring({{ mission.phases_ref }}.{{ ts.phase_instance }}) }
\t\t\t/* TODO[P4]: Add snapshots with system state, missionTime, and perform actions.
\t\t\t   Pattern per snapshot:
\t\t\t   snapshot atEventName :> {{ mission.system_instance }} {
\t\t\t       attribute missionTime :> ISQ::time = VALUE ['s'];
\t\t\t       part :>> constituentName { perform LogicalComponent::functionName; }
\t\t\t   }
\t\t\t*/
\t\t}
{% endfor %}
\t}
}
'''

TEMPLATES['RootModel'] = '''
package {{ model_name }} {

\t// CoSMA framework
\tprivate import CoSMAPackage::*;
\tprivate import CoSMAQuantitiesAndUnitsPackage::*;
\tprivate import CoSMAViewsPackage::*;

\t// Purpose layer
\tprivate import CapabilitiesPackage::*;
\tprivate import ContextPackage::*;
\tprivate import MissionPackage::*;
\tprivate import MissionPhasesPackage::*;
\tprivate import MissionSpecificationPackage::*;
\tprivate import StakeholderPackage::*;

\t// Operation layer
\tprivate import OperationsPackage::*;

\t// Function layer
\tprivate import FunctionsPackage::*;
\tprivate import FunctionSpecificationPackage::*;

\t// Logical layer
\tprivate import LogicalComponentsPackage::*;

\t// Technical layer
\tprivate import SystemPackage::*;
\tprivate import SystemSpecificationPackage::*;
\tprivate import TechnicalComponentsPackage::*;
\tprivate import TechnicalPortsPackage::*;
\tprivate import TechnicalIndividualsPackage::*;

\t// Program
\tprivate import ProgramPackage::*;

\t// Requirements
\tprivate import MissionRequirementsPackage::*;
\tprivate import FunctionalRequirementsPackage::*;
\tprivate import TechnicalRequirementsPackage::*;
\tprivate import StakeholderNeedsPackage::*;

\t// Analysis
\tprivate import AnalysisPackage::*;
\tprivate import CalculationsPackage::*;

\t// Execution
\tprivate import MissionExecutionPackage::*;
}
'''


# ── TemplateGeneratorClass ──────────────────────────────

class TemplateGeneratorClass:
    """Deterministic Jinja2 engine for generating SysML v2 skeletons.
    Calibrated against the 28 Apollo 11 packages.
    """

    def __init__(self, data: dict, id_map: dict, templates: dict):
        self.data = data
        self.id_map = id_map
        self.templates = templates
        self.results = {}
        self.refine_map = build_refine_map(data)
        self.cap_to_shn = build_cap_to_shn_transitive(data)
        self.env = Environment(
            loader=BaseLoader(), undefined=StrictUndefined,
            trim_blocks=True, lstrip_blocks=True,
        )
        self.env.filters['sysml_id'] = to_sysml_id
        self.env.filters['camel'] = to_camel_case
        self.env.filters['safe_doc'] = safe_doc
        self.env.filters['resolve_id'] = lambda x: self.id_map.get(x, to_sysml_id(x))

    def _render(self, template_name: str, context: dict) -> str:
        tmpl = self.env.from_string(self.templates[template_name])
        rendered = tmpl.render(**context)
        lines = rendered.split('\n')
        cleaned = []
        prev_blank = False
        for line in lines:
            is_blank = line.strip() == ''
            if is_blank and prev_blank:
                continue
            cleaned.append(line)
            prev_blank = is_blank
        return '\n'.join(cleaned).strip() + '\n'

    def _get_sos_type_name(self):
        """Returns SoS type name, disambiguated from Mission name if needed (D-purpose-4.7)."""
        sos_name = to_sysml_id(self.data['logical_data']['sos']['name'])
        mission_name = to_sysml_id(self.data['mission_data']['mission']['name'])
        if sos_name == mission_name:
            return sos_name + 'System'
        return sos_name

    def _prep_stakeholders(self):
        shs = []
        seen_types = set()
        type_defs = []
        for s in self.data['stakeholder_data']['stakeholders']:
            type_name = to_sysml_id(s.get('type_name', s['name']))
            if type_name not in seen_types:
                seen_types.add(type_name)
                type_defs.append(type_name)
            shs.append({
                'type_name': type_name,
                'instance_name': to_sysml_id(s['name']),
                'influence_level': str(s.get('influence_level', 'medium')).split('.')[-1],
                'influence_kind': str(s.get('influence_kind', 'direct')).split('.')[-1],
                'concerns': s.get('concerns', []),
                'need_ids': s.get('need_ids', []),
            })
        return {'stakeholders': shs, 'type_defs': type_defs}

    def _prep_stakeholder_needs(self):
        return {'stakeholder_needs': [
            {'id': sn['id'], 'name': to_sysml_id(sn['name']),
             'text': safe_doc(sn.get('text', 'TBD')),
             'rationale': sn.get('rationale', '').replace('\\', '\\\\').replace('"', '\\"').replace('\n', ' ')}
            for sn in self.data['stakeholder_data']['stakeholder_needs']
        ]}

    def _prep_capabilities(self):
        # --- Deterministic chain: CAP → GOAL (supports_goal_ids) → SHN (refines_shn_ids) ---
        goal_map = {g['id']: g for g in self.data['mission_data']['goals']}
        cap_to_shn_deterministic = {}
        for c in self.data['capability_data']['capabilities']:
            shn_ids = set()
            for goal_id in c.get('supports_goal_ids', []):
                goal = goal_map.get(goal_id)
                if goal:
                    for shn_id in goal.get('refines_shn_ids', []):
                        shn_ids.add(shn_id)
            cap_to_shn_deterministic[c['id']] = shn_ids

        caps = []
        for c in self.data['capability_data']['capabilities']:
            # Positive redundancy: union of deterministic chain + SpecificationExtractor links
            shn_from_spec = set(self.cap_to_shn.get(c['id'], []))
            shn_from_det = cap_to_shn_deterministic.get(c['id'], set())
            combined_shn_ids = sorted(shn_from_spec | shn_from_det)
            caps.append({
                'name': to_sysml_id(c['name']),
                'description': safe_doc(c.get('description', 'TBD')),
                'refines_shn_ids': combined_shn_ids,
            })
        return {'capabilities': caps}

    def _prep_operations(self):
        return [{'type_name': to_sysml_id(o['name']),
                 'instance_name': to_camel_case(o['name']),
                 'description': safe_doc(o.get('description', 'TBD')),
                 'id': o['id']} for o in self.data['operation_data']['operations']]

    def _prep_phases(self, ops_list):
        op_map = {o['id']: o for o in self.data['operation_data']['operations']}
        phase_id_map = {p['id']: p for p in self.data['mission_data']['phases']}
        phases = []
        for p in self.data['mission_data']['phases']:
            type_name = to_sysml_id(p['name'])
            phase_ops = []
            for op_id in p.get('operation_ids', []):
                if op_id in op_map:
                    op = op_map[op_id]
                    phase_ops.append({'type_name': to_sysml_id(op['name']),
                                     'instance_name': to_camel_case(op['name'])})
            next_ids = p.get('next_phase_ids', [])
            next_instance = ''
            if next_ids and next_ids[0] in phase_id_map:
                next_instance = to_camel_case(phase_id_map[next_ids[0]]['name'])
            phases.append({
                'type_name': type_name,
                'instance_name': to_camel_case(p['name']),
                'operations': phase_ops,
                'ops_block_name': to_camel_case(p['name']) + 'Operations',
                'next_phase_ids': next_ids,
                'next_instance_name': next_instance or 'final',
            })
        return phases

    def _prep_mission(self, phases):
        m = self.data['mission_data']['mission']
        mission_name = to_sysml_id(m['name'])
        sos_name = self._get_sos_type_name()
        ctx_name = to_sysml_id(self.data['context_data']['context']['name'])
        top_fn = self.data['function_data']['functions'][0]
        top_fn_name = to_sysml_id(top_fn['name'])
        goals = []
        goal_instance_map = {}
        for g in self.data['mission_data']['goals']:
            inst = to_camel_case(g['name'])
            goals.append({'instance_name': inst, 'description': safe_doc(g.get('description', 'TBD'))})
            goal_instance_map[g['id']] = inst
        cap_instances = []
        cap_instance_map = {}
        for cap in self.data['capability_data']['capabilities']:
            cap_name = to_sysml_id(cap['name'])
            inst = to_camel_case(cap['name'])
            cap_instances.append({'type_name': cap_name, 'instance_name': inst})
            cap_instance_map[cap['id']] = inst
        cap_goal_links = defaultdict(list)
        for rl in self.data['specification_data']['refine_links']:
            if rl['relationship'] == 'capability_refines_goal':
                cap_goal_links[rl['source_id']].append(rl['target_id'])
        connections = []
        for cap_id, goal_ids in cap_goal_links.items():
            if cap_id not in cap_instance_map:
                continue
            goal_ends = []
            for i, gid in enumerate(goal_ids):
                if gid not in goal_instance_map:
                    continue
                suffix = str(i+1) if len(goal_ids) > 1 else ''
                goal_ends.append({'end_name': f'goal{suffix}', 'goal_instance': goal_instance_map[gid]})
            if goal_ends:
                connections.append({'cap_instance': cap_instance_map[cap_id], 'goal_ends': goal_ends})
        return {
            'mission': {
                'name': mission_name, 'description': safe_doc(m.get('description', 'TBD')),
                'system_type': sos_name, 'system_instance': to_camel_case(sos_name),
                'context_type': ctx_name, 'context_instance': to_camel_case(ctx_name),
                'top_function_type': top_fn_name, 'top_function_instance': to_camel_case(top_fn_name),
                'phases_name': to_camel_case(mission_name) + 'Phases',
            },
            'goals': goals, 'phases': phases,
            'capability_instances': cap_instances, 'capability_goal_connections': connections,
        }

    def _prep_context(self):
        ctx = self.data['context_data']['context']
        ext_systems = [{'type_name': to_sysml_id(es.get('name', 'ExternalSystem')),
                        'instance_name': to_camel_case(es.get('name', 'externalSystem')),
                        'description': safe_doc(es.get('description', 'TBD'))}
                       for es in ctx.get('external_systems', [])]
        return {'context_name': to_sysml_id(ctx['name']),
                'context_description': safe_doc(ctx.get('scenario_purpose', 'TBD')),
                'external_systems': ext_systems}

    def _prep_functions(self):
        """Prepares function data with hierarchy (leaf vs orchestrating) and builds
        the function path map for FunctionSpecificationPackage satisfy links."""
        fns_raw = self.data['function_data']['functions']
        item_defs_raw = self.data['function_data'].get('item_defs', [])

        # --- Item definitions ---
        item_defs = [{'name': to_sysml_id(d.get('name', 'DataValue')),
                      'description': safe_doc(d.get('description', 'TBD'))}
                     for d in item_defs_raw]

        # --- Build hierarchy maps ---
        fn_by_id = {fn['id']: fn for fn in fns_raw}
        children_map = defaultdict(list)
        parent_map = {}
        for fn in fns_raw:
            pid = fn.get('parent_id')
            if pid:
                children_map[pid].append(fn)
                parent_map[fn['id']] = pid
        has_children_ids = set(children_map.keys())

        # --- Build function dot-notation path map (for FS-1 satisfy links) ---
        # Stored on self so _prep_specifications() can use it
        self._fn_path_map = {}
        for fn in fns_raw:
            parts = []
            cid = fn['id']
            while cid:
                parts.append(to_camel_case(fn_by_id[cid]['name']))
                cid = parent_map.get(cid)
            parts.reverse()
            dot_path = '.'.join(parts)
            self._fn_path_map[fn['id']] = dot_path
            self._fn_path_map[to_sysml_id(fn['name'])] = dot_path

        # --- Helper to build common fn data ---
        def _build_fn(fn):
            inputs = [{'name': to_camel_case(i.get('name', 'input')),
                       'type_name': to_sysml_id(i.get('type_desc', 'DataValue'))}
                      for i in fn.get('inputs', [])]
            outputs = [{'name': to_camel_case(o.get('name', 'output')),
                        'type_name': to_sysml_id(o.get('type_desc', 'DataValue'))}
                       for o in fn.get('outputs', [])]
            refines_ops = [self.id_map[op_id] for op_id in fn.get('refines_op_ids', [])
                           if op_id in self.id_map]
            return {
                'type_name': to_sysml_id(fn['name']),
                'description': safe_doc(fn.get('description', 'TBD')),
                'inputs': inputs,
                'outputs': outputs,
                'refines_ops': refines_ops,
            }

        # --- Separate leaf vs orchestrating ---
        leaf_functions = []
        orchestrating_functions = []
        for fn in fns_raw:
            fn_data = _build_fn(fn)
            if fn['id'] in has_children_ids:
                fn_data['sub_actions'] = [
                    {'instance_name': to_camel_case(c['name']),
                     'type_name': to_sysml_id(c['name'])}
                    for c in children_map[fn['id']]
                ]
                fn_data['refines_ops'] = []  # orchestrating functions don't refine operations
                orchestrating_functions.append(fn_data)
            else:
                leaf_functions.append(fn_data)

        return {
            'item_defs': item_defs,
            'leaf_functions': leaf_functions,
            'orchestrating_functions': orchestrating_functions,
        }

    def _prep_logical(self):
        # Build set of leaf function IDs (exclude parents/orchestrating)
        fns_raw = self.data.get('function_data', {}).get('functions', [])
        parent_ids = {fn.get('parent_id') for fn in fns_raw if fn.get('parent_id')}
        leaf_fn_ids = {fn['id'] for fn in fns_raw if fn['id'] not in parent_ids}

        lcs = []
        seen_fn_ids = set()
        for lc in self.data['logical_data']['logical_components']:
            performs = []
            seen_in_this_lc = set()
            for fn_id in lc.get('performs_fn_ids', []):
                # Filter: only leaf functions, no duplicates within same LC
                if fn_id not in leaf_fn_ids:
                    continue
                if fn_id in seen_in_this_lc:
                    continue
                if fn_id in self.id_map:
                    fn_name = self.id_map[fn_id]
                    performs.append({'type_name': fn_name, 'instance_name': to_camel_case(fn_name)})
                    seen_in_this_lc.add(fn_id)
            lcs.append({'type_name': to_sysml_id(lc['name']),
                        'description': safe_doc(lc.get('description', 'TBD')),
                        'performs': performs})
        return {'logical_components': lcs}

    def _prep_system(self):
        sos = self.data['logical_data']['sos']
        sos_type_name = self._get_sos_type_name()
        constituents = []
        for cs in sos.get('constituents', []):
            lc_name = self.id_map.get(cs.get('lc_id', ''), 'Unknown')
            constituents.append({'type_name': lc_name, 'instance_name': to_camel_case(lc_name)})
        return {'sos': {'type_name': sos_type_name,
                        'description': safe_doc(sos.get('description', 'TBD')),
                        'constituents': constituents}}

    def _prep_technical(self):
        # Build port-to-component map
        tc_ports_map = defaultdict(list)
        seen_ports_per_tc = defaultdict(set)
        for p in self.data['technical_data']['ports']:
            tc_id = p.get('owner_tc_id', '')
            if not tc_id:
                continue
            port_type = p.get('port_def_name') or (to_sysml_id(p['name']) + 'Port')
            if port_type in seen_ports_per_tc[tc_id]:
                continue
            seen_ports_per_tc[tc_id].add(port_type)
            tc_ports_map[tc_id].append({
                'instance_name': to_camel_case(p['name']) + 'Port',
                'type_name': port_type,
            })

        tcs = []
        for tc in self.data['technical_data']['technical_components']:
            has_parent = bool(tc.get('parent_tc_id'))
            basetype = self._classify_component_basetype(tc['name'], has_parent)
            specs = [{'name': sp.get('name', 'unknown'), 'value': sp.get('value', 'TBD'),
                      'unit': sp.get('unit', '')} for sp in tc.get('specifications', [])]
            tcs.append({'type_name': to_sysml_id(tc['name']),
                        'description': safe_doc(tc.get('description', 'TBD')),
                        'basetype': basetype,
                        'specifications': specs,
                        'ports': tc_ports_map.get(tc['id'], [])})
        return {'technical_components': tcs}

    @staticmethod
    def _classify_component_basetype(name: str, has_parent: bool = False) -> str:
        """Classifies a technical component into CoSMA base types.
        Primary signal: parent_tc_id (subsystem → HardwareComponent).
        Secondary signal: keyword heuristic for software/people.
        Default: System for top-level, HardwareComponent for subsystems.
        """
        nl = name.lower()
        # Software — always SoftwareComponent regardless of parent
        if any(kw in nl for kw in ('software', 'algorithm')):
            return 'SoftwareComponent'
        # People, organizations, management entities
        if any(kw in nl for kw in ('pilot', 'aircrew', 'crew', 'personnel',
                                    'management', 'program office', 'officer')):
            return 'SimpleThing'
        # Subsystems with a parent are HardwareComponent
        if has_parent:
            return 'HardwareComponent'
        # Top-level TCs without parent default to System
        return 'System'

    def _prep_ports(self):
        """Prepares TechnicalPortsPackage: flow item defs, port defs, interface defs.
        Supports new structured schema (port_defs, interface_defs) with fallback
        to legacy flat port list for backward compatibility."""
        td = self.data['technical_data']

        # --- Flow item defs ---
        flow_item_defs = [{'name': to_sysml_id(d.get('name', 'DataFlow')),
                           'description': safe_doc(d.get('description', 'TBD'))}
                          for d in td.get('port_flow_item_defs', [])]

# --- Port defs ---
        port_defs = []
        for pd in td.get('port_defs', []):
            items = [{'direction': item.get('direction', 'inout'),
                      'name': to_camel_case(item.get('name', 'data')),
                      'type_name': to_sysml_id(item.get('type_name', 'DataFlow'))}
                     for item in pd.get('items', [])]
            port_defs.append({
                'name': to_sysml_id(pd['name']),
                'description': safe_doc(pd.get('description', 'TBD')),
                'port_items': items,
            })

        # --- Interface defs ---
        interface_defs = []
        for idef in td.get('interface_defs', []):
            e1_port = to_sysml_id(idef.get('end1_port_type', 'Port'))
            e2_port = to_sysml_id(idef.get('end2_port_type', 'Port'))
            interface_defs.append({
                'name': to_sysml_id(idef['name']),
                'description': safe_doc(idef.get('description', 'TBD')),
                'end1_name': to_camel_case(idef.get('end1_name', 'source')),
                'end1_port': ('~' + e1_port) if idef.get('end1_conjugated', False) else e1_port,
                'end2_name': to_camel_case(idef.get('end2_name', 'target')),
                'end2_port': ('~' + e2_port) if idef.get('end2_conjugated', True) else e2_port,
            })

        # --- Always add TC-specific port defs from ports list ---
        if td.get('ports'):
            seen_port_types = {pd['name'] for pd in port_defs}
            for p in td['ports']:
                type_name = p.get('port_def_name') or (to_sysml_id(p['name']) + 'Port')
                if type_name not in seen_port_types:
                    seen_port_types.add(type_name)
                    port_defs.append({
                        'name': type_name,
                        'description': safe_doc(p.get('notes', 'TBD'), 200),
                        'port_items': [{'direction': p.get('direction', 'inout'),
                                   'name': to_camel_case(p.get('item_type', 'data')),
                                   'type_name': 'DataFlow'}],
                    })
            if not flow_item_defs:
                flow_item_defs = [{'name': 'DataFlow',
                                   'description': 'Generic data flow'}]

        return {'flow_item_defs': flow_item_defs,
                'port_defs': port_defs,
                'interface_defs': interface_defs}

    def _prep_technical_individuals(self):
        """Prepares TechnicalIndividualsPackage: named instances of technical components."""
        individuals = []
        for ti in self.data['technical_data'].get('technical_individuals', []):
            tc_id = ti.get('type_tc_id', '')
            type_name = self.id_map.get(tc_id, to_sysml_id(ti.get('name', 'Unknown')))
            individuals.append({
                'display_name': ti.get('name', 'Unknown'),
                'type_name': type_name,
                'description': safe_doc(ti.get('description', 'TBD')),
            })
        return {'individuals': individuals}


    def _prep_requirements(self, req_type: str):
        key_map = {'mission': 'mission_requirements', 'functional': 'functional_requirements',
                   'technical': 'technical_requirements'}
        refine_rel_map = {'mission': 'mission_requirement_refines_capability',
                          'functional': 'functional_requirement_refines_mission_requirement',
                          'technical': 'technical_requirement_refines_functional_requirement'}
        rel_type = refine_rel_map[req_type]
        source_to_targets = defaultdict(list)
        for src, tgt in self.refine_map.get(rel_type, []):
            source_to_targets[src].append(tgt)
        reqs = []
        for r in self.data['requirements_data'][key_map[req_type]]:
            # Use descriptive_name if available, fall back to generic ID-based name
            desc_name = r.get('descriptive_name', '').strip()
            if desc_name:
                name = to_sysml_id(desc_name)
                # Ensure it ends with 'Requirement' for consistency
                if not name.endswith('Requirement'):
                    name += 'Requirement'
            else:
                name = to_sysml_id(r['id'].replace('-', '')) + 'Requirement'

            rationale = r.get('rationale', '').replace('\\', '\\\\').replace('"', '\\"').replace('\n', ' ')

            refine_targets = []
            for tgt_id in source_to_targets.get(r['id'], []):
                if req_type == 'mission':
                    refine_targets.append(self.id_map.get(tgt_id, tgt_id))
                else:
                    refine_targets.append(tgt_id)
            reqs.append({
                'id': r['id'],
                'name': name,
                'text': safe_doc(r.get('text', 'TBD')),
                'rationale': rationale,
                'refines_cap': refine_targets if req_type == 'mission' else [],
                'refines_mr': refine_targets if req_type == 'functional' else [],
                'refines_fr': refine_targets if req_type == 'technical' else [],
            })
        return reqs

    def _build_system_path_map(self):
        """Builds map of LC/TC IDs and names to dot-notation feature paths
        through the SoS hierarchy, for SystemSpecificationPackage satisfy links.

        Pattern (Apollo 11 ground truth):
          satisfy 'clr-R001' by apollo11MissionSystem.launchVehicle.stage1;
        Path = sosInstance.constituentInstance
        """
        sos = self.data['logical_data']['sos']
        sos_instance = to_camel_case(self._get_sos_type_name())

        # Map LC IDs and names to their SoS constituent instance paths
        lc_to_path = {}
        for cs in sos.get('constituents', []):
            lc_id = cs.get('lc_id', '')
            lc_name = self.id_map.get(lc_id, '')
            if lc_name:
                constituent_instance = to_camel_case(lc_name)
                path = f'{sos_instance}.{constituent_instance}'
                lc_to_path[lc_id] = path
                lc_to_path[lc_name] = path

        # Map TC IDs and names to their realizing LC's SoS path
        tc_to_path = {}
        for tc in self.data['technical_data']['technical_components']:
            tc_name = to_sysml_id(tc['name'])
            lc_id = tc.get('realizes_lc_id', '')
            lc_name = self.id_map.get(lc_id, '')
            # Try LC path first, fall back to TC name
            path = lc_to_path.get(lc_id) or lc_to_path.get(lc_name)
            if path:
                tc_to_path[tc['id']] = path
                tc_to_path[tc_name] = path

        # Merge: LC paths + TC paths
        self._system_path_map = {}
        self._system_path_map.update(lc_to_path)
        self._system_path_map.update(tc_to_path)

    def _build_mission_path_map(self):
        """Builds map of OP-IDs, operation names, FN-IDs, and function names
        to dot-notation feature paths through the mission's state machine,
        for MissionSpecificationPackage satisfy links.

        Pattern (Apollo 11 ground truth):
          satisfy 'hlr-R001' by apollo11Mission.apollo11Phases.recoveryQuarantine
              .recoveryQuarantineOperations.retrieveCrewAndCM;
        Path = missionInstance.phasesName.phaseInstance.opsBlockName.opInstance
        """
        m = self.data['mission_data']['mission']
        mission_name = to_sysml_id(m['name'])
        mission_instance = to_camel_case(mission_name)
        phases_name = to_camel_case(mission_name) + 'Phases'

        op_map = {o['id']: o for o in self.data['operation_data']['operations']}
        phase_id_map = {p['id']: p for p in self.data['mission_data']['phases']}

        self._mission_path_map = {}

        for p in self.data['mission_data']['phases']:
            phase_instance = to_camel_case(p['name'])
            ops_block_name = phase_instance + 'Operations'

            for op_id in p.get('operation_ids', []):
                if op_id not in op_map:
                    continue
                op = op_map[op_id]
                op_instance = to_camel_case(op['name'])
                path = f'{mission_instance}.{phases_name}.{phase_instance}.{ops_block_name}.{op_instance}'

                self._mission_path_map[op_id] = path
                op_name = to_sysml_id(op['name'])
                self._mission_path_map[op_name] = path

        # Extend map: index by function IDs/names that refine operations (D137)
        # Functions refine operations via refines_op_ids; satisfy links reference
        # function names, but the path goes through the state machine (operations).
        for fn in self.data.get('function_data', {}).get('functions', []):
            fn_id = fn.get('id', '')
            fn_name = to_sysml_id(fn.get('name', ''))
            for ref_op_id in fn.get('refines_op_ids', []):
                if ref_op_id in self._mission_path_map:
                    self._mission_path_map[fn_id] = self._mission_path_map[ref_op_id]
                    self._mission_path_map[fn_name] = self._mission_path_map[ref_op_id]

    def _prep_specifications(self):
        spec = self.data['specification_data']
        m = self.data['mission_data']['mission']
        mission_name = to_sysml_id(m['name'])
        sos_name = self._get_sos_type_name()
        top_fn_name = to_sysml_id(self.data['function_data']['functions'][0]['name'])
        sat_mr, sat_fr, sat_tr = [], [], []
        fn_paths = getattr(self, '_fn_path_map', {})
        # Build system path map for TR- satisfy links
        self._build_system_path_map()
        sys_paths = getattr(self, '_system_path_map', {})
        # Build mission path map for MR- satisfy links
        self._build_mission_path_map()
        msn_paths = getattr(self, '_mission_path_map', {})

        for sl in spec.get('satisfy_links', []):
            req_id = sl['requirement_id']
            raw_ref = sl['satisfied_by_id']
            resolved_name = self.id_map.get(raw_ref, raw_ref)

            if req_id.startswith('FR-'):
                path = fn_paths.get(raw_ref) or fn_paths.get(resolved_name) or resolved_name
                sat_fr.append({'req_id': req_id, 'satisfied_by': path})
            elif req_id.startswith('TR-'):
                path = sys_paths.get(raw_ref) or sys_paths.get(resolved_name) or resolved_name
                sat_tr.append({'req_id': req_id, 'satisfied_by': path})
            elif req_id.startswith('MR-'):
                path = msn_paths.get(raw_ref) or msn_paths.get(resolved_name) or resolved_name
                sat_mr.append({'req_id': req_id, 'satisfied_by': path})

        # Dedup satisfy entries
        def _dedup(entries):
            seen = set()
            unique = []
            for e in entries:
                key = (e['req_id'], e['satisfied_by'])
                if key not in seen:
                    seen.add(key)
                    unique.append(e)
            return unique

        sat_mr = _dedup(sat_mr)
        sat_fr = _dedup(sat_fr)
        sat_tr = _dedup(sat_tr)

        return {
            'mission_spec': {'spec_name': to_camel_case(spec.get('mission_specification_name', 'missionSpec')),
                             'mission_name': mission_name, 'mission_type': mission_name,
                             'mission_instance': to_camel_case(mission_name), 'satisfy_links': sat_mr},
            'function_spec': {'spec_name': to_camel_case(spec.get('function_specification_name', 'functionSpec')),
                              'function_type': top_fn_name, 'function_instance': to_camel_case(top_fn_name),
                              'satisfy_links': sat_fr},
            'system_spec': {'spec_name': to_camel_case(spec.get('system_specification_name', 'systemSpec')),
                            'system_type': sos_name, 'system_instance': to_camel_case(sos_name),
                            'satisfy_links': sat_tr},
        }

    def _prep_program(self):
        m = self.data['mission_data']['mission']
        mission_name = to_sysml_id(m['name'])

        # Related missions (excluding current)
        related = []
        for rm in m.get('related_missions', []):
            if rm.get('is_current', False):
                continue
            related.append({
                'instance': to_camel_case(rm['name']),
                'description': safe_doc(rm.get('description', 'TBD')),
            })

        return {
            'program_name': to_sysml_id(m.get('program_name', 'MissionProgram')),
            'program_description': safe_doc(m.get('program_description',
                                                   m.get('program_name', 'TBD'))),
            'current_mission': {
                'instance': to_camel_case(mission_name),
                'type': mission_name,
            },
            'related_missions': related,
        }
    def _prep_analysis(self):
        m = self.data['mission_data']['mission']
        mission_name = to_sysml_id(m['name'])
        def prep_measures(items):
            result = []
            for item in items:
                entry = {
                    'name': to_sysml_id(item['name']),
                    'description': safe_doc(item.get('description', 'TBD')),
                    'subject_type': mission_name,
                    'subject_instance': to_camel_case(mission_name),
                    'out_name': to_camel_case(item['name']),
                    'formula': item.get('formula', ''),
                    'threshold': item.get('threshold', 'TBD'),
                    'variables': item.get('variables', []),
                }
                result.append(entry)
            return result
        return {
            'measures_of_success': prep_measures(self.data['measures_data'].get('measures_of_success', [])),
            'measures_of_effectiveness': prep_measures(self.data['measures_data'].get('measures_of_effectiveness', [])),
            'measures_of_performance': prep_measures(self.data['measures_data'].get('measures_of_performance', [])),
        }

    def _prep_calculations(self):
        """Prepares CalculationsPackage: one calc def skeleton per measure.
        Uses same data as _prep_analysis — the CalculationsPackage provides
        the calc defs that AnalysisPackage analysis defs invoke."""
        return self._prep_analysis()

    def _prep_execution(self):
        """Prepares MissionExecutionPackage: individual part def + timeline of timeslices per phase.
        Follows Apollo 11 MissionExecutionPackage pattern:
          individual part def MissionIndividual :> Mission { outcome attributes }
          individual part missionIndividual : MissionIndividual { timeslice per phase }
        """
        m = self.data['mission_data']['mission']
        mission_name = to_sysml_id(m['name'])
        sos_type = self._get_sos_type_name()

        # Outcome attributes from MOS
        outcomes = []
        for mos in self.data['measures_data'].get('measures_of_success', []):
            outcomes.append({
                'attr_name': to_camel_case(mos['name']),
                'attr_type': 'ScalarValues::Real' if mos.get('formula') else 'ScalarValues::Boolean',
            })

        # Timeslices from phases
        phases = self.data['mission_data']['phases']
        timeslices = []
        for p in phases:
            phase_type = to_sysml_id(p['name'])
            phase_instance = to_camel_case(p['name'])
            timeslices.append({
                'instance_name': to_camel_case(p['name']) + 'Execution',
                'phase_instance': phase_instance,
                'description': f"Timeslice covering the {p.get('display_name', p['name'])} phase.",
            })

        # Build phases ref: missionInstance.phasesName (from MissionPackage exhibit state)
        phases_ref = to_camel_case(mission_name) + 'Phases'

        return {
            'mission': {
                'name': mission_name,
                'display_name': m.get('display_name', m['name']),
                'individual_type': mission_name + 'Individual',
                'individual_instance': to_camel_case(mission_name) + 'Individual',
                'system_instance': to_camel_case(sos_type),
                'phases_ref': phases_ref,
            },
            'outcomes': outcomes,
            'timeslices': timeslices,
        }

    def generate_all(self) -> dict:
        def gen(name, ctx):
            self.results[name] = self._render(name, ctx)
            print(f"  ✅ {name:45s} ({len(self.results[name]):>6,} chars)")

        # CoSMA framework packages (static, not Jinja2)
        self.results['CoSMAPackage'] = COSMA_PACKAGE
        print(f"  ✅ {'CoSMAPackage':45s} ({len(COSMA_PACKAGE):>6,} chars) [framework]")
        self.results['CoSMAQuantitiesAndUnitsPackage'] = COSMA_QUANTITIES_AND_UNITS_PACKAGE
        print(f"  ✅ {'CoSMAQuantitiesAndUnitsPackage':45s} ({len(COSMA_QUANTITIES_AND_UNITS_PACKAGE):>6,} chars) [framework]")
        self.results['CoSMAViewsPackage'] = COSMA_VIEWS_PACKAGE
        print(f"  ✅ {'CoSMAViewsPackage':45s} ({len(COSMA_VIEWS_PACKAGE):>6,} chars) [framework]")


        gen('StakeholderPackage', self._prep_stakeholders())
        gen('StakeholderNeedsPackage', self._prep_stakeholder_needs())
        gen('CapabilitiesPackage', self._prep_capabilities())
        ops_list = self._prep_operations()
        gen('OperationsPackage', {'operations': ops_list})
        phases = self._prep_phases(ops_list)
        gen('MissionPhasesPackage', {'phases': phases})
        gen('ContextPackage', self._prep_context())
        gen('MissionPackage', self._prep_mission(phases))
        gen('FunctionsPackage', self._prep_functions())
        gen('LogicalComponentsPackage', self._prep_logical())
        gen('SystemPackage', self._prep_system())
        gen('TechnicalComponentsPackage', self._prep_technical())
        gen('TechnicalPortsPackage', self._prep_ports())
        gen('TechnicalIndividualsPackage', self._prep_technical_individuals())
        mr_list = self._prep_requirements('mission')
        fr_list = self._prep_requirements('functional')
        tr_list = self._prep_requirements('technical')
        gen('MissionRequirementsPackage', {'mission_requirements': mr_list})
        gen('FunctionalRequirementsPackage', {'functional_requirements': fr_list})
        gen('TechnicalRequirementsPackage', {'technical_requirements': tr_list})
        specs = self._prep_specifications()
        gen('MissionSpecificationPackage', {**specs['mission_spec'], 'mission_requirements': mr_list})
        gen('FunctionSpecificationPackage', {**specs['function_spec'], 'functional_requirements': fr_list})
        gen('SystemSpecificationPackage', {**specs['system_spec'], 'technical_requirements': tr_list})
        gen('ProgramPackage', self._prep_program())
        gen('AnalysisPackage', self._prep_analysis())
        gen('CalculationsPackage', self._prep_calculations())
        gen('MissionExecutionPackage', self._prep_execution())
        # Root model — dynamic name derived from the mission
        _mission_name = self.data.get('mission_data', {}).get('mission', {}).get('name', 'MissionModel')
        _root_model_name = to_sysml_id(_mission_name) + 'Model'
        gen('RootModel', {'model_name': _root_model_name})
        # Rename key to the dynamic name
        self.results[_root_model_name] = self.results.pop('RootModel')
        total = sum(len(v) for v in self.results.values())
        print(f"\nTotal: {len(self.results)} packages, {total:,} chars")
        return self.results

# ── Package → output-folder map (mirrors ground-truth structure) ──

PACKAGE_TO_FOLDER = {
    'CoSMAPackage': 'CoSMA',
    'CoSMAQuantitiesAndUnitsPackage': 'CoSMA',
    'CoSMAViewsPackage': 'CoSMA',
    'StakeholderPackage': 'Purpose',
    'StakeholderNeedsPackage': 'Requirements',
    'CapabilitiesPackage': 'Purpose',
    'MissionPackage': 'Purpose',
    'MissionPhasesPackage': 'Purpose',
    'MissionSpecificationPackage': 'Purpose',
    'ContextPackage': 'Purpose',
    'OperationsPackage': 'Operation',
    'FunctionsPackage': 'Function',
    'FunctionSpecificationPackage': 'Function',
    'LogicalComponentsPackage': 'Logical',
    'SystemPackage': 'Technical',
    'SystemSpecificationPackage': 'Technical',
    'TechnicalComponentsPackage': 'Technical',
    'TechnicalPortsPackage': 'Technical',
    'TechnicalIndividualsPackage': 'Technical',
    'MissionRequirementsPackage': 'Requirements',
    'FunctionalRequirementsPackage': 'Requirements',
    'TechnicalRequirementsPackage': 'Requirements',
    'ProgramPackage': 'Program',
    'AnalysisPackage': 'Analysis',
    'CalculationsPackage': 'Analysis',
    'MissionExecutionPackage': 'Execution',
}

# ── run_p3: wrapper State → State ──────────────────────────

def run_p3(state: dict) -> dict:
    """TemplateGeneratorClass (P3): JSON → 21 SysML v2 skeletons.
    100% deterministic, cost $0.00.

    State input: mission_json
    Output: skeletons {pkg_name: sysml_str}
    """
    print(f'\n{"="*60}')
    print(f'  P3 — TemplateGeneratorClass (deterministic)')
    print(f'{"="*60}')
    t0 = time.time()

    data = state['mission_json']

    # Build maps (D60)
    id_map = build_id_to_name_map(data)
    print(f'  ID map: {len(id_map)} entries')

    # Generate
    generator = TemplateGeneratorClass(data, id_map, TEMPLATES)
    skeletons = generator.generate_all()

    # Save to Drive — organized by CoSMA layer folders
    skel_dir = TEMPLATE_DIR / f'skeletons_{state["run_timestamp"]}'
    skel_dir.mkdir(parents=True, exist_ok=True)
    for pkg_name, content in skeletons.items():
        # Determine subfolder (default to root if not mapped)
        folder = PACKAGE_TO_FOLDER.get(pkg_name, '')
        if folder:
            target_dir = skel_dir / folder
        else:
            # RootModel and any unmapped packages go to root
            target_dir = skel_dir
        target_dir.mkdir(parents=True, exist_ok=True)
        (target_dir / f'{pkg_name}.sysml').write_text(content, encoding='utf-8')
    # Print summary by folder
    from collections import Counter
    folder_counts = Counter(PACKAGE_TO_FOLDER.get(p, 'root') for p in skeletons)
    for folder, count in sorted(folder_counts.items()):
        print(f'    {folder:15s}: {count} packages')
    print(f'  Saved to: {skel_dir}')

    elapsed = time.time() - t0
    log_cost(state, 'P3_TemplateGenerator', 0.0, elapsed)
    print(f'  P3 finished: {len(skeletons)} packages, {sum(len(v) for v in skeletons.values()):,} chars in {elapsed:.3f}s')

    return {'skeletons': skeletons}


# ── Verification ───────────────────────────────────────────
print(f'P3 defined: run_p3(state) → skeletons')
print(f'  3 CoSMA framework + 21 Jinja2 templates + TemplateGeneratorClass + helpers')
print(f'  Cost: $0.00 (deterministic)')



### Cell 7 — P4 RefinementAgent and run_p4()

In [ ]:
# ============================================================
# Master Orchestrator
# Cell 7 — P4 (RefinementAgent) — semantic LLM
# ============================================================
# Refines 3 packages via LLM (TODO[P4] → formal SysML v2).
# 18 passthrough packages.
# ============================================================


# ── P4 constants ───────────────────────────────────────────

P4_MAX_TOKENS = {
    'TechnicalComponentsPackage':    16000,
    'AnalysisPackage':               16000,
    'CalculationsPackage':           16000,
    'ContextPackage':                 8000,
    'MissionPackage':                12000,
    'MissionRequirementsPackage':    16000,
    'FunctionalRequirementsPackage': 16000,
    'TechnicalRequirementsPackage':  16000,
    'MissionExecutionPackage':       35000,
}

P4_PASSTHROUGH = [
    # CoSMA framework (static, never refined)
    'CoSMAPackage', 'CoSMAQuantitiesAndUnitsPackage', 'CoSMAViewsPackage',
    # Purpose layer
    'StakeholderPackage', 'StakeholderNeedsPackage', 'CapabilitiesPackage',
    'MissionPhasesPackage',
    # Operational layer
    'OperationsPackage',
    # Functional layer
    'FunctionsPackage', 'FunctionSpecificationPackage',
    # Logical layer
    'LogicalComponentsPackage',
    # Technical layer
    'SystemPackage', 'SystemSpecificationPackage',
    'TechnicalPortsPackage', 'TechnicalIndividualsPackage',
    # Specification
    'MissionSpecificationPackage',
    # Program
    'ProgramPackage',
]

P4_LLM_PACKAGES = list(P4_MAX_TOKENS.keys())

# RAG top_k — configurable per model context window size
# Large context (200K): 12-15 examples | Medium (100K): 6-8 | Small (32K): 3-4
P4_RAG_TOP_K = 6

# ── P4 System Prompt ───────────────────────────────────────

P4_SYSTEM_PROMPT = """\
You are a SysML v2 expert completing partially-generated model files for a military mission simulation.
You follow the CoSMA framework (Helle & Schramm, 2026) and the Apollo 11 SysML v2 model as ground truth.

## ABSOLUTE RULES
1. OUTPUT ONLY valid SysML v2 syntax. No explanations, no markdown fences, no preamble.
2. NEVER truncate the output. Output the COMPLETE file from first line to closing brace.
3. NEVER modify any line that does not contain a TODO[P4] marker.
4. REPLACE every /* TODO[P4]: ... */ comment with the correct SysML v2 declaration.
5. After replacing a TODO[P4], DELETE the comment — do not leave it in the output.
6. If you are uncertain about a value, use a physically plausible default and add // estimated inline.
7. Use ISQ/SI type annotations exactly as shown in the examples below.
8. NEVER invent function calls or calc references that don't exist in the model.
   If a calculation is not defined, use a placeholder value with // calc TBD inline.

## SysML v2 ATTRIBUTE PATTERNS
There are TWO distinct patterns — do NOT confuse them:

### Pattern 1: Subsetting (:>) — declares a NEW attribute typed by an ISQ quantity
  attribute maxSpeed :> ISQ::speed = 320 [kn];
  attribute mass :> ISQ::mass = 5840 [kg];
  attribute range :> ISQ::length = 1550 [nmi];
  attribute acuity :> ISQ::angle = 1.0 ['arcmin'];

### Pattern 2: Redefinition (:>>) — OVERRIDES an inherited attribute from a parent def
  attribute :>> dryMass = 2100 [kg];
  attribute :>> powerLoad = 1000 [W];
  attribute :>> failureRate = 3.0E-6 [1/h];
  attribute :>> mass = 5840 [kg];

Use :>> ONLY when the parent part def already declares that attribute.
Use :> for all new attributes.

## UNIT SYMBOLS — MANDATORY REFERENCE
Use ONLY these unit symbols (defined in CoSMAQuantitiesAndUnitsPackage):

  SI base & derived:  kg, m, s, W, N, Pa, K, Hz, A, V, rad
  SI prefixed:        kN, kPa, MN, kB, km, mm
  Angle:              ['°'] for degrees, [rad] for radians
  Frequency:          Hz ONLY (convert MHz → multiply by 1E6, GHz → multiply by 1E9)
  Time:               h, min, yr
  Aviation/Maritime:  kn (knot), nmi (nautical mile)
  Sensor/ISR:         'arcmin' (arcminute), 'arcsec' (arcsecond)
  Maritime:           ftm (fathom)
  Other:              gn (standard gravity), '%' (percent), '$' (dollar)

  WRONG: ['knot'], ['ft'/'min'], ['nautical mile'], ['knots']
  RIGHT: [kn], [nmi], ['arcmin'], [m], [kg], [kN]

  For compound units, use SI notation: [m/s], [kg/h], [1/h], [W/m^2]

## ISQ QUANTITY TYPES
  ISQ::mass, ISQ::length, ISQ::speed, ISQ::duration, ISQ::power, ISQ::force,
  ISQ::pressure, ISQ::frequency, ISQ::planeAngle, ISQ::time, ISQ::acceleration,
  ISQ::thermodynamicTemperature, ISQ::irradiance, ISQ::electricCurrent,
  ISQ::energy, ISQ::area, ISQ::volume, ISQ::density, ISQ::angularSpeed,
  ISQ::electricCharge, ISQ::electricPotential, ISQ::luminousFlux
  ScalarValues::Real (dimensionless), ScalarValues::Boolean, ScalarValues::String, ScalarValues::Integer

  WRONG ISQ types (do NOT use):
    ISQ::angle (use ISQ::planeAngle)
    ISQ::velocity (use ISQ::speed)
    ISQ::weight (use ISQ::force or ISQ::mass)
    ISQ::temperature (use ISQ::thermodynamicTemperature)
"""


# ── P4 Prompts ─────────────────────────────────────────────

def _p4_prompt_tc(skeleton, json_data):
    tc_data = json_data.get('technical_data', {})
    components = tc_data.get('technical_components', [])
    specs_summary = [{'id': tc.get('id'), 'name': tc.get('name'),
                      'specifications': tc.get('specifications', [])}
                     for tc in components if tc.get('specifications')]
    return f"""\
## TASK: Refine TechnicalComponentsPackage

Convert every /* TODO[P4]: attribute NAME = VALUE [UNIT]; */ comment into a
formal SysML v2 attribute declaration inside the enclosing `part def` block.

### PATTERNS (from Apollo 11 ground truth)
```
part def LunarModuleDescentStage :> PropelledSpacecraft, PowerConsumer, PowerProvider {{
    doc /* The lower portion of the Lunar Module. */
    attribute :>> dryMass = 2100 [kg];
    attribute :>> propellantMass = 8250 [kg];
    attribute :>> maxThrust = 45 [kN];
    attribute :>> specificImpulse = 305 ['s'];
    attribute :>> powerGenerated = 1500 [W];
    attribute :>> powerLoad = 1000 [W];
    attribute :>> failureRate = 3.0E-6 [1/h];
    port upperStagePort : LMStagingPort;
}}

part def SaturnV :> MultistageRocket, LaunchSystem {{
    attribute height :> ISQ::length = 110.6 [m];
    attribute launchMass :> ISQ::mass = 2970000 [kg];
}}
```
### IMPORTANT: Pattern selection for our model
Our components inherit from `System` or `HardwareComponent` — these do NOT declare
pre-existing attributes. Therefore, use `:>` (subsetting) for ALL new attributes:
    attribute maxSpeed :> ISQ::speed = 320 [kn];
    attribute mass :> ISQ::mass = 5840 [kg];
Do NOT use `:>>` (redefinition) unless the parent type explicitly declares that attribute.
The `:>>` examples above (dryMass, powerLoad, failureRate) apply ONLY to components that
inherit from abstract types like PropelledSpacecraft or PowerConsumer.

### UNIT MAPPING — use ONLY these symbols
Speed: [kn] for knots, [m/s] for metres per second
Length: [m], [km], [nmi] for nautical miles
Duration: [s], [min], [h]
Mass: [kg]
Force: [kN], [N]
Pressure: [kPa], [Pa]
Frequency: [Hz] ONLY (convert MHz → value×1E6, GHz → value×1E9)
Angle: ['°'] for degrees, [rad] for radians, ['arcmin'] for arcminutes
Dimensionless: use ScalarValues::Real (no unit brackets)
Boolean: use ScalarValues::Boolean
String: use ScalarValues::String = "value"

WRONG unit examples → CORRECT:
  ['knot'] → [kn]
  ['ft'/'min'] → [m/s]  (convert to SI or use defined unit)
  ['nmi'] → [nmi]
  ['arcmin'] → ['arcmin']

If a value is in non-SI units (feet, statute miles), convert to the nearest
defined unit (m, km, nmi, kn) and add // converted from X inline.

### TECHNICAL COMPONENTS DATA (from MissionAgent P2)
{_json.dumps(specs_summary, ensure_ascii=False, indent=2)}

### SKELETON TO REFINE (output this file COMPLETE, from first line to last brace)
{skeleton}"""


def _p4_prompt_ports(skeleton, json_data):
    ports = json_data.get('technical_data', {}).get('ports', [])
    ports_summary = [{'name': p.get('name'), 'direction': p.get('direction', 'inout'),
                      'item_type': p.get('item_type', 'ScalarValues::Real'),
                      'owner_tc_id': p.get('owner_tc_id'), 'notes': p.get('notes', '')}
                     for p in ports]
    return f"""\
## TASK: Refine TechnicalPortsPackage

Convert every /* TODO[P4]: direction: DIR | item: ITEM | owner: TC-XXX */ comment
into a formal SysML v2 item declaration inside the enclosing `port def` block.

### PATTERN
```
port def RadioFrequencyPort {{
    doc /* RF signal port for C2 link */
    in item commandSignal   : ScalarValues::Real;
    out item telemetryData  : ScalarValues::Real;
}}
```

### PORT DATA (from MissionAgent P2)
{_json.dumps(ports_summary, ensure_ascii=False, indent=2)}

### SKELETON TO REFINE (output this file COMPLETE, from first line to last brace)
{skeleton}"""


def _p4_prompt_analysis(skeleton, json_data):
    measures = json_data.get('measures_data', {})
    # Extract available calc def names from measures data
    calc_names = []
    for category in ('measures_of_success', 'measures_of_effectiveness', 'measures_of_performance'):
        for m in measures.get(category, []):
            calc_names.append('calculate' + m.get('name', ''))
    calc_list = '\n'.join(f'  - {n}' for n in calc_names)
    return f"""\
## TASK: Refine AnalysisPackage

Replace every /* TODO[P4]: bind out to calc def from CalculationsPackage; add assert constraint */
comment with:
1. Keep the existing `out` line but change its type from `ScalarValues::Real` to the
   appropriate ISQ type if applicable (e.g., ISQ::duration for time metrics)
2. Add an `assert constraint {{ ... }}` with a threshold from the JSON data

### CRITICAL RULES
- For each `analysis def`, bind the `out` parameter to its corresponding calc def
  from CalculationsPackage using the pattern: out x :> type = calculateX(param1, param2);
- Use the calc def names exactly as defined in CalculationsPackage (e.g., calculateNeutralizationRate,
  calculateEngagementTimeRate). The in parameters should reference the subject's attributes.
- Add `assert constraint {{ ... }}` with threshold from the JSON measures data.
- Do NOT modify the `analysis` instances (the ones that specialize the def) — leave them as-is.
- For ratio/percentage metrics (0.0–1.0), keep `: ScalarValues::Real`.
- For time/duration metrics, use `:> ISQ::duration`.
- Do NOT reference subject attributes with dot notation (e.g., subject.neutralizedCount).
  Instead, declare `in` parameters in the analysis def and pass them to the calc def.
  CORRECT: in neutralizedCount : ScalarValues::Integer;
           out rate : ScalarValues::Real = calculateNeutralizationRate(neutralizedCount, totalIncursions);
  WRONG:   out rate :> ScalarValues::Real = calculateNeutralizationRate(subject.neutralizedCount, subject.totalIncursions);

### CORRECT PATTERN (Apollo 11 ground truth)

```
analysis def SystemPowerAnalysis {{{{
    doc /* Analysis to calculate total power generation, load, and margin. */
    subject missionSystem : System;
    out totalPowerGenerated :> ISQ::power = rollupPowerGeneration(missionSystem);
    out totalPowerLoad :> ISQ::power = rollupPowerConsumption(missionSystem);
    out powerMargin :> ISQ::power = calculatePowerMargin(totalPowerGenerated, totalPowerLoad);
    assert constraint {{{{
        powerMargin > 0 [W]
    }}}}
}}}}

analysis def NeutralizationRateAnalysis {{{{
    doc /* Percentage of UAS neutralized before boundary. */
    subject missionSystem : MissionSystem;
    out neutralizationRate : ScalarValues::Real = calculateNeutralizationRate(missionSystem.neutralizedCount, missionSystem.totalIncursions);
    assert constraint {{{{
        neutralizationRate >= 0.80
    }}}}
}}}}
```

### WRONG PATTERN

```
// WRONG: modifying analysis instances (bindings go in the DEF, not the instance)
analysis operacaoEscudoNeutralizationRateAnalysis : NeutralizationRateAnalysis {{{{
    out neutralizationRate :> ScalarValues::Real = calculateNeutralizationRate(...);
}}}}

// WRONG: inventing calc defs that don't exist in CalculationsPackage
out rate :> ScalarValues::Real = computeCustomMetric(x, y);
```
### AVAILABLE CALC DEFS (use ONLY these — do NOT invent, split, or rename)
{calc_list}

### MEASURES DATA
MOS: {_json.dumps(measures.get('measures_of_success', []), ensure_ascii=False, indent=2)}
MOE: {_json.dumps(measures.get('measures_of_effectiveness', []), ensure_ascii=False, indent=2)}
MOP: {_json.dumps(measures.get('measures_of_performance', []), ensure_ascii=False, indent=2)}

### SKELETON TO REFINE (output this file COMPLETE, from first line to last brace)
{skeleton}"""

def _p4_prompt_calculations(skeleton, json_data):
    measures = json_data.get('measures_data', {})
    mos = measures.get('measures_of_success', [])
    moe = measures.get('measures_of_effectiveness', [])
    mop = measures.get('measures_of_performance', [])
    return f"""\
## TASK: Refine CalculationsPackage

Replace every /* TODO[P4]: define in parameters, return type ... */ comment with
a complete `calc def` body: `in` parameters, `return` with ISQ type, and formula.

### CRITICAL RULES
- Each `calc def` corresponds to ONE measure (MOS, MOE, or MOP).
- Use ISQ types for return and parameters where applicable.
- For ratios/percentages, return `:> ScalarValues::Real`.
- For durations, return `:> ISQ::duration`.
- For lengths/distances, return `:> ISQ::length`.
- Formulas should be simple arithmetic (division, subtraction, comparison).
  Do NOT reference external functions that are not defined in this file.
- Every `calc def` MUST be self-contained.

### CORRECT PATTERN (Apollo 11 ground truth)

```
calc def calculatePowerMargin {{{{
    doc /* Calculates the difference between available power and total loads. */
    in sourcePower :> ISQ::power;
    in totalLoadPower :> ISQ::power;
    return powerMargin :> ISQ::power = sourcePower - totalLoadPower;
}}}}

calc def calculateDeltaV {{{{
    doc /* Calculates max velocity change using the Tsiolkovsky Rocket Equation. */
    in isp :> specificImpulse;
    in g0 :> ISQ::acceleration;
    in m0 :> ISQ::mass;
    in mf :> ISQ::mass;
    return deltaV :> ISQ::speed = isp * g0 * ln(m0 / mf);
}}}}

calc def evaluateMissionSuccess {{{{
    doc /* Evaluates overall mission success based on primary objectives. */
    in crewSafe : ScalarValues::Boolean;
    in targetsNeutralized : ScalarValues::Boolean;
    return isSuccess : ScalarValues::Boolean = crewSafe and targetsNeutralized;
}}}}
```

### KEY RULES FOR TRANSFORMING TODO HINTS INTO CALC DEFS
- Replace `/* TODO[P4]: in X — description (unit) */` with `in X :> ISQ::type;` or `in X : ScalarValues::Real;`
- Replace `/* TODO[P4]: formula = EXPR */` by putting EXPR in the return line
- Replace `return result :> ScalarValues::Real;` with `return namedResult :> ISQ::type = FORMULA;`
- The return MUST have: a descriptive name (not "result"), the correct ISQ type, and the formula inline
- For dimensionless ratios use `: ScalarValues::Real`; for physical quantities use `:> ISQ::type`
- Map units from hints: (s) → ISQ::duration, (m) → ISQ::length, (km) → ISQ::length, (m/s) → ISQ::speed, (count) → ScalarValues::Integer, (proportion) → ScalarValues::Real
- PRESERVE ALL in parameters mentioned in TODO hints — do NOT aggregate or simplify them away.
  If a hint lists 10 input variables, the calc def MUST have 10 `in` declarations.
- The return formula MUST match the TODO hint formula as closely as possible.
  For summations (Σ), use a simplified aggregated input: declare an `in totalSum :> ISQ::type;`
  that represents the pre-computed summation result, then use it in the formula.
  Example: hint says "formula = (1/N) · Σ(MOE2_i)" → write:
    in totalMOE2Sum : ScalarValues::Real;
    in N : ScalarValues::Integer;
    return trackingRate : ScalarValues::Real = totalMOE2Sum / N;
  NOT: return trackingRate : ScalarValues::Real = Ptrack / N;

### MEASURES DATA (one calc def per measure)
MOS: {_json.dumps(mos, ensure_ascii=False, indent=2)}
MOE: {_json.dumps(moe, ensure_ascii=False, indent=2)}
MOP: {_json.dumps(mop, ensure_ascii=False, indent=2)}

### SKELETON TO REFINE (output this file COMPLETE, from first line to last brace)
{skeleton}"""


def _p4_prompt_context(skeleton, json_data):
    ctx = json_data.get('context_data', {}).get('context', {})
    ext_systems = ctx.get('external_systems', [])
    geo_refs = ctx.get('geographic_references', [])
    boundaries = ctx.get('operational_boundaries', [])
    ext_summary = [{'name': e.get('name',''), 'description': e.get('description','')} for e in ext_systems]
    ext_json = _json.dumps(ext_summary, ensure_ascii=False, indent=2)
    return f"""\

## TASK: Refine ContextPackage

Enrich the external system `part def` blocks with:
1. Attributes for location (name, coordinates, description) where known
2. Environment `part def` blocks describing the physical operating environment

### CRITICAL RULES
- Do NOT modify imports or the Context part def structure.
- Do NOT remove any existing `part def` or `part` declarations.
- ADD attributes inside existing `part def` blocks using pattern:
    attribute name : ScalarValues::String = "value";
    attribute coordinates : ScalarValues::String = "lat, lon";
- ADD environment `part def` blocks BEFORE the Context part def, using pattern:
    part def OperatingEnvironment {{{{
        doc /* Physical environment description */
        attribute temperature :> ISQ::thermodynamicTemperature = 305 [K];
        attribute pressure :> ISQ::pressure = 101325 [Pa];
    }}}}
- Use ISQ types for physical quantities, ScalarValues::String for text.

### CORRECT PATTERN (Apollo 11 ground truth)

```
part def EarthEnvironment {{{{
    doc /* The starting and ending physical environment. */
    attribute surfaceGravity :> ISQ::acceleration = 1 [gn];
    attribute standardAtmosphericPressure :> ISQ::pressure = 101325 [Pa];
}}}}

part def LaunchSite :> System {{{{
    doc /* The ground facility from which the mission originates. */
    attribute name : ScalarValues::String = "Kennedy Space Center";
    attribute location : ScalarValues::String = "Merritt Island, Florida, USA";
    attribute geographicCoordinates : ScalarValues::String = "28.57 N, 80.65 W";
}}}}
```

### CONTEXT DATA
Geographic references: {_json.dumps(geo_refs, ensure_ascii=False, indent=2)}
Operational boundaries: {_json.dumps(boundaries, ensure_ascii=False, indent=2)}
External systems: {ext_json}

### SKELETON TO REFINE (output this file COMPLETE, from first line to last brace)
{skeleton}"""


def _p4_prompt_mission(skeleton, json_data):
    phases = json_data.get('mission_data', {}).get('phases', [])
    phase_docs = []
    for p in phases:
        phase_docs.append({
            'name': p.get('name', ''),
            'entry_trigger': p.get('entry_trigger', ''),
            'exit_trigger': p.get('exit_trigger', ''),
        })
    return f"""\
## TASK: Refine MissionPackage

Add `doc /* if "..." */` blocks to each transition in the exhibit state machine.

### CRITICAL RULES
- Do NOT modify any structural element (part def, requirements, parts, connections).
- Do NOT modify the state declarations or transition targets.
- ONLY add `doc` blocks to transitions that accept a notification.
- The doc should describe the conditions under which the transition fires.
- Use the phase entry/exit triggers from the data below.
- The first transition (initial -> first phase) has no doc.

### CORRECT PATTERN (Apollo 11 ground truth)

```
transition first preparation accept PreparationPhaseCompletedNotification then launch {{
    doc /* if "all preparations concluded successfully" */
}}

transition first launch accept LaunchPhaseCompletedNotification then tli {{
    doc /* if "Successful insertion into stable Earth parking orbit" and
        "All spacecraft systems verified nominal or go for TLI" */
}}
```

### PHASE TRANSITION DATA
{_json.dumps(phase_docs, ensure_ascii=False, indent=2)}

### SKELETON TO REFINE (output this file COMPLETE, from first line to last brace)
{skeleton}"""

def _p4_prompt_requirements(skeleton, json_data, pkg_name, rag_context=''):
    """Shared prompt for MR, FR, TR packages.
    Few-Shot ICL via RAG: ground truth requirement defs as exemplars.
    """
    req_type_map = {
        'MissionRequirementsPackage': ('mission_requirements', 'MR'),
        'FunctionalRequirementsPackage': ('functional_requirements', 'FR'),
        'TechnicalRequirementsPackage': ('technical_requirements', 'TR'),
    }
    data_key, prefix = req_type_map[pkg_name]
    reqs = json_data.get('requirements_data', {}).get(data_key, [])

    return f"""\
## TASK: Refine {pkg_name}

Replace every /* TODO[P4]: attributes and require constraint */ comment with:
1. Attribute declarations for the quantifiable aspects of the requirement
2. A `require constraint` block that formalizes the SHALL statement

### CRITICAL RULES
- KEEP all existing content: doc, @Rationale, #refinement dependency — do NOT remove them.
- INSERT attributes and require constraint AFTER the @Rationale block and BEFORE the #refinement dependency.
- For each requirement, identify WHAT is being measured and add 1-3 attributes with types.
- Add a `require constraint` that formalizes the threshold or condition.
- Use ISQ types from the ground truth examples below. If unsure, use ScalarValues::Real.
- Do NOT invent function calls. Constraints use simple comparisons (>, <, >=, <=, ==).

### ISQ TYPE SELECTION GUIDE
- Distance/range: ISQ::length
- Speed/velocity: ISQ::speed
- Time/duration: ISQ::duration or ISQ::time
- Mass/weight: ISQ::mass
- Probability/ratio (0.0-1.0): ScalarValues::Real
- Count: ScalarValues::Integer
- Yes/No: ScalarValues::Boolean
- Text: ScalarValues::String
- Angle/bearing: ISQ::angle
- Frequency: ISQ::frequency
- Force: ISQ::force
- Pressure: ISQ::pressure

### UNIT SYMBOLS (from CoSMAQuantitiesAndUnitsPackage)
  [kn] knot, [nmi] nautical mile, [m], [km], [kg], [s], [min], [h],
  [kN], [kPa], [Hz], [MHz], [GHz], ['arcmin'], ['arcsec'], [rad], [deg],
  ['%'], [W], [N], [Pa], [K], [ftm]

### CORRECT PATTERN (Apollo 11 ground truth)

```
requirement def <'HLR-R002'> LunarLanderSoftLandingRequirement {{{{
    doc /* The mission shall achieve a soft landing of the crewed lunar lander. */
    @Rationale {{{{
        text = "A soft and precise lunar landing is fundamental.";
    }}}}
    attribute actualVerticalVelocity :> ISQ::speed;
    attribute maxVerticalVelocity :> ISQ::speed = 2 [m/s];
    attribute actualLandingZoneDeviation :> ISQ::length;
    attribute maxLandingZoneDeviation :> ISQ::length = 1000 [m];
    require constraint {{{{
        (actualVerticalVelocity <= maxVerticalVelocity) and
        (actualLandingZoneDeviation <= maxLandingZoneDeviation)
    }}}}
    #refinement dependency 'HLR-R002' to CapabilitiesPackage::LunarSurfaceLandingAndAscent;
}}}}

requirement def <'HLR-R004'> CommunicationUptimeRequirement {{{{
    doc /* The mission's communication system shall maintain continuous contact for >= 95% of duration. */
    @Rationale {{{{
        text = "Reliable communication is vital for real-time mission control.";
    }}}}
    attribute actualCommunicationUptimePercentage : ScalarValues::Real;
    attribute minCommunicationUptimePercentage : ScalarValues::Real = 95;
    attribute maxSingleBlackoutDuration :> ISQ::time;
    attribute allowedSingleBlackoutDuration :> ISQ::time = 10 [min];
    require constraint {{{{
        (actualCommunicationUptimePercentage >= minCommunicationUptimePercentage) and
        (maxSingleBlackoutDuration <= allowedSingleBlackoutDuration)
    }}}}
    #refinement dependency 'HLR-R004' to CapabilitiesPackage::GlobalTrackingAndCommunication;
}}}}
```

{rag_context}

### REQUIREMENTS DATA ({prefix})
{_json.dumps(reqs, ensure_ascii=False, indent=2)}

### SKELETON TO REFINE (output this file COMPLETE, from first line to last brace)
{skeleton}"""

def _p4_prompt_execution(skeleton, json_data, rag_context=''):
    """Prompt for MissionExecutionPackage.
    Few-Shot ICL via RAG: Apollo 11 timeslice/snapshot patterns as exemplars.
    Least-to-Most: generates timeline (missionTime) + docs + safe attributes first,
    then perform actions where LogicalComponent names are known from the skeleton.
    """
    phases = json_data.get('mission_data', {}).get('phases', [])
    ops = json_data.get('operation_data', {}).get('operations', [])

    # Build phase timeline with estimated mission times
    phase_timeline = []
    for i, p in enumerate(phases):
        phase_timeline.append({
            'name': p.get('name', ''),
            'display_name': p.get('display_name', p.get('name', '')),
            'description': p.get('description', ''),
            'operations': [o.get('name', '') for o in ops
                          if o.get('id', '') in p.get('operation_ids', [])],
        })

    # Build FN-ID → name lookup
    fn_id_to_name = {fn['id']: fn['name'] for fn in
                     json_data.get('function_data', {}).get('functions', [])}

    # Build list of available LogicalComponent::function names for perform refs
    lc_functions = []
    for lc in json_data.get('logical_data', {}).get('logical_components', []):
        lc_name = lc.get('name', '')
        for fn_id in lc.get('performs_fn_ids', []):
            fn_name = fn_id_to_name.get(fn_id, '')
            if lc_name and fn_name:
                fn_instance = fn_name[0].lower() + fn_name[1:]
                lc_functions.append(f'{lc_name}::{fn_instance}')

    return f"""\

## TASK: Refine MissionExecutionPackage

Replace every /* TODO[P4]: Add snapshots with system state, missionTime, and perform actions. */
comment with concrete snapshot blocks inside each timeslice.

### CRITICAL RULES — READ CAREFULLY
1. KEEP all existing structure: individual part def, individual part, timeslice declarations,
   assert constraint blocks — do NOT modify them.
2. For EACH timeslice, add 1-3 snapshots with this structure:
   a. `attribute missionTime :> ISQ::time = VALUE ['s'];` — estimated seconds from mission start
   b. `doc /* description of the event */`
   c. OPTIONALLY: `attribute status : ScalarValues::String = "description";`
3. For perform actions, use ONLY names from the AVAILABLE FUNCTIONS list below.
   If no matching function exists, use a doc comment instead of perform.
4. NEVER invent part :>> references to components not in the skeleton.
   Safe pattern: `attribute systemStatus : ScalarValues::String = "description";`
   Risky pattern (AVOID unless component name is in skeleton): `part :>> componentName {{{{ ... }}}}`
5. Mission times should be chronologically increasing across timeslices.
   Use realistic estimates based on the phase descriptions.
6. For outcome attributes (ScalarValues::Boolean in individual part def),
   assign `attribute :>> outcomeName = true;` in the appropriate timeslice.

### SNAPSHOT COMPLEXITY LEVELS (use Level 1-2, avoid Level 3)

Level 1 — SAFE (always valid):

```
snapshot atEventName {{{{
    doc /* Description of the event. */
    attribute missionTime :> ISQ::time = 3600 ['s'];
    attribute status : ScalarValues::String = "Phase active";
}}}}
```

Level 2 — MODERATE (valid if function name exists in AVAILABLE FUNCTIONS):

```
snapshot atEventName {{{{
    doc /* Description of the event. */
    attribute missionTime :> ISQ::time = 7200 ['s'];
    attribute altitude :> ISQ::length = 8000 [m];
    attribute speed :> ISQ::speed = 250 [kn];
    perform DetectionSystem::detectUasIncursion;
}}}}
```

Level 3 — RISKY (AVOID — part :>> may reference nonexistent features):

```
snapshot systemAtEvent :> missionSystem {{{{
    part :>> interceptor {{{{
        attribute weaponStatus : ScalarValues::String = "Armed";
    }}}}
}}}}
```

{rag_context}

### PHASE TIMELINE
{_json.dumps(phase_timeline, ensure_ascii=False, indent=2)}

### AVAILABLE FUNCTIONS (for perform references — use ONLY these)
{chr(10).join(f'  {fn}' for fn in lc_functions) if lc_functions else '  (none extracted — use Level 1 snapshots only)'}

### SKELETON TO REFINE (output this file COMPLETE, from first line to last brace)
{skeleton}"""


# ── RefinementAgent ───────────────────────────────────

class RefinementAgent:
    """LLM agent that converts TODO[P4] into formal SysML v2 declarations.
    Anti-truncation: streaming + automatic continuation + retry."""

    MAX_CONTINUATIONS = 3
    MAX_RETRIES = 1

    def __init__(self, skeletons, mission_data, client):
        self.skeletons = skeletons
        self.mission_data = mission_data
        self.client = client
        self.results = {}
        self.cost_total_in = 0
        self.cost_total_out = 0

    def _call_llm_streaming(self, user_prompt, max_tokens, prior_messages=None):
        messages = list(prior_messages or [])
        messages.append({'role': 'user', 'content': user_prompt})
        wait = 10
        for attempt in range(4):
            try:
                collected = ''
                with self.client.messages.stream(
                    model=LLM_MODEL, max_tokens=max_tokens,
                    system=P4_SYSTEM_PROMPT, messages=messages,
                ) as stream:
                    for chunk in stream.text_stream:
                        collected += chunk
                        print(chunk, end='', flush=True)
                    final = stream.get_final_message()
                return collected, final.stop_reason, final.usage.input_tokens, final.usage.output_tokens
            except anthropic.RateLimitError:
                print(f'\n  ⚠ Rate limit (attempt {attempt+1}/4). Waiting {wait}s...')
                time.sleep(wait)
                wait *= 2
            except Exception as e:
                err_msg = str(e).lower()
                if any(kw in err_msg for kw in ['chunked', 'peer closed', 'connection', 'timeout', 'reset', 'eof']):
                    print(f'\n  ⚠ Network error (attempt {attempt+1}/4): {e}')
                    print(f'    Waiting {wait}s before retry...')
                    time.sleep(wait)
                    wait *= 2
                else:
                    raise
        raise RuntimeError('P4 _call_llm_streaming failed after 4 attempts')

    def _call_with_continuation(self, user_prompt, max_tokens):
        full_text, total_in, total_out, conts = '', 0, 0, 0
        print(f'\n[LLM] Initial call (max_tokens={max_tokens}) ...')
        text, stop, t_in, t_out = self._call_llm_streaming(user_prompt, max_tokens)
        full_text += text
        total_in += t_in
        total_out += t_out
        conversation = [{'role': 'user', 'content': user_prompt}, {'role': 'assistant', 'content': text}]
        while stop == 'max_tokens' and conts < self.MAX_CONTINUATIONS:
            conts += 1
            print(f'\n[ANTI-TRUNCATION] continuation {conts}/{self.MAX_CONTINUATIONS} ...')
            cont_prompt = ('Continue the SysML v2 file exactly from where you stopped. '
                          'Do NOT repeat any content already written. '
                          'Output only the remaining SysML v2 code until the closing brace.')
            cont_text, stop, t_in, t_out = self._call_llm_streaming(cont_prompt, max_tokens, conversation)
            full_text += cont_text
            total_in += t_in
            total_out += t_out
            conversation.extend([{'role': 'user', 'content': cont_prompt},
                                {'role': 'assistant', 'content': cont_text}])
        return full_text, total_in, total_out, conts

    def _trim_after_root_close(self, text):
        depth, in_doc, in_line, i = 0, False, False, 0
        last_close = len(text)
        while i < len(text):
            c = text[i]
            if not in_doc and not in_line and text[i:i+2] == '/*':
                in_doc = True; i += 2; continue
            if in_doc and text[i:i+2] == '*/':
                in_doc = False; i += 2; continue
            if in_doc: i += 1; continue
            if not in_line and text[i:i+2] == '//':
                in_line = True; i += 2; continue
            if in_line and c == '\n':
                in_line = False; i += 1; continue
            if in_line: i += 1; continue
            if c in ('"', "'"):
                q = c; i += 1
                while i < len(text) and text[i] != q:
                    if text[i] == '\\': i += 1
                    i += 1
                i += 1; continue
            if c == '{': depth += 1
            elif c == '}':
                depth -= 1
                if depth == 0: last_close = i + 1; break
            i += 1
        return text[:last_close]

    def _clean_llm_output(self, text, pkg_name):
        cleanings = []
        stripped = re.sub(r'^```[\w]*\s*\n', '', text.strip())
        stripped = re.sub(r'\n```\s*$', '', stripped)
        if len(stripped) != len(text.strip()): cleanings.append('fences')
        text = stripped
        match = re.search(rf'^package\s+{re.escape(pkg_name)}\s*{{', text, re.MULTILINE)
        if match and match.start() > 0:
            cleanings.append(f'preamble({match.start()})')
            text = text[match.start():]
        trimmed = self._trim_after_root_close(text)
        if len(trimmed) < len(text):
            cleanings.append(f'epilogue({len(text)-len(trimmed)})')
            text = trimmed

        # Fix :> ScalarValues:: → : ScalarValues:: (subsetting vs typing)
        text = re.sub(r':>\s*(ScalarValues::\w+)', r': \1', text)

        # Fix multi-line assert constraints missing 'and'
        _and_pattern = re.compile(
            r'(\S+\s*(?:>=|<=|>|<|==)\s*\S+)\s*\n(\s+)(\S+\s*(?:>=|<=|>|<|==))'
        )
        while _and_pattern.search(text):
            text = _and_pattern.sub(r'\1 and\n\2\3', text)

        # Fix LLM-generated attribute names with spaces
        text = re.sub(
            r'(attribute\s+)(\w+)\s+(\w+)(\s*[:,;:>])',
            lambda m: m.group(1) + m.group(2) + m.group(3)[0].upper() + m.group(3)[1:] + m.group(4),
            text
        )

        # Fix ISQ::angle → ISQ::planeAngle
        text = re.sub(r'ISQ::angle\b', 'ISQ::planeAngle', text)

        # Fix [deg] → ['°']
        text = re.sub(r"\[deg\]", "['°']", text)

        # Fix MHz/GHz → Hz with scientific notation
        def _convert_freq(m):
            val = float(m.group(1))
            unit = m.group(2)
            if unit == 'MHz':
                return f'{val}E6 [Hz]'
            elif unit == 'GHz':
                return f'{val}E9 [Hz]'
            return m.group(0)
        text = re.sub(r'(\d+\.?\d*)\s*\[(MHz|GHz)\]', _convert_freq, text)

        # Fix ISQ::angularSpeed
        text = re.sub(r':>\s*ISQ::angularSpeed\b', ': ScalarValues::Real', text)

        # Fix ['°/s']
        text = re.sub(r"\s*\['°/s'\]", " // [°/s]", text)

        # Fix 'attribute state' — 'state' é keyword SysML v2
        text = re.sub(r'\battribute\s+state\s*:', 'attribute federativeState :', text)

        # Fix inteiros
        def _fix_large_int(m):
            val = int(m.group(1))
            if val > 2147483647:
                return f'{val:.2E} {m.group(2)}'
            return m.group(0)
        text = re.sub(r'\b(\d{10,})\s*(\[)', _fix_large_int, text)

        text = text.replace('\r\n', '\n').strip() + '\n'
        if cleanings:
            print(f'\n[CLEAN] {pkg_name}: {cleanings}')
        return text, cleanings


    def _refine_package(self, pkg_name):
        """Refines a package with RAG: retrieves similar Apollo 11 examples
        and injects them into the prompt as a ground-truth reference.

        CHANGE v2→v3: adds RAG context before the refinement prompt.
        """
        skeleton = self.skeletons[pkg_name]
        max_tokens = P4_MAX_TOKENS[pkg_name]

        # ── RAG: retrieve similar Apollo 11 examples ──
        retrieved = retrieve_sysml_examples(
            query=f'{pkg_name} SysML v2 attribute declarations ISQ units',
            top_k=P4_RAG_TOP_K,
            pkg_name_filter=pkg_name,
        )
        # If no exact match was found, search by layer
        if not retrieved or retrieved[0].get('package_name') != pkg_name:
            layer = PACKAGE_TO_LAYER.get(pkg_name, 'technical')
            retrieved = retrieve_sysml_examples(
                query=f'{pkg_name} SysML v2 attribute declarations ISQ units',
                top_k=P4_RAG_TOP_K,
                layer_filter=layer,
            )
        rag_context = format_rag_context(retrieved, 'GROUND TRUTH REFERENCE — Use as syntactic pattern')
        if rag_context:
            print(f'  [RAG] {pkg_name}: {len(retrieved)} examples retrieved ({sum(len(r["text"]) for r in retrieved):,} chars)')

        # ── Build prompt (original + RAG context) ──
        if pkg_name == 'TechnicalComponentsPackage':
            prompt = _p4_prompt_tc(skeleton, self.mission_data)
        elif pkg_name == 'AnalysisPackage':
            prompt = _p4_prompt_analysis(skeleton, self.mission_data)
        elif pkg_name == 'CalculationsPackage':
            prompt = _p4_prompt_calculations(skeleton, self.mission_data)
        elif pkg_name == 'ContextPackage':
            prompt = _p4_prompt_context(skeleton, self.mission_data)
        elif pkg_name == 'MissionPackage':
            prompt = _p4_prompt_mission(skeleton, self.mission_data)
        elif pkg_name in ('MissionRequirementsPackage',
                          'FunctionalRequirementsPackage',
                          'TechnicalRequirementsPackage'):
            prompt = _p4_prompt_requirements(skeleton, self.mission_data,
                                             pkg_name, rag_context)
        elif pkg_name == 'MissionExecutionPackage':
            prompt = _p4_prompt_execution(skeleton, self.mission_data,
                                          rag_context)
        else:
            raise ValueError(f'Unknown LLM package: {pkg_name}')

        # Inject RAG context BEFORE the prompt (except packages that already receive it internally)
        internally_rag = ('MissionRequirementsPackage', 'FunctionalRequirementsPackage',
                          'TechnicalRequirementsPackage', 'MissionExecutionPackage')
        if rag_context and pkg_name not in internally_rag:
            prompt = f'{rag_context}\n\n---\n\n{prompt}'

        print(f'\n{"="*60}\n[P4] Refining: {pkg_name} (max_tokens={max_tokens})\n{"="*60}')
        t0 = time.time()
        full_text, t_in, t_out, conts = self._call_with_continuation(prompt, max_tokens)
        self.cost_total_in += t_in
        self.cost_total_out += t_out
        full_text, _ = self._clean_llm_output(full_text, pkg_name)
        residuals = len(re.findall(r'/\* TODO\[P4\]', full_text))

        if residuals > 0:
            print(f'\n[RETRY] {residuals} TODO[P4] residuais...')
            retry_prompt = (f'The output still has {residuals} TODO[P4]. '
                          f'Output the COMPLETE file again with ALL resolved.\n\n{prompt}')
            full_text, t_in2, t_out2, conts2 = self._call_with_continuation(retry_prompt, max_tokens)
            self.cost_total_in += t_in2
            self.cost_total_out += t_out2
            t_in += t_in2; t_out += t_out2; conts += conts2
            full_text, _ = self._clean_llm_output(full_text, pkg_name)
            residuals = len(re.findall(r'/\* TODO\[P4\]', full_text))

        status = 'OK' if residuals == 0 else f'INCOMPLETE({residuals})'
        elapsed = time.time() - t0
        print(f'\n[P4] {pkg_name}: {len(full_text):,} chars | {t_in:,}+{t_out:,} tok | {elapsed:.1f}s | {status}')
        return {'content': full_text, 'tokens_in': t_in, 'tokens_out': t_out,
                'continuations': conts, 'elapsed_s': elapsed, 'status': status,
                'residuals': residuals, 'type': 'llm'}


    def _passthrough(self, pkg_name):
        return {'content': self.skeletons[pkg_name], 'tokens_in': 0, 'tokens_out': 0,
                'continuations': 0, 'elapsed_s': 0.0, 'status': 'passthrough',
                'residuals': 0, 'type': 'passthrough'}

    def run(self):
        t0 = time.time()
        for pkg in P4_LLM_PACKAGES:
            if pkg in self.skeletons:
                self.results[pkg] = self._refine_package(pkg)
        for pkg in P4_PASSTHROUGH:
            if pkg in self.skeletons:
                self.results[pkg] = self._passthrough(pkg)
        # Passthrough for any remaining packages (root model, future additions)
        for pkg in self.skeletons:
            if pkg not in self.results:
                self.results[pkg] = self._passthrough(pkg)
        cost = estimate_cost(self.cost_total_in, self.cost_total_out)
        print(f'\n[P4] Finished in {time.time()-t0:.1f}s | ${cost:.4f}')
        return self.results


# ── run_p4: wrapper State → State ──────────────────────────

def run_p4(state: dict) -> dict:
    """RefinementAgent (P4): skeletons + JSON → refined packages.

    State input: skeletons, mission_json
    Output: refined_packages {pkg_name: sysml_str}
    """
    print(f'\n{"="*60}')
    print(f'  P4 — RefinementAgent (semantic LLM)')
    print(f'{"="*60}')
    t0 = time.time()

    agent = RefinementAgent(
        skeletons=state['skeletons'],
        mission_data=state['mission_json'],
        client=CLIENT,
    )
    results = agent.run()

    # Extract content into the state
    refined_packages = {pkg: r['content'] for pkg, r in results.items()}

    # Save to Drive
    ref_dir = REFINEMENT_DIR / f'refined_{state["run_timestamp"]}'
    write_packages_to_dir(refined_packages, ref_dir, 'P4 refined')

    elapsed = time.time() - t0
    cost = estimate_cost(agent.cost_total_in, agent.cost_total_out)
    log_cost(state, 'P4_RefinementAgent', cost, elapsed,
             agent.cost_total_in, agent.cost_total_out)

    print(f'  P4 finished: {len(refined_packages)} packages, '
          f'{sum(len(v) for v in refined_packages.values()):,} chars in {elapsed:.1f}s')

    return {'refined_packages': refined_packages}


# ── Verification ───────────────────────────────────────────
print(f'P4 defined: run_p4(state) → refined_packages')
print(f'  {len(P4_LLM_PACKAGES)} LLM packages: {P4_LLM_PACKAGES}')
print(f'  {len(P4_PASSTHROUGH)} passthrough packages')
print(f'\nBlock C complete. Ready for Block D (P5 + P6). ✅')



---

## Block D — P5 (ValidatorAgent) and P6 (FixerAgent)


### Cell 8 — P5 ValidatorAgent and run_p5()

In [ ]:
# ============================================================
# Master Orchestrator
# Cell 8 — P5 (ValidatorAgent) — 3 validation layers
# ============================================================

import subprocess
from dataclasses import dataclass, field, asdict
from collections import Counter

# ── P5 constants ───────────────────────────────────────────

P5_ERROR   = 'ERROR'
P5_WARNING = 'WARNING'
P5_INFO    = 'INFO'

AUTO_SURGICAL   = 'AUTO_SURGICAL'
AUTO_TARGETED   = 'AUTO_TARGETED'      # NEW: Tier 2 (surgical LLM by snippet)
AUTO_LLM        = 'AUTO_LLM'
HUMAN_ASSISTED  = 'HUMAN_ASSISTED'
KNOWN_LIMITATION = 'KNOWN_LIMITATION'   # Pattern detected but not fixable (architectural limitation)
MONTICORE_TIMEOUT = 120


# ── Issue dataclass  ──────────────────────────────────────

@dataclass
class Issue:
    severity: str            # P5_ERROR | P5_WARNING | P5_INFO
    layer: str               # 'syntax' | 'structural' | 'coherence'
    package: str             # Package name
    code: str                # MC01, S07, C02, etc.
    message: str             # Human-readable description
    fix_strategy: str = HUMAN_ASSISTED  # AUTO_SURGICAL | AUTO_TARGETED | AUTO_LLM | HUMAN_ASSISTED | KNOWN_LIMITATION
    line: Optional[int] = None
    column: Optional[int] = None
    current_value: Optional[str] = None
    expected_value: Optional[str] = None
    suggested_fix: Optional[str] = None       # Renamed from suggested_action
    regex_pattern: Optional[str] = None       # NEW: regex search→replace for P6
    spec_reference: Optional[str] = None      # NEW: spec reference (§9.8.4, etc.)
    affected_lines: list = field(default_factory=list)  # ALL lines with occurrence (for SLOC metrics)


# ── Static dictionaries (spec-anchored) ────────────────────

# §3.1 — Valid ISQ features (OMG SysML v2 Part 1, §9.8.4)
VALID_ISQ_FEATURES = {
    # ISQ Base (§9.8.4.2) — ISO 80000-1
    'mass', 'length', 'duration', 'time', 'electricCurrent',
    'thermodynamicTemperature', 'amountOfSubstance', 'luminousIntensity',
    # ISQSpaceTime (ISO 80000-3)
    'speed', 'acceleration', 'angularSpeed', 'angularAcceleration',
    'planeAngle', 'solidAngle', 'area', 'volume', 'angularFrequency',
    # ISQMechanics (ISO 80000-4)
    'force', 'pressure', 'power', 'energy', 'torque',
    'density', 'momentum', 'massFlowRate', 'volumeFlowRate',
    # ISQElectromagnetism (IEC 80000-6)
    'frequency', 'electricCharge', 'electricPotential',
    'capacitance', 'resistance', 'inductance', 'magneticFlux',
    # ISQThermodynamics (ISO 80000-5)
    'heatCapacity', 'entropy', 'irradiance', 'thermalConductivity',
    # ISQLight (ISO 80000-7)
    'luminousFlux', 'illuminance',
}

# Common aliases that LLMs invent → suggested correction
ISQ_ALIASES = {
    'angle': 'planeAngle',
    'velocity': 'speed',
    'weight': 'force',
    'temperature': 'thermodynamicTemperature',
    'distance': 'length',
    'height': 'length',
    'width': 'length',
    'depth': 'length',
    'radius': 'length',
    'altitude': 'length',
    'range': 'length',
    'current': 'electricCurrent',
    'voltage': 'electricPotential',
}

# §3.2 — Valid units (SI §9.8.6 + CoSMAQuantitiesAndUnitsPackage)
VALID_UNIT_SYMBOLS = {
    # SI base (§9.8.6)
    'kg', 'm', 's', 'A', 'K', 'mol', 'cd',
    # SI derived
    'Hz', 'N', 'Pa', 'J', 'W', 'V', 'rad',
    # SI time
    'h', 'min',
    # CoSMA extensions (CoSMAQuantitiesAndUnitsPackage.sysml)
    'kN', 'kPa', 'MN', 'gn', 'kB', 'yr',
    'kn', 'nmi', 'ftm',
    # CoSMA derived/custom
    'km', 'mm',
    # Quoted units
    "'°'", "'arcmin'", "'arcsec'", "'%'", "'$'",
    "'kN'", "'kPa'", "'s'", "'m/s'", "'h'",
    # Compound SI units
    'm/s', 'kg/h', 'm/s²',
}

# Useful conversions
INVALID_UNITS_MAP = {
    'deg': "'°'",
    'MHz': None,            # Convert value * 1E6 [Hz]
    'GHz': None,            # Convert value * 1E9 [Hz]
    'knot': 'kn',
    'knots': 'kn',
    'nautical mile': 'nmi',
    'ft': None,             # Convert to [m] (factor 0.3048)
    'in': None,             # Convert to [m] (factor 0.0254)
    'lb': None,             # Convert to [kg] (factor 0.4536)
    'mi': None,             # Convert to [km] (factor 1.609)
}

UNIT_CONVERSION_FACTORS = {
    'kHz': ('Hz', 1e3),
    'MHz': ('Hz', 1e6),
    'GHz': ('Hz', 1e9),
    'knot': ('kn', 1.852),
    'knots': ('kn', 1.852),
    'nautical mile': ('nmi', 1.852),
    'kHz': ('Hz', 1e3),
    'km/h': ('m/s', 0.2778),
    'ft':  ('m',  0.3048),
    'in':  ('m',  0.0254),
    'lb':  ('kg', 0.4536),
    'mi':  ('km', 1.609),
}

# §3.3 — CoSMA basetypes (Helle & Schramm 2026, Table 1)
COSMA_PART_BASETYPES = {
    'System', 'HardwareComponent', 'SoftwareComponent',
    'SimpleThing', 'CompositeThing',
    # CoSMA abstract types
    'Stakeholder', 'Mission', 'Context', 'Phase',
    'SystemOfSystems', 'ExternalSystem',
    # CoSMA abstract types (from CoSMAPackage.sysml)
    'Capability', 'LogicalComponent',
    # Extension types (from GT Apollo 11)
    'PowerConsumer', 'PowerProvider', 'FuelledComponent',
    'PropelledSpacecraft', 'EVASystem', 'Spacecraft', 'LaunchSystem',
}


# ── Layer 1: SyntaxValidator (MontiCore) ───────────────────

# ── SLOC helpers (line-of-code-based metrics) ──────────────

def _line_at(content: str, pos: int) -> int:
    """Returns the line number (1-indexed) for a character position in the content."""
    return content[:pos].count('\n') + 1


def _all_lines_for(content: str, regex, limit: int = 500) -> list:
    """Returns a list of line numbers (1-indexed) for ALL regex occurrences."""
    lines = []
    for m in re.finditer(regex, content):
        lines.append(_line_at(content, m.start()))
        if len(lines) >= limit:
            break
    return lines


def _count_sloc(content: str) -> dict:
    """Classifies lines of a SysML v2 package into SLOC, blank and comment.

    Rule: SLOC = every line that is not blank nor a standalone //.
    Includes doc /* ... */, @Rationale, imports, defs, attributes, constraints, }.
    Excludes only blank lines and lines starting with //.
    """
    lines = content.split('\n')
    total = len(lines)
    blank = 0
    comment = 0
    sloc = 0
    for line in lines:
        stripped = line.strip()
        if not stripped:
            blank += 1
        elif stripped.startswith('//'):
            comment += 1
        else:
            sloc += 1
    return {'loc': total, 'blank': blank, 'comment': comment, 'sloc': sloc}

class SyntaxValidator:
    RE_ERROR = re.compile(r'\[ERROR\]\s+(?P<file>[^:]+):<(?P<line>\d+),(?P<col>\d+)>:\s+(?P<msg>.+)')
    RE_WARN  = re.compile(r'\[WARN\]\s+(?P<msg>.+)')
    RE_SYMBOL_WARN = re.compile(r'Defining symbol for .+ was not set')
    RE_SYMBOL_NAME = re.compile(r'Defining symbol for (\S+) was not set')

    VERIFY_TIMEOUT = 90

    # Symbol prefixes from the SysML v2 / KerML standard library
    # These are suppressed from the report (MontiCore single-file parser limitation)
    STDLIB_SYMBOL_PREFIXES = {
        'ScalarValues.', 'ISQ.', 'SI.', 'USCustomaryUnits.',
        'Quantities.', 'NumericalFunctions.', 'ScalarFunctions.',
        'BaseFunctions.', 'DataFunctions.', 'ControlFunctions.',
        'States.', 'Actions.', 'Events.', 'Occurrences.',
        'OccurrenceFunctions.', 'Time.', 'Metaobjects.',
        'Objects.', 'Performances.', 'Transfers.',
        'Links.', 'Items.', 'Parts.', 'Ports.',
        'Connections.', 'Interfaces.', 'Allocations.',
        'Requirements.', 'Constraints.', 'Analysis.',
        'Calculations.', 'Cases.', 'Views.', 'Viewpoints.',
        'Metadata.', 'ShapeItems.',
    }

    def __init__(self, jar_path, timeout=MONTICORE_TIMEOUT, packages=None):
        self.jar_path = jar_path
        self.timeout = timeout
        self.available = False
        self.packages = packages or {}
        self.stdlib_suppressed_count = 0  # Count of suppressed MC02s

        if jar_path is not None and Path(str(jar_path)).exists():
            self._verify_jar()

    def _verify_jar(self):
        try:
            test = subprocess.run(
                ['java', '-jar', str(self.jar_path), '-h'],
                capture_output=True, text=True, timeout=self.VERIFY_TIMEOUT)
            if test.returncode != 0 and 'usage' not in (test.stdout + test.stderr).lower():
                print(f'  ⚠️ MontiCore JAR not executable (returncode={test.returncode})')
                self.available = False
            else:
                print(f'  ✅ MontiCore JAR verified: {self.jar_path}')
                self.available = True
        except subprocess.TimeoutExpired:
            print(f'  ⚠️ MontiCore JAR verification timeout (>{self.VERIFY_TIMEOUT}s)')
            self.available = False
        except Exception as e:
            print(f'  ⚠️ MontiCore JAR verification failed: {e}')
            self.available = False

    def _trace_symbol(self, symbol, source_pkg):
        """Searches which package defines the symbol."""
        for pkg_name, content in self.packages.items():
            if pkg_name == source_pkg:
                continue
            if re.search(rf'\bdef\s+{re.escape(symbol)}\b', content):
                return f'Defined in {pkg_name} — verify in Eclipse'
            if re.search(rf'\b(?:part|action|state|port)\s+{re.escape(symbol)}\s*:', content):
                return f'Instantiated in {pkg_name} — verify in Eclipse'
        return f'Symbol not found in generated packages — verify standard library in Eclipse'

    def _is_stdlib_symbol(self, symbol):
        """Checks whether the symbol belongs to the standard library."""
        return any(symbol.startswith(prefix) for prefix in self.STDLIB_SYMBOL_PREFIXES)

    def validate_file(self, filepath, pkg_name):
        issues = []
        if not self.available:
            print(f'  ⚠️ [{pkg_name}] MontiCore DISABLED — Layer 1 skipped')
            issues.append(Issue(severity=P5_WARNING, layer='syntax', package=pkg_name,
                code='MC00', message='MontiCore JAR not available — Layer 1 skipped (no formal syntactic validation)',
                fix_strategy=HUMAN_ASSISTED))
            return issues
        try:
            result = subprocess.run(
                ['java', '-jar', str(self.jar_path), '-i', str(filepath), '-nc'],
                capture_output=True, text=True, timeout=self.timeout)
            output = result.stdout + '\n' + result.stderr
            for m in self.RE_ERROR.finditer(output):
                msg = m.group('msg').strip()
                # MC07: known limitations of MontiCore v7.6.2
                # (a) nonunique: keyword valid in the spec but not supported by the parser
                # (b) reserved words: valid SysML v2 identifiers that conflict with the MontiCore grammar
                is_parser_limitation = (
                    'nonunique' in msg
                    or ('no viable alternative' in msg and re.search(r"'state\w+'", msg))
                )
                if is_parser_limitation:
                    issues.append(Issue(severity=P5_INFO, layer='syntax', package=pkg_name,
                        code='MC07', message=f'MontiCore parser limitation: {msg}',
                        line=int(m.group('line')), column=int(m.group('col')),
                        fix_strategy=KNOWN_LIMITATION,
                        spec_reference='MontiCore v7.6.2 grammar limitation (valid SysML v2 construct not supported by parser)'))
                    continue
                issues.append(Issue(severity=P5_ERROR, layer='syntax', package=pkg_name,
                    code='MC01', message=msg,
                    line=int(m.group('line')), column=int(m.group('col')),
                    fix_strategy=AUTO_LLM,
                    spec_reference='MontiCore SysML v2 grammar'))
            for m in self.RE_WARN.finditer(output):
                msg = m.group('msg').strip()
                if self.RE_SYMBOL_WARN.search(msg):
                    # Extract symbol name
                    sym_match = self.RE_SYMBOL_NAME.search(msg)
                    if sym_match:
                        symbol = sym_match.group(1)
                        # Suppress if standard library
                        if self._is_stdlib_symbol(symbol):
                            self.stdlib_suppressed_count += 1
                            continue
                        # Project symbol — keep with traceability
                        trace = self._trace_symbol(symbol, pkg_name)
                        issues.append(Issue(severity=P5_INFO, layer='syntax', package=pkg_name,
                            code='MC02', message=f'Symbol resolution: {msg}',
                            fix_strategy=HUMAN_ASSISTED,
                            current_value=symbol,
                            suggested_fix=trace,
                            spec_reference='MontiCore single-file parsing — verify in Eclipse'))
                    else:
                        # Fallback if regex did not match (should not happen)
                        issues.append(Issue(severity=P5_INFO, layer='syntax', package=pkg_name,
                            code='MC02', message=f'Symbol resolution: {msg}',
                            fix_strategy=HUMAN_ASSISTED))
                else:
                    issues.append(Issue(severity=P5_WARNING, layer='syntax', package=pkg_name,
                        code='MC03', message=msg, fix_strategy=AUTO_LLM,
                        spec_reference='MontiCore SysML v2 grammar'))
            if result.returncode != 0 and not any(i.code in ('MC01', 'MC07') for i in issues):
                issues.append(Issue(severity=P5_ERROR, layer='syntax', package=pkg_name,
                    code='MC04', message=f'MontiCore returncode={result.returncode}',
                    fix_strategy=AUTO_LLM))
        except subprocess.TimeoutExpired:
            issues.append(Issue(severity=P5_WARNING, layer='syntax', package=pkg_name,
                code='MC05', message=f'MontiCore timeout after {self.timeout}s',
                fix_strategy=HUMAN_ASSISTED))
        except Exception as e:
            issues.append(Issue(severity=P5_WARNING, layer='syntax', package=pkg_name,
                code='MC06', message=f'MontiCore exception: {e}',
                fix_strategy=HUMAN_ASSISTED))
        return issues


# ── MontiCore availability guard (polling loop) ────────────
# Ensures MontiCore JAR is available before validating.
# If absent: prints instructions and polls until it appears.
# Goal: pipeline does not break — after timeout, continues without Layer 1 with a warning.

def _ensure_monticore(expected_path, poll_interval=15, max_wait=600):
    """Locates and validates MontiCore JAR, polling if needed.

    Returns valid jar_path or None (after timeout).
    """
    expected = Path(str(expected_path))

    # ── 1. Quick check: default path ──
    if expected.exists() and expected.stat().st_size > 1_000_000:
        return expected

    # ── 2. Search alternative locations ──
    search_root = expected.parent.parent  # BASE_DIR
    candidates = list(search_root.glob('**/MCSysMLv2.jar'))
    for c in candidates:
        if c.stat().st_size > 1_000_000:
            print(f'  ✅ MontiCore found at alternative location: {c}')
            return c

    # ── 3. Not found — polling loop with instructions ──
    print(f'\n{"="*70}')
    print(f'  ⚠️  MontiCore JAR NOT FOUND — Layer 1 MANDATORY')
    print(f'{"="*70}')
    print(f'  Formal syntactic validation (MontiCore) is mandatory.')
    print(f'  The pipeline will wait until the JAR is available.\n')
    print(f'  📥 Option 1 — wget (recommended):')
    print(f'     Open another Colab cell and run:')
    print(f'     !wget -q -O "{expected}" https://www.monticore.de/download/MCSysMLv2.jar\n')
    print(f'  📥 Option 2 — manual upload:')
    print(f'     1. Download: https://www.monticore.de/download/MCSysMLv2.jar')
    print(f'     2. In Google Drive, navigate to: {expected.parent}/')
    print(f'     3. Upload the MCSysMLv2.jar file\n')
    print(f'  ⏳ Polling every {poll_interval}s (max {max_wait // 60}min)...')
    print(f'{"="*70}\n')

    elapsed = 0
    attempt = 0
    while elapsed < max_wait:
        time.sleep(poll_interval)
        elapsed += poll_interval
        attempt += 1
        mins, secs = divmod(elapsed, 60)

        # Re-check default path
        if expected.exists():
            size = expected.stat().st_size
            if size < 1_000_000:
                print(f'  [{attempt}] {mins}m{secs:02d}s — JAR found but too small '
                      f'({size:,} bytes), possibly corrupted or still downloading...')
                continue
            print(f'  [{attempt}] {mins}m{secs:02d}s — JAR found ({size:,} bytes)! Verifying...')
            return expected

        # Re-check alternatives
        candidates = list(search_root.glob('**/MCSysMLv2.jar'))
        for c in candidates:
            if c.stat().st_size > 1_000_000:
                print(f'  [{attempt}] {mins}m{secs:02d}s — JAR found at {c}!')
                return c

        print(f'  [{attempt}] {mins}m{secs:02d}s — waiting for JAR...')

    # ── 4. Timeout — continue without Layer 1 (safety valve) ──
    print(f'\n{"="*70}')
    print(f'  ❌ TIMEOUT: MontiCore not available within {max_wait // 60}min')
    print(f'  The pipeline will continue WITHOUT syntactic validation (Layer 1).')
    print(f'  ⚠️ SCR will be marked as N/A — INCOMPLETE results.')
    print(f'{"="*70}\n')
    return None


# ── Layer 2: StructuralValidator (S01–S14) ─────────────────

class StructuralValidator:
    RE_MARKDOWN_FENCE = re.compile(r'```')
    RE_TODO_P4 = re.compile(r'/\*\s*TODO\[P4\]')
    RE_DEFINITION = re.compile(
        r'^\s*(?:part|action|state|item|requirement|attribute|port|constraint|analysis|'
        r'concern|verification|use\s+case|view|viewpoint|allocation|connection|flow|'
        r'interface|calc|enum)\s+def\s+(\w+)', re.MULTILINE)
    RE_IMPORT = re.compile(r'private\s+import\s+([\w:]+)::\*;')

    # S07: subsetting (:>) used with ScalarValues (should be typing :)
    RE_SCALAR_SUBSETTING = re.compile(r':>\s*(ScalarValues::\w+)')
    # S08: ISQ:: followed by feature name
    RE_ISQ_FEATURE = re.compile(r'ISQ::(\w+)')
    # S09: units inside brackets
    RE_UNIT_BRACKET = re.compile(r"\[([^\]']+)\]")
    # S10: perform with PascalCase after ::
    RE_PERFORM_PASCAL = re.compile(r'(perform\s+\w+::)([A-Z])(\w+)')
    # S11: two consecutive comparisons without 'and' (only inside assert constraint)
    RE_CONSECUTIVE_COMPARE = re.compile(
        r'(\S+\s*(?:>=|<=|(?<!:)(?<!:)>(?!>)|(?<!<)(?<!\')<(?!\w+>)|==)\s*\S+)\s*\n(\s+)(\S+\s*(?:>=|<=|(?<!:)(?<!:)>(?!>)|(?<!<)(?<!\')<(?!\w+>)|==))')
    # S12: attribute with space in the name (two words before : or :>)
    RE_ATTR_SPACE = re.compile(r'(attribute\s+)(?!def\b)(\w+)\s+(\w+)(\s*[:,;])')
    # S13: satisfy with bare name (no path with .)
    RE_SATISFY_BARE = re.compile(r"satisfy\s+'[^']+'\s+by\s+([A-Z]\w+)\s*;")
    # S16: transition with invalid syntax (-> instead of first/then)
    RE_TRANSITION_ARROW = re.compile(r'transition\s+(\w+)\s*->\s*(\w+)')
    # S16b: trigger accept (should be accept inline in the transition)
    RE_TRIGGER_ACCEPT = re.compile(r'trigger\s+accept\s+(\w+)\s*;')
    # S14: reference to calc def
    RE_CALC_CALL = re.compile(r'=\s*(\w+)\s*\(')
    RE_CALC_DEF = re.compile(r'calc\s+def\s+(\w+)')

    def validate(self, content, pkg_name):
        issues = []
        issues.extend(self._s01_markdown_fences(content, pkg_name))
        issues.extend(self._s02_todo_residuals(content, pkg_name))
        issues.extend(self._s03_braces_balance(content, pkg_name))
        issues.extend(self._s04_duplicate_defs(content, pkg_name))
        issues.extend(self._s05_file_start(content, pkg_name))
        issues.extend(self._s06_cosma_import(content, pkg_name))
        issues.extend(self._s07_scalar_subsetting(content, pkg_name))
        issues.extend(self._s08_isq_features(content, pkg_name))
        issues.extend(self._s09_unit_symbols(content, pkg_name))
        issues.extend(self._s10_perform_naming(content, pkg_name))
        issues.extend(self._s11_constraint_conjunction(content, pkg_name))
        issues.extend(self._s12_attribute_spaces(content, pkg_name))
        issues.extend(self._s13_satisfy_paths(content, pkg_name))
        issues.extend(self._s14_calc_crossref(content, pkg_name))
        issues.extend(self._s15_duplicate_imports(content, pkg_name))
        issues.extend(self._s16_transition_syntax(content, pkg_name))
        return issues

    # ── S01–S06  ─────────────────────────────────────

    def _s01_markdown_fences(self, content, pkg_name):
        alines = _all_lines_for(content, self.RE_MARKDOWN_FENCE)
        if alines:
            return [Issue(severity=P5_ERROR, layer='structural', package=pkg_name,
                code='S01', message='Residual markdown fences',
                fix_strategy=AUTO_SURGICAL,
                line=alines[0], affected_lines=alines,
                regex_pattern=r'```\w*\n? → (empty)',
                spec_reference='SysML v2 textual notation (no markdown)')]
        return []

    def _s02_todo_residuals(self, content, pkg_name):
        alines = _all_lines_for(content, self.RE_TODO_P4)
        if alines:
            return [Issue(severity=P5_ERROR, layer='structural', package=pkg_name,
                code='S02', message=f'Residual TODO[P4]: {len(alines)}',
                fix_strategy=AUTO_LLM,
                line=alines[0], affected_lines=alines,
                spec_reference='Pipeline convention: P4 must resolve all TODOs')]
        return []

    def _s03_braces_balance(self, content, pkg_name):
        issues = []
        depth = 0
        in_block, in_line, in_str = False, False, False
        str_char = None
        i = 0
        while i < len(content):
            c = content[i]
            if not in_block and not in_line and not in_str:
                if content[i:i+2] == '/*': in_block = True; i += 2; continue
                if content[i:i+2] == '//': in_line = True; i += 2; continue
            if in_block:
                if content[i:i+2] == '*/': in_block = False; i += 2; continue
                i += 1; continue
            if in_line:
                if c == '\n': in_line = False
                i += 1; continue
            if not in_str and c in ('"', "'"):
                in_str = True; str_char = c; i += 1; continue
            if in_str:
                if c == '\\': i += 2; continue
                if c == str_char: in_str = False
                i += 1; continue
            if c == '{': depth += 1
            elif c == '}':
                depth -= 1
                if depth < 0:
                    ln = _line_at(content, i)
                    issues.append(Issue(severity=P5_ERROR, layer='structural', package=pkg_name,
                        code='S03', message=f'Extra brace at position {i}',
                        fix_strategy=AUTO_LLM,
                        line=ln, affected_lines=[ln],
                        spec_reference='SysML v2 block structure'))
                    break
            i += 1
        if depth > 0:
            last_line = _line_at(content, len(content) - 1)
            issues.append(Issue(severity=P5_ERROR, layer='structural', package=pkg_name,
                code='S03', message=f'Unbalanced braces: depth={depth}',
                fix_strategy=AUTO_LLM,
                line=last_line, affected_lines=[last_line],
                spec_reference='SysML v2 block structure'))
        return issues

    def _s04_duplicate_defs(self, content, pkg_name):
        issues = []
        defs_with_pos = [(m.group(1), m.start()) for m in self.RE_DEFINITION.finditer(content)]
        seen = {}
        for d, pos in defs_with_pos:
            seen.setdefault(d, []).append(_line_at(content, pos))
        for d, lines in seen.items():
            if len(lines) > 1:
                issues.append(Issue(severity=P5_WARNING, layer='structural', package=pkg_name,
                    code='S04', message=f'Duplicate definition: "{d}" declared {len(lines)}x',
                    fix_strategy=AUTO_SURGICAL,
                    line=lines[1], affected_lines=lines[1:],
                    suggested_fix=f'Remove duplicate declaration of "{d}"'))
        return issues

    def _s05_file_start(self, content, pkg_name):
        stripped = content.lstrip()
        if not (stripped.startswith('//') or stripped.startswith('package')):
            return [Issue(severity=P5_ERROR, layer='structural', package=pkg_name,
                code='S05', message='File does not start with // or package',
                fix_strategy=AUTO_SURGICAL,
                line=1, affected_lines=[1])]
        return []

    def _s06_cosma_import(self, content, pkg_name):
        issues = []
        imports = set(self.RE_IMPORT.findall(content))
        cosma_framework = {'CoSMAPackage', 'CoSMAQuantitiesAndUnitsPackage', 'CoSMAViewsPackage'}
        needs_cosma = (pkg_name not in ('ProgramPackage',)
                       and not pkg_name.endswith('Model')
                       and pkg_name not in cosma_framework)
        if needs_cosma and 'CoSMAPackage' not in imports and 'CoSMAPackage' not in content:
            issues.append(Issue(severity=P5_WARNING, layer='structural', package=pkg_name,
                code='S06', message='Missing CoSMAPackage import',
                fix_strategy=AUTO_SURGICAL,
                line=1, affected_lines=[1]))

        # Mission-agnostic: OccurrenceFunctions functions require explicit import
        occurrence_fns = ('isDuring', 'isBefore', 'isAfter', 'startsBefore', 'endsBefore')
        if any(fn in content for fn in occurrence_fns):
            if 'OccurrenceFunctions' not in imports and 'OccurrenceFunctions' not in content:
                issues.append(Issue(severity=P5_WARNING, layer='structural', package=pkg_name,
                    code='S06', message='Missing OccurrenceFunctions import (required by isDuring/isBefore/isAfter)',
                    fix_strategy=AUTO_SURGICAL,
                    line=1, affected_lines=[1]))

        return issues

    # ── S07–S14  ────────────────────────────────────────

    def _s07_scalar_subsetting(self, content, pkg_name):
        """S07: :> ScalarValues::X should be : ScalarValues::X (typing, not subsetting)."""
        issues = []
        type_lines = {}  # sv_type -> [line numbers]
        for m in self.RE_SCALAR_SUBSETTING.finditer(content):
            sv_type = m.group(1) if m.lastindex else m.group(0)
            type_lines.setdefault(sv_type, []).append(_line_at(content, m.start()))
        for sv_type, alines in type_lines.items():
            issues.append(Issue(severity=P5_ERROR, layer='structural', package=pkg_name,
                code='S07',
                message=f'Subsetting (:>) used with {sv_type} ({len(alines)}x); should be typing (:)',
                fix_strategy=AUTO_SURGICAL,
                line=alines[0], affected_lines=alines,
                current_value=f':> {sv_type}',
                expected_value=f': {sv_type}',
                suggested_fix=f'Replace :> with : before ScalarValues types',
                regex_pattern=rf':>\s*({re.escape(sv_type)}) → : \1',
                spec_reference='OMG SysML v2 Part 1, §7.4 (Feature Typing vs Subsetting)'))
        return issues

    def _s08_isq_features(self, content, pkg_name):
        """S08: ISQ::featureName must be in VALID_ISQ_FEATURES."""
        issues = []
        # Collect distinct invalid features with line numbers
        invalid_features = {}  # feat -> [line numbers]
        for m in self.RE_ISQ_FEATURE.finditer(content):
            feat = m.group(1)
            if feat not in VALID_ISQ_FEATURES:
                invalid_features.setdefault(feat, []).append(_line_at(content, m.start()))

        for feat, alines in invalid_features.items():
            correction = ISQ_ALIASES.get(feat)
            if correction:
                issues.append(Issue(severity=P5_ERROR, layer='structural', package=pkg_name,
                    code='S08',
                    message=f'Invalid ISQ feature: ISQ::{feat} → ISQ::{correction} ({len(alines)}x)',
                    fix_strategy=AUTO_SURGICAL,
                    line=alines[0], affected_lines=alines,
                    current_value=f'ISQ::{feat}',
                    expected_value=f'ISQ::{correction}',
                    suggested_fix=f'Replace ISQ::{feat} with ISQ::{correction}',
                    regex_pattern=rf'ISQ::{re.escape(feat)}\b → ISQ::{correction}',
                    spec_reference='OMG SysML v2 Part 1, §9.8.4 (ISQ)'))
            else:
                issues.append(Issue(severity=P5_WARNING, layer='structural', package=pkg_name,
                    code='S08',
                    message=f'Unknown ISQ feature: ISQ::{feat} ({len(alines)}x, no known alias)',
                    fix_strategy=HUMAN_ASSISTED,
                    line=alines[0], affected_lines=alines,
                    current_value=f'ISQ::{feat}',
                    spec_reference='OMG SysML v2 Part 1, §9.8.4 (ISQ)'))
        return issues

    def _s09_unit_symbols(self, content, pkg_name):
        """S09: Units inside [] must be in VALID_UNIT_SYMBOLS."""
        issues = []
        for m in self.RE_UNIT_BRACKET.finditer(content):
            unit = m.group(1).strip()
            # Skip SysML v2 multiplicities (not units)
            if re.match(r'^\d*(\.\.\d+|\.\.\*)?$', unit) or unit == '*':
                continue
            # Skip value ranges like [1, 10]
            if re.match(r'^\d+\s*,\s*\d+$', unit):
                continue
            # Skip already-valid units
            if unit in VALID_UNIT_SYMBOLS:
                continue
            # Skip quoted units
            if unit.startswith("'") and unit.endswith("'"):
                if unit in VALID_UNIT_SYMBOLS:
                    continue
            # Check whether it is an invalid unit with direct mapping
            if unit in INVALID_UNITS_MAP:
                replacement = INVALID_UNITS_MAP[unit]
                if replacement is not None:
                    issues.append(Issue(severity=P5_ERROR, layer='structural', package=pkg_name,
                        code='S09',
                    line=_line_at(content, m.start()), affected_lines=[_line_at(content, m.start())],
                        message=f'Invalid unit [{unit}] → [{replacement}]',
                        fix_strategy=AUTO_SURGICAL,
                        current_value=f'[{unit}]',
                        expected_value=f'[{replacement}]',
                        suggested_fix=f'Replace [{unit}] with [{replacement}]',
                        regex_pattern=rf'\[{re.escape(unit)}\\] → [{replacement}]',
                        spec_reference='OMG SysML v2 Part 1, §9.8.5–9.8.6 (SI Units)'))
                else:
                    issues.append(Issue(severity=P5_ERROR, layer='structural', package=pkg_name,
                        code='S09',
                    line=_line_at(content, m.start()), affected_lines=[_line_at(content, m.start())],
                        message=f'Unit [{unit}] requires value conversion (does not exist in SI/CoSMA)',
                        fix_strategy=AUTO_SURGICAL,
                        current_value=f'[{unit}]',
                        suggested_fix=f'Convert value and unit (e.g., MHz→Hz*1E6)',
                        spec_reference='OMG SysML v2 Part 1, §9.8.5–9.8.6 (SI Units)'))
            # Check whether it is a convertible unit (mission-agnostic: any unit in UNIT_CONVERSION_FACTORS)
            elif unit in UNIT_CONVERSION_FACTORS:
                target_unit, factor = UNIT_CONVERSION_FACTORS[unit]
                issues.append(Issue(severity=P5_ERROR, layer='structural', package=pkg_name,
                    code='S09',
                    line=_line_at(content, m.start()), affected_lines=[_line_at(content, m.start())],
                    message=f'Unit [{unit}] requires value conversion (does not exist in SI/CoSMA)',
                    fix_strategy=AUTO_SURGICAL,
                    current_value=f'[{unit}]',
                    suggested_fix=f'Convert value and unit: [{unit}] → [{target_unit}] (×{factor})',
                    spec_reference='OMG SysML v2 Part 1, §9.8.5–9.8.6 (SI Units)'))
        return issues

    def _s10_perform_naming(self, content, pkg_name):
        """S10: perform Namespace::InstanceName must be camelCase."""
        issues = []
        seen = {}  # full_name -> [lines]
        for m in self.RE_PERFORM_PASCAL.finditer(content):
            first_char = m.group(2)
            rest = m.group(3)
            full_name = f'{first_char}{rest}'
            seen.setdefault(full_name, []).append(_line_at(content, m.start()))
        for full_name, alines in seen.items():
            camel = f'{full_name[0].lower()}{full_name[1:]}'
            issues.append(Issue(severity=P5_ERROR, layer='structural', package=pkg_name,
                code='S10',
                message=f'perform instance PascalCase: {full_name} → {camel}',
                fix_strategy=AUTO_SURGICAL,
                line=alines[0], affected_lines=alines,
                current_value=full_name,
                expected_value=camel,
                suggested_fix=f'Convert perform instance name to camelCase',
                regex_pattern=r'perform\s+(\w+)::([A-Z])(\w+) → (camelCase conversion)',
                spec_reference='OMG SysML v2 Part 1, §8.2 (Instance Naming)'))
        return issues

    def _s11_constraint_conjunction(self, content, pkg_name):
        """S11: Two consecutive comparisons without 'and' — only inside assert constraint."""
        issues = []
        # Extract only assert constraint { ... } blocks to avoid false positives
        # with :> (subsetting) and <name> (short names) outside constraints
        alines = []
        for bm in re.finditer(r'assert\s+constraint\s*\{([^}]+)\}', content, re.DOTALL):
            block = bm.group(1)
            block_offset = bm.start(1)
            for cm in self.RE_CONSECUTIVE_COMPARE.finditer(block):
                alines.append(_line_at(content, block_offset + cm.start()))
        if alines:
            issues.append(Issue(severity=P5_ERROR, layer='structural', package=pkg_name,
                code='S11',
                message=f'Consecutive comparisons without "and" ({len(alines)}x)',
                fix_strategy=AUTO_SURGICAL,
                line=alines[0], affected_lines=alines,
                suggested_fix='Insert "and" between consecutive comparisons',
                spec_reference='OMG SysML v2 Part 1, §9.3 (Constraint Expressions)'))
        return issues

    def _s12_attribute_spaces(self, content, pkg_name):
        """S12: attribute with space in the name (two words before : or :>)."""
        issues = []
        for m in self.RE_ATTR_SPACE.finditer(content):
            word1, word2 = m.group(2), m.group(3)
            merged = f'{word1}{word2[0].upper()}{word2[1:]}' if word2 else word1
            ln = _line_at(content, m.start())
            issues.append(Issue(severity=P5_ERROR, layer='structural', package=pkg_name,
                code='S12',
                message=f'Space in attribute name: "{word1} {word2}" → "{merged}"',
                line=ln, affected_lines=[ln],
                fix_strategy=AUTO_SURGICAL,
                current_value=f'{word1} {word2}',
                expected_value=merged,
                suggested_fix=f'Merge attribute name to camelCase',
                regex_pattern=rf'(attribute\s+){re.escape(word1)}\s+{re.escape(word2)} → \1{merged}',
                spec_reference='OMG SysML v2 Part 1, §7.1 (Identifiers)'))
        return issues

    def _s13_satisfy_paths(self, content, pkg_name):
        """S13: satisfy with bare name (no path with .) — HUMAN_ASSISTED."""
        issues = []
        for m in self.RE_SATISFY_BARE.finditer(content):
            bare_name = m.group(1)
            ln = _line_at(content, m.start())
            issues.append(Issue(severity=P5_WARNING, layer='structural', package=pkg_name,
                code='S13',
                message=f'satisfy bare name "{bare_name}" — requires full path (e.g.: mission.phases...)',
                line=ln, affected_lines=[ln],
                fix_strategy=HUMAN_ASSISTED,
                current_value=bare_name,
                suggested_fix='Resolve full satisfy path via mission/phase/operation hierarchy',
                spec_reference='CoSMA metamodel rel. #28 (satisfy path resolution)'))
        return issues

    def _s14_calc_crossref(self, content, pkg_name):
        """S14: Reference to calc def that may not exist in CalculationsPackage."""
        if 'CalculationsPackage' not in content:
            return []
        issues = []
        local_defs = set(self.RE_CALC_DEF.findall(content))
        builtins = {'min', 'max', 'abs', 'sqrt', 'sum', 'round', 'ceil', 'floor'}
        call_lines = {}  # call_name -> [lines]
        for m in self.RE_CALC_CALL.finditer(content):
            call_name = m.group(1)
            if call_name not in local_defs and call_name not in builtins:
                call_lines.setdefault(call_name, []).append(_line_at(content, m.start()))
        for call_name, alines in call_lines.items():
            issues.append(Issue(severity=P5_WARNING, layer='structural', package=pkg_name,
                code='S14',
                message=f'Calc def "{call_name}" referenced but not declared locally',
                fix_strategy=AUTO_TARGETED,
                line=alines[0], affected_lines=alines,
                current_value=call_name,
                suggested_fix=f'Verify calc def exists in CalculationsPackage or remove reference',
                spec_reference='Package dependency: AnalysisPackage imports CalculationsPackage'))
        return issues

    def _s15_duplicate_imports(self, content, pkg_name):
        """S15: Duplicate imports in the same package."""
        issues = []
        imp_lines = {}  # imp -> [lines]
        for m in self.RE_IMPORT.finditer(content):
            imp = m.group(1)
            imp_lines.setdefault(imp, []).append(_line_at(content, m.start()))
        for imp, lines_list in imp_lines.items():
            if len(lines_list) > 1:
                issues.append(Issue(severity=P5_WARNING, layer='structural', package=pkg_name,
                    code='S15',
                    message=f'Duplicate import: "{imp}::*" appears {len(lines_list)}x',
                    fix_strategy=AUTO_SURGICAL,
                    line=lines_list[1], affected_lines=lines_list[1:],
                    current_value=f'private import {imp}::*;',
                    suggested_fix=f'Remove {len(lines_list) - 1} duplicate import(s)',
                    spec_reference='OMG SysML v2 Part 1, §8.2.2.5 (Package Elements)'))
        return issues

    def _s16_transition_syntax(self, content, pkg_name):
        """S16: Transition with invalid syntax (-> instead of first/then, trigger accept)."""
        issues = []
        alines = _all_lines_for(content, self.RE_TRANSITION_ARROW)
        if alines:
            issues.append(Issue(severity=P5_ERROR, layer='structural', package=pkg_name,
                code='S16',
                message=f'Invalid transition syntax: "->" used {len(alines)}x (should be "first ... then ...")',
                fix_strategy=AUTO_SURGICAL,
                line=alines[0], affected_lines=alines,
                suggested_fix='Convert "transition A -> B" to "transition first A then B"',
                spec_reference='OMG SysML v2 Part 1, §8.2.2.18.3 (Transition Usages)'))
        return issues

# ── Layer 3: CoherenceValidator (C01–C10) ──────────────────
# Deterministic cross-package validation.
# Receives all packages as dict[str, str] and verifies
# coherence of references between them.

class CoherenceValidator:
    RE_IMPORT = re.compile(r'private\s+import\s+([\w:]+)::\*;')
    RE_PORT_USAGE = re.compile(r'port\s+\w+\s*:\s*(\w+)\s*;')
    RE_PORT_DEF = re.compile(r'port\s+def\s+(\w+)')
    RE_SATISFY = re.compile(r"satisfy\s+'[^']+'\s+by\s+(\w[\w.]*)\s*;")
    RE_PART_DEF = re.compile(r'(?:part|action|state)\s+def\s+(\w+)')
    RE_PART_INSTANCE = re.compile(r'(?:part|action|state)\s+(\w+)\s*:')
    RE_CALC_DEF = re.compile(r'calc\s+def\s+(\w+)')
    RE_CALC_REF = re.compile(r'=\s*(\w+)\s*\(')
    RE_PARTDEF_BASETYPE = re.compile(r'part\s+def\s+(\w+)\s*:>\s*(\w+)')
    RE_ABSTRACT_PARTDEF = re.compile(r'abstract\s+part\s+def\s+(\w+)')

    # C06: part instance typing — part instanceName : TypeDef (neither :> nor :>>)
    RE_PART_TYPED_USAGE = re.compile(r'(?:part)\s+(\w+)\s*:\s*(?!>)(\w+)')
    # C07: action instance typing — action instanceName : ActionDef
    RE_ACTION_TYPED_USAGE = re.compile(r'(?:action)\s+(\w+)\s*:\s*(?!>)(\w+)')
    # C10: perform target — perform Namespace::actionName
    RE_PERFORM_TARGET = re.compile(r'perform\s+(\w+)::(\w+)')

    COSMA_FRAMEWORK_PKGS = {'CoSMAPackage', 'CoSMAQuantitiesAndUnitsPackage', 'CoSMAViewsPackage'}

    def __init__(self, packages):
        self.packages = packages  # dict[str, str] — all packages

    def run_all(self):
        """Runs all cross-package checks. Returns list[Issue]."""
        issues = []
        issues.extend(self._c03_satisfy_target_exists())
        issues.extend(self._c04_calc_def_completeness())
        issues.extend(self._c05_basetype_consistency())
        issues.extend(self._c06_part_type_resolution())
        issues.extend(self._c07_action_type_resolution())
        issues.extend(self._c10_perform_target_resolution())
        return issues

    def _c01_import_resolution(self):
        """C01: For each 'private import X::*', verify that X exists."""
        issues = []
        known_packages = set(self.packages.keys())
        # External packages always valid (Eclipse libraries)
        external_valid = {
            'ScalarValues', 'ISQ', 'SI', 'USCustomaryUnits',
            'Quantities', 'UnitsAndScales', 'MeasurementReferences',
            # CoSMA framework packages (static, not generated by the pipeline)
            'CoSMAPackage', 'CoSMAQuantitiesAndUnitsPackage', 'CoSMAViewsPackage',
            'ModelingMetadata',
            # SysML v2 / KerML standard library packages
            'NumericalFunctions', 'ScalarFunctions', 'BaseFunctions',
            'DataFunctions', 'ControlFunctions',
            'States', 'Actions', 'Events', 'Occurrences',
            'OccurrenceFunctions', 'Time', 'Metaobjects',
            'Objects', 'Performances', 'Transfers',
            'Links', 'Items', 'Parts', 'Ports',
            'Connections', 'Interfaces', 'Allocations',
            'Requirements', 'Constraints',
            'Calculations', 'Cases', 'Views', 'Viewpoints',
            'Metadata', 'ShapeItems',
        }
        for pkg_name, content in self.packages.items():
            for m in self.RE_IMPORT.finditer(content):
                imported = m.group(1)
                # Extract base name (before ::, if there is a sub-path)
                base_name = imported.split('::')[0] if '::' in imported else imported
                if base_name not in known_packages and base_name not in external_valid:
                    ln = _line_at(content, m.start())
                    issues.append(Issue(severity=P5_ERROR, layer='coherence', package=pkg_name,
                        code='C01',
                        message=f'Unresolved import: "{imported}" does not exist in the generated packages',
                        line=ln, affected_lines=[ln],
                        fix_strategy=AUTO_SURGICAL,
                        current_value=f'private import {imported}::*;',
                        suggested_fix=f'Remove import or correct package name',
                        spec_reference='SysML v2 package import resolution'))
        return issues

    def _c02_port_def_consistency(self):
        """C02: port x : PortType → verify that port def PortType exists."""
        issues = []
        # Collect all port defs from all packages
        all_port_defs = set()
        for pkg_name, content in self.packages.items():
            all_port_defs.update(self.RE_PORT_DEF.findall(content))

        # Verify usages in TechnicalComponentsPackage and SystemPackage
        target_pkgs = ['TechnicalComponentsPackage', 'SystemPackage',
                       'SystemSpecificationPackage', 'LogicalComponentsPackage']
        for pkg_name in target_pkgs:
            content = self.packages.get(pkg_name, '')
            if not content:
                continue
            for m in self.RE_PORT_USAGE.finditer(content):
                port_type = m.group(1)
                if port_type not in all_port_defs:
                    available_defs = ', '.join(sorted(all_port_defs))
                    ln = _line_at(content, m.start())
                    issues.append(Issue(severity=P5_WARNING, layer='coherence', package=pkg_name,
                        code='C02',
                        message=f'port type "{port_type}" with no matching port def in any package',
                        line=ln, affected_lines=[ln],
                        fix_strategy=AUTO_TARGETED,
                        current_value=port_type,
                        suggested_fix=f'Replace with semantically matching port def from: {available_defs}',
                        spec_reference='CoSMA metamodel: port def consistency'))
        return issues

    def _c03_satisfy_target_exists(self):
        """C03: satisfy target path — verify that the root name exists."""
        issues = []
        # Collect all part/action/state defs and instances from all packages
        all_names = set()
        for pkg_name, content in self.packages.items():
            all_names.update(self.RE_PART_DEF.findall(content))
            all_names.update(self.RE_PART_INSTANCE.findall(content))

        # Verify satisfy targets in the requirements packages
        req_pkgs = ['MissionRequirementsPackage', 'FunctionalRequirementsPackage',
                     'TechnicalRequirementsPackage', 'StakeholderNeedsPackage']
        for pkg_name in req_pkgs:
            content = self.packages.get(pkg_name, '')
            if not content:
                continue
            for m in self.RE_SATISFY.finditer(content):
                target = m.group(1)
                # Extract root name (before the first .)
                root_name = target.split('.')[0]
                if root_name not in all_names:
                    ln = _line_at(content, m.start())
                    issues.append(Issue(severity=P5_WARNING, layer='coherence', package=pkg_name,
                        code='C03',
                        message=f'satisfy target "{root_name}" not found as def/instance in any package',
                        line=ln, affected_lines=[ln],
                        fix_strategy=HUMAN_ASSISTED,
                        current_value=target,
                        suggested_fix=f'Verify satisfy path references an existing part/action/state',
                        spec_reference='CoSMA metamodel rel. #28 (satisfy path resolution)'))
        return issues

    def _c04_calc_def_completeness(self):
        """C04: calc defs referenced in AnalysisPackage must exist in CalculationsPackage."""
        issues = []
        calc_content = self.packages.get('CalculationsPackage', '')
        analysis_content = self.packages.get('AnalysisPackage', '')
        if not calc_content or not analysis_content:
            return issues

        # calc defs declared in CalculationsPackage
        declared_calcs = set(self.RE_CALC_DEF.findall(calc_content))

        # calc defs referenced in AnalysisPackage (via function call)
        builtins = {'min', 'max', 'abs', 'sqrt', 'sum', 'round', 'ceil', 'floor'}
        local_defs = set(self.RE_CALC_DEF.findall(analysis_content))
        ref_lines = {}  # ref -> [lines]
        for m in self.RE_CALC_REF.finditer(analysis_content):
            ref = m.group(1)
            if ref not in builtins and ref not in local_defs:
                ref_lines.setdefault(ref, []).append(_line_at(analysis_content, m.start()))

        for ref, alines in ref_lines.items():
            if ref not in declared_calcs:
                issues.append(Issue(severity=P5_ERROR, layer='coherence', package='AnalysisPackage',
                    code='C04',
                    message=f'Calc def "{ref}" referenced in AnalysisPackage but not declared in CalculationsPackage',
                    line=alines[0], affected_lines=alines,
                    fix_strategy=AUTO_TARGETED,
                    current_value=ref,
                    suggested_fix=f'Add calc def "{ref}" to CalculationsPackage or fix reference',
                    spec_reference='Package dependency: AnalysisPackage imports CalculationsPackage'))
        return issues

    def _c05_basetype_consistency(self):
        """C05: part def X :> Y — verify that Y is a valid CoSMA basetype."""
        issues = []
        # Collect local abstract part defs from all packages
        local_abstracts = set()
        for content in self.packages.values():
            local_abstracts.update(self.RE_ABSTRACT_PARTDEF.findall(content))

        # Collect ALL part/action/state/item defs from all packages
        all_defs = set()
        for content in self.packages.values():
            all_defs.update(self.RE_PART_DEF.findall(content))
            # item def can also be a basetype
            all_defs.update(re.findall(r'item\s+def\s+(\w+)', content))

        valid_bases = COSMA_PART_BASETYPES | local_abstracts | all_defs

        for pkg_name, content in self.packages.items():
            # Skip CoSMA framework packages (they define the basetypes)
            if pkg_name in self.COSMA_FRAMEWORK_PKGS:
                continue
            for m in self.RE_PARTDEF_BASETYPE.finditer(content):
                def_name = m.group(1)
                basetype = m.group(2)
                if basetype not in valid_bases:
                    ln = _line_at(content, m.start())
                    issues.append(Issue(severity=P5_WARNING, layer='coherence', package=pkg_name,
                        code='C05',
                        message=f'part def "{def_name}" inherits from "{basetype}" — not a recognised CoSMA basetype',
                        line=ln, affected_lines=[ln],
                        fix_strategy=HUMAN_ASSISTED,
                        current_value=basetype,
                        suggested_fix=f'Verify basetype is from CoSMA metamodel (Table 1) or declared locally',
                        spec_reference='Helle & Schramm (2026), Table 1 (CoSMA basetypes)'))
        return issues

    def _c06_part_type_resolution(self):
        """C06: part x : TypeDef → verify that TypeDef exists as part/item def in some package."""
        issues = []
        # Collect all part/item defs from all packages
        all_part_defs = set()
        for content in self.packages.values():
            all_part_defs.update(re.findall(r'(?:part|item)\s+def\s+(\w+)', content))

        # Standard library types (always valid, no local def needed)
        stdlib_types = {
            'Boolean', 'Integer', 'Real', 'String', 'Natural',
            'Complex', 'Number', 'Anything',
        }
        valid_types = all_part_defs | stdlib_types | COSMA_PART_BASETYPES

        for pkg_name, content in self.packages.items():
            if pkg_name in self.COSMA_FRAMEWORK_PKGS:
                continue
            for m in self.RE_PART_TYPED_USAGE.finditer(content):
                instance_name = m.group(1)
                type_name = m.group(2)
                # Skip SysML v2 keywords that are not types
                if type_name in ('def', 'in', 'out', 'inout', 'ref', 'abstract', 'end'):
                    continue
                if type_name not in valid_types:
                    ln = _line_at(content, m.start())
                    issues.append(Issue(severity=P5_WARNING, layer='coherence', package=pkg_name,
                        code='C06',
                        message=f'part "{instance_name}" typed as "{type_name}" — not found as part/item def',
                        line=ln, affected_lines=[ln],
                        fix_strategy=HUMAN_ASSISTED,
                        current_value=type_name,
                        suggested_fix=f'Verify "{type_name}" exists as part def in an imported package',
                        spec_reference='OMG SysML v2 Part 1, §8.2.2.11 (Part typing requires existing definition)'))
        return issues

    def _c07_action_type_resolution(self):
        """C07: action x : ActionDef → verify that ActionDef exists as action def in some package."""
        issues = []
        # Collect all action defs from all packages
        all_action_defs = set()
        for content in self.packages.values():
            all_action_defs.update(re.findall(r'action\s+def\s+(\w+)', content))

        for pkg_name, content in self.packages.items():
            if pkg_name in self.COSMA_FRAMEWORK_PKGS:
                continue
            for m in self.RE_ACTION_TYPED_USAGE.finditer(content):
                instance_name = m.group(1)
                type_name = m.group(2)
                if type_name in ('def', 'in', 'out', 'inout', 'ref', 'abstract'):
                    continue
                if type_name not in all_action_defs:
                    ln = _line_at(content, m.start())
                    issues.append(Issue(severity=P5_WARNING, layer='coherence', package=pkg_name,
                        code='C07',
                        message=f'action "{instance_name}" typed as "{type_name}" — not found as action def',
                        line=ln, affected_lines=[ln],
                        fix_strategy=HUMAN_ASSISTED,
                        current_value=type_name,
                        suggested_fix=f'Verify "{type_name}" exists as action def in an imported package',
                        spec_reference='OMG SysML v2 Part 1, §8.2.2.17.2 (Action typing requires existing definition)'))
        return issues

    def _c10_perform_target_resolution(self):
        """C10: perform Namespace::actionName → verify existence separately."""
        issues = []
        # Collect all part defs (potential namespaces) and action/part instances
        all_defs = set()
        all_instances = set()
        for content in self.packages.values():
            all_defs.update(self.RE_PART_DEF.findall(content))
            all_instances.update(self.RE_PART_INSTANCE.findall(content))

        all_names = all_defs | all_instances

        for pkg_name, content in self.packages.items():
            if pkg_name in self.COSMA_FRAMEWORK_PKGS:
                continue
            for m in self.RE_PERFORM_TARGET.finditer(content):
                namespace = m.group(1)
                action_name = m.group(2)
                if namespace not in all_names:
                    ln = _line_at(content, m.start())
                    issues.append(Issue(severity=P5_WARNING, layer='coherence', package=pkg_name,
                        code='C10',
                        message=f'perform target namespace "{namespace}" not found as def/instance',
                        line=ln, affected_lines=[ln],
                        fix_strategy=HUMAN_ASSISTED,
                        current_value=f'{namespace}::{action_name}',
                        suggested_fix=f'Verify "{namespace}" is a valid part def with "{action_name}" as member',
                        spec_reference='OMG SysML v2 Part 1, §8.2.2.17 (Perform action requires valid namespace)'))
        return issues

# ── Layer 3.5: LimitationDetector (L01–L04) ───────────────
# Detects patterns the system CANNOT resolve due to architectural
# limitations. Issues are documented in the report for traceability.

class LimitationDetector:
    """Detects patterns that require an Xtext/Eclipse parser for resolution.

    Each detector emits KNOWN_LIMITATION issues: detected, counted in
    the metrics, but NOT corrected by P6. Documented in the report
    with academic justification for the paper.
    """

    # Connection end with redefines (§8.2.2.13, Table 11)
    RE_CONNECTION_END = re.compile(
        r'connection\s*(?::\s*\w+)?\s*\{[^}]*end\s+\w+\s*:>>',
        re.DOTALL)

    # Attribute redefines (§8.2.2.7)
    RE_ATTR_REDEFINES = re.compile(
        r'attribute\s+:>>\s*(\w+)')

    # Transition first X then Y (§8.2.2.18.3)
    RE_TRANSITION = re.compile(
        r'transition\s+(?:\w+\s+)?first\s+(\w+)\s+'
        r'(?:accept\s+.+?\s+)?(?:if\s+.+?\s+)?(?:do\s+.+?\s+)?'
        r'then\s+(\w+)', re.DOTALL)

    # Shorthand transition: then X (after entry or state)
    RE_TRANSITION_SHORT = re.compile(
        r'(?:entry|state\s+\w+)[^;{]*;\s*\n\s*then\s+(\w+)')

    # #refinement dependency X to Package::'REQ' (§8.2.2.3)
    RE_REFINEMENT = re.compile(
        r"#refinement\s+dependency\s+.*?to\s+(\w+)::'([^']+)'")

    def __init__(self, packages):
        self.packages = packages

    def detect_all(self):
        """Runs all limitation detectors. Returns list[Issue]."""
        issues = []
        for pkg_name, content in self.packages.items():
            issues.extend(self._l01_connection_end_resolution(content, pkg_name))
            issues.extend(self._l02_attribute_redefines(content, pkg_name))
            issues.extend(self._l03_transition_targets(content, pkg_name))
            issues.extend(self._l04_refinement_target(content, pkg_name))
        return issues

    def _l01_connection_end_resolution(self, content, pkg_name):
        """L01: connection end with :>> — requires type conformance (Xtext)."""
        issues = []
        alines = _all_lines_for(content, self.RE_CONNECTION_END)
        if alines:
            issues.append(Issue(severity=P5_WARNING, layer='structural', package=pkg_name,
                code='L01',
                message=f'Connection end redefines (:>>) detected ({len(alines)}x) — requires Xtext validation for type conformance',
                fix_strategy=KNOWN_LIMITATION,
                line=alines[0], affected_lines=alines,
                spec_reference='OMG SysML v2 Part 1, §8.2.2.13 (Connection end redefines require type resolution)'))
        return issues

    def _l02_attribute_redefines(self, content, pkg_name):
        """L02: attribute :>> X — requires inheritance traversal (Xtext)."""
        issues = []
        alines = _all_lines_for(content, self.RE_ATTR_REDEFINES)
        if alines:
            attrs = self.RE_ATTR_REDEFINES.findall(content)
            issues.append(Issue(severity=P5_WARNING, layer='structural', package=pkg_name,
                code='L02',
                message=f'Attribute redefines (:>>) detected ({len(alines)}x) — requires inheritance traversal to validate existence in the supertype',
                fix_strategy=KNOWN_LIMITATION,
                line=alines[0], affected_lines=alines,
                current_value=', '.join(set(attrs)),
                spec_reference='OMG SysML v2 Part 1, §8.2.2.7 (Attribute redefines require supertype traversal)'))
        return issues

    def _l03_transition_targets(self, content, pkg_name):
        """L03: transition first X then Y — requires state-machine context (Xtext)."""
        issues = []
        alines = _all_lines_for(content, self.RE_TRANSITION)
        alines += _all_lines_for(content, self.RE_TRANSITION_SHORT)
        if alines:
            issues.append(Issue(severity=P5_WARNING, layer='structural', package=pkg_name,
                code='L03',
                message=f'Transition usages detected ({len(alines)}x) — source/target resolution requires state-machine context',
                fix_strategy=KNOWN_LIMITATION,
                line=alines[0], affected_lines=alines,
                spec_reference='OMG SysML v2 Part 1, §8.2.2.18.3 (Transition target resolution requires state machine context)'))
        return issues

    def _l04_refinement_target(self, content, pkg_name):
        """L04: #refinement dependency X to Pkg::'REQ' — requires cross-package symbol resolution."""
        issues = []
        for m in self.RE_REFINEMENT.finditer(content):
            target_pkg = m.group(1)
            req_id = m.group(2)
            ln = _line_at(content, m.start())
            issues.append(Issue(severity=P5_WARNING, layer='coherence', package=pkg_name,
                code='L04',
                message=f"#refinement target '{req_id}' in {target_pkg} — requires cross-package symbol resolution",
                fix_strategy=KNOWN_LIMITATION,
                line=ln, affected_lines=[ln],
                current_value=f"{target_pkg}::'{req_id}'",
                spec_reference='OMG SysML v2 Part 1, §8.2.2.3 (Dependency target resolution requires qualified name resolution)'))
        return issues


# ── Counting functions for metrics  ──────────────────
# Deterministic counting of SysML v2 elements and references.

def count_elements(content):
    """Counts SysML v2 elements declared in the package.

    Elements = definitions (def), typed instances, attributes,
    relationships (satisfy, perform, refinement, connection),
    constraints, exhibit states.
    Aligned with OMG SysML v2 Part 1 concrete textual notation.
    """
    patterns = [
        # Definitions (all def kinds from the spec)
        r'(?:part|action|state|item|requirement|port|interface|calc|analysis|'
        r'enum|concern|view|viewpoint|connection|flow|use\s+case|verification|'
        r'allocation)\s+def\s+',
        # Attributes
        r'attribute\s+\w+\s*[:,;:>]',
        # Typed instances (usages)
        r'(?:part|action|port|state|item|requirement)\s+\w+\s*:',
        # Relationships
        r'satisfy\s+',
        r'perform\s+',
        r'exhibit\s+state\s+',
        r'assert\s+constraint',
        r'connection\s*:',
        r'#refinement\s+dependency',
        # Individual
        r'individual\s+part\s+',
    ]
    total = 0
    for p in patterns:
        total += len(re.findall(p, content))
    return max(total, 1)


def count_cross_references(content):
    """Counts cross-package references in the content.

    References = imports, satisfy links, perform actions with namespace,
    type references with namespace, cross-package refinement links.

    Returns max(total, 1) to avoid division by zero.
    """
    patterns = [
        r'private\s+import\s+\w+::',
        r'satisfy\s+.*\s+by\s+',
        r'perform\s+\w+::\w+',
        r':\s+\w+Package::\w+',
        r'#refinement\s+dependency.*to\s+\w+::',
    ]
    total = 0
    for p in patterns:
        total += len(re.findall(p, content))
    return max(total, 1)

# ── ValidatorAgent ────────────────────────────────────

class ValidatorAgent:
    def __init__(self, refined_packages, mission_data, client, jar_path, refined_dir):
        self.refined_packages = refined_packages
        self.refined_dir = Path(str(refined_dir))
        self.syntax_validator = SyntaxValidator(jar_path, packages=refined_packages)
        self.structural_validator = StructuralValidator()
        self.coherence_validator = CoherenceValidator(refined_packages)
        self.limitation_detector = LimitationDetector(refined_packages)
        self.all_issues = []
        self.report = {}

    def run(self, iteration=0):
        start_time = time.time()
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        self.all_issues = []
        pkg_names = sorted(self.refined_packages.keys())

        # Accumulators for per-package metrics
        per_package = {}
        total_syntax_errors = 0
        total_structural_errors = 0
        total_elements = 0

        print(f'\n{"="*70}')
        print(f'  P5 — ValidatorAgent v5 (3 layers, deterministic)')
        print(f'{"="*70}')

        # ── Layer 1 (MontiCore) ──
        print(f'\n--- Layer 1 (MontiCore) ---')
        for idx, pkg_name in enumerate(pkg_names, 1):
            # Locate actual file (may be in a subfolder via PACKAGE_TO_FOLDER)
            filepath = self.refined_dir / f'{pkg_name}.sysml'
            if not filepath.exists():
                candidates = list(self.refined_dir.rglob(f'{pkg_name}.sysml'))
                if candidates:
                    filepath = candidates[0]
            print(f'  [{idx:2d}/{len(pkg_names)}] {pkg_name:42s} ⏳ ...', end='', flush=True)
            l1 = self.syntax_validator.validate_file(filepath, pkg_name)
            self.all_issues.extend(l1)
            e = sum(1 for i in l1 if i.severity == P5_ERROR)
            w = sum(1 for i in l1 if i.severity == P5_WARNING)
            if e > 0:
                icon = '❌'
            elif w > 0:
                icon = '⚠️'
            else:
                icon = '✅'
            print(f'\r  [{idx:2d}/{len(pkg_names)}] {pkg_name:42s} {icon}  E={e} W={w}')

            # Initialise per_package
            per_package[pkg_name] = {'syntax_errors': e, 'syntax_warnings': w,
                                     'structural_errors': 0, 'structural_warnings': 0}
            total_syntax_errors += e

        if self.syntax_validator.stdlib_suppressed_count > 0:
            print(f'  ℹ️ {self.syntax_validator.stdlib_suppressed_count} standard-library warnings suppressed (MC02 — single-file limitation)')

        # ── Layer 2 (Structural — S01–S14) ──
        print(f'\n--- Layer 2 (Structural — 16 validators) ---')
        for idx, pkg_name in enumerate(pkg_names, 1):
            content = self.refined_packages[pkg_name]
            l2 = self.structural_validator.validate(content, pkg_name)
            self.all_issues.extend(l2)
            e = sum(1 for i in l2 if i.severity == P5_ERROR)
            w = sum(1 for i in l2 if i.severity == P5_WARNING)
            per_package[pkg_name]['structural_errors'] = e
            per_package[pkg_name]['structural_warnings'] = w
            total_structural_errors += e

            # Count elements and SLOC for metrics
            pkg_elements = count_elements(content)
            pkg_sloc = _count_sloc(content)
            per_package[pkg_name]['elements'] = pkg_elements
            per_package[pkg_name]['loc'] = pkg_sloc['loc']
            per_package[pkg_name]['sloc'] = pkg_sloc['sloc']
            per_package[pkg_name]['blank'] = pkg_sloc['blank']
            per_package[pkg_name]['comment'] = pkg_sloc['comment']
            total_elements += pkg_elements

            if e > 0:
                icon = '❌'
            elif w > 0:
                icon = '⚠️'
            else:
                icon = '✅'
            print(f'  [{idx:2d}/{len(pkg_names)}] {pkg_name:42s} {icon}  E={e} W={w}')
            # Detail issues (only when there are errors)
            if e > 0:
                for issue in l2:
                    if issue.severity == P5_ERROR:
                        print(f'    {issue.code}: {issue.message[:75]}')
        s2_clean = sum(1 for p in per_package.values() if p['structural_errors'] == 0)
        print(f'  {s2_clean}/{len(pkg_names)} packages without structural errors')

        # ── Layer 3 (Cross-package coherence — C01–C05) ──
        print(f'\n--- Layer 3 (Cross-package coherence — C01–C10) ---')
        coherence_issues = self.coherence_validator.run_all()
        self.all_issues.extend(coherence_issues)
        total_coherence_errors = sum(1 for i in coherence_issues if i.severity == P5_ERROR)
        total_coherence_warnings = sum(1 for i in coherence_issues if i.severity == P5_WARNING)

        if coherence_issues:
            for issue in coherence_issues:
                icon = '❌' if issue.severity == P5_ERROR else '⚠️'
                print(f'  {icon} [{issue.code}] {issue.package}: {issue.message[:75]}')
        else:
            print(f'  ✅ No cross-package coherence issue')

        # ── Layer 3.5 (Limitation Detection — L01–L04) ──
        print(f'\n--- Layer 3.5 (Limitation Detection — L01–L04) ---')
        limitation_issues = self.limitation_detector.detect_all()
        self.all_issues.extend(limitation_issues)
        total_limitation_warnings = len(limitation_issues)

        if limitation_issues:
            by_code = Counter(i.code for i in limitation_issues)
            for code, count in sorted(by_code.items()):
                print(f'  ℹ️ [{code}] {count} patterns detected — KNOWN_LIMITATION (requires Xtext/Eclipse)')
        else:
            print(f'  ✅ No limitation pattern detected')
        print(f'  ℹ️ Total: {total_limitation_warnings} warnings counted in the metrics')

# ── SLOC metrics (D156) ──
        total_cross_refs = sum(count_cross_references(c) for c in self.refined_packages.values())

        # Aggregated SLOC
        total_loc = sum(p.get('loc', 0) for p in per_package.values())
        total_sloc = sum(p.get('sloc', 0) for p in per_package.values())
        total_blank = sum(p.get('blank', 0) for p in per_package.values())
        total_comment = sum(p.get('comment', 0) for p in per_package.values())

        # SLOC lines with issue (deduplicated by package+line)
        sloc_issue_set = set()
        total_occurrences = 0
        for issue in self.all_issues:
            if issue.affected_lines:
                for ln in issue.affected_lines:
                    sloc_issue_set.add((issue.package, ln))
                total_occurrences += len(issue.affected_lines)
            elif issue.line is not None:
                sloc_issue_set.add((issue.package, issue.line))
                total_occurrences += 1
        sloc_with_issues = len(sloc_issue_set)
        sloc_clean = total_sloc - sloc_with_issues
        sloc_rate = round(sloc_clean / max(total_sloc, 1), 4)

        # SLOC with issue per package
        sloc_issues_per_pkg = {}
        for pkg, ln in sloc_issue_set:
            sloc_issues_per_pkg[pkg] = sloc_issues_per_pkg.get(pkg, 0) + 1
        for pkg_name in pkg_names:
            pkg_issues_count = sloc_issues_per_pkg.get(pkg_name, 0)
            pkg_sloc_val = per_package[pkg_name].get('sloc', 1)
            per_package[pkg_name]['sloc_with_issues'] = pkg_issues_count
            per_package[pkg_name]['sloc_rate'] = round((pkg_sloc_val - pkg_issues_count) / max(pkg_sloc_val, 1), 4)

        # Clean packages (zero issues)
        pkgs_clean = sum(1 for p in per_package.values()
                         if p.get('sloc_with_issues', 0) == 0)

        # Legacy metrics (kept for should_fix_or_deliver compatibility)
        pkgs_syntax_clean = sum(1 for p in per_package.values() if p['syntax_errors'] == 0)
        scr_value = round(pkgs_syntax_clean / max(len(pkg_names), 1), 4)
        gc_value = round(1.0 - (total_structural_errors / max(total_elements, 1)), 4)
        sc_value = round(1.0 - (total_coherence_errors / max(total_cross_refs, 1)), 4)

        # CoSMA layer (derived from PACKAGE_TO_FOLDER)
        folder_map = globals().get('PACKAGE_TO_FOLDER', {})
        layer_stats = {}
        for pkg_name in pkg_names:
            layer = folder_map.get(pkg_name, 'Other')
            if layer not in layer_stats:
                layer_stats[layer] = {'pkgs': 0, 'sloc': 0, 'issues': 0, 'clean': 0}
            ls = layer_stats[layer]
            ls['pkgs'] += 1
            ls['sloc'] += per_package[pkg_name].get('sloc', 0)
            ls['issues'] += per_package[pkg_name].get('sloc_with_issues', 0)
            if per_package[pkg_name].get('sloc_with_issues', 0) == 0:
                ls['clean'] += 1

        # Classify issues
        auto_fixable = [i for i in self.all_issues
                        if i.fix_strategy in (AUTO_SURGICAL, AUTO_TARGETED, AUTO_LLM)]
        human_review = [i for i in self.all_issues if i.fix_strategy == HUMAN_ASSISTED]
        known_limitations = [i for i in self.all_issues if i.fix_strategy == KNOWN_LIMITATION]

        elapsed = time.time() - start_time

        # ── Formatted SLOC print  ──
        print(f'\n{"="*70}')
        print(f'  PIPELINE METRICS — SLOC analysis (iteration {iteration})')
        print(f'{"="*70}')

        print(f'  ┌──────────────────────────────────────────────────────────┐')
        print(f'  │ SLOC generated:   {total_sloc:>5d}  (of {total_loc} LOC — {total_blank} blank, {total_comment} //)   │')
        print(f'  │ SLOC with issue:  {sloc_with_issues:>5d}  ({sloc_with_issues} unique lines with P5 issue)    │')
        print(f'  │ Occurrences:      {total_occurrences:>5d}  (total errors/warnings detected)        │')
        print(f'  │ SLOC rate (P5):   {sloc_rate:6.1%}  ({sloc_clean}/{total_sloc} SLOC without issues)    │')
        print(f'  │ Clean packages:   {pkgs_clean:>3d}/{len(pkg_names):<3d}  ({pkgs_clean/max(len(pkg_names),1):.1%})                            │')
        print(f'  └──────────────────────────────────────────────────────────┘')
        print()
        print(f'  Issues (for P6):   {len(self.all_issues)}')
        print(f'    Auto-fixable:    {len(auto_fixable)} ({len(auto_fixable)/max(len(self.all_issues),1):.1%}) → forwarded to P6')
        print(f'    Human-review:    {len(human_review)} ({len(human_review)/max(len(self.all_issues),1):.1%}) → documented for Eclipse')
        print(f'    Known-limit.:    {len(known_limitations)} ({len(known_limitations)/max(len(self.all_issues),1):.1%}) → documented (requires Xtext/Eclipse)')
        print(f'  Cost:              $0.00 (deterministic)')
        print(f'  Time:              {elapsed:.1f}s')

        # SLOC per CoSMA layer
        if layer_stats:
            print(f'\n  SLOC PER CoSMA LAYER:')
            print(f'  {"Layer":<18s} {"Pkgs":>5s} {"SLOC":>6s} {"Issues":>7s} {"Clean":>7s} {"Rate":>7s}')
            print(f'  {"-"*52}')
            layer_order = ['Purpose', 'Operation', 'Function', 'Logical', 'Technical',
                           'Requirements', 'Analysis', 'Execution', 'Program', 'CoSMA', 'root', 'Other']
            for layer in layer_order:
                if layer not in layer_stats:
                    continue
                ls = layer_stats[layer]
                rate = (ls["sloc"] - ls["issues"]) / max(ls["sloc"], 1)
                print(f'  {layer:<18s} {ls["pkgs"]:>5d} {ls["sloc"]:>6d} {ls["issues"]:>7d} '
                      f'{ls["clean"]:>3d}/{ls["pkgs"]:<3d} {rate:>6.1%}')
            print(f'  {"-"*52}')

        if known_limitations:
            print(f'\n  KNOWN LIMITATIONS (validation deferred → Eclipse):')
            by_code = Counter(i.code for i in known_limitations)
            for code, count in sorted(by_code.items()):
                labels = {
                    'L01': 'Connection end :>> resolution',
                    'L02': 'Attribute :>> redefines resolution',
                    'L03': 'Transition target resolution',
                    'L04': '#refinement target resolution',
                    'MC07': 'MontiCore parser limitation (valid SysML v2, unsupported by parser)',
                }
                print(f'    [{code}] {labels.get(code, code)}: {count}')
            print(f'    Total: {len(known_limitations)} patterns deferred for human validation (Eclipse IDE)')

        print(f'{"="*70}')

        # ── Compile report  ──
        self.report = {
            'timestamp': timestamp,
            'iteration': iteration,
            'model': 'deterministic (zero-LLM)',
            'monticore_available': self.syntax_validator.available,

            # SLOC metrics
            'metrics': {
                'sloc': {
                    'total_loc': total_loc,
                    'total_sloc': total_sloc,
                    'total_blank': total_blank,
                    'total_comment': total_comment,
                    'sloc_with_issues': sloc_with_issues,
                    'sloc_clean': sloc_clean,
                    'sloc_rate': sloc_rate,
                    'total_occurrences': total_occurrences,
                    'packages_total': len(pkg_names),
                    'packages_clean': pkgs_clean,
                    'description': 'SLOC = lines of effective SysML v2 notation (excludes blanks and standalone //). Issues = unique lines with at least one P5 error/warning.',
                },
                # Legacy (compatibility)
                'scr': {
                    'value': scr_value if self.syntax_validator.available else None,
                    'packages_clean': pkgs_syntax_clean,
                    'total_packages': len(pkg_names),
                },
                'gc': {
                    'value': gc_value,
                    'structural_errors': total_structural_errors,
                    'total_elements': total_elements,
                },
                'sc': {
                    'value': sc_value,
                    'coherence_errors': total_coherence_errors,
                    'total_references': total_cross_refs,
                },
            },

            # CoSMA layer
            'layer_stats': {layer: stats for layer, stats in layer_stats.items()},

            # Issues for P6
            'issues': [asdict(i) for i in self.all_issues],
            'auto_fixable': [asdict(i) for i in auto_fixable],
            'human_review': [asdict(i) for i in human_review],
            'auto_fixable_count': len(auto_fixable),
            'human_review_count': len(human_review),
            'known_limitations': [asdict(i) for i in known_limitations],
            'known_limitations_count': len(known_limitations),
            'mc02_stdlib_suppressed': self.syntax_validator.stdlib_suppressed_count,
            'known_limitations_summary': {
                'description': 'Patterns detected but not resolvable by automated pipeline — require Xtext/Eclipse validation',
                'total': len(known_limitations),
                'by_code': dict(Counter(i.code for i in known_limitations)),
                'academic_justification': 'Cibrián et al. (2025b): semantic validation of cross-package references requires metamodel-driven analysis beyond regex/ANTLR capabilities',
            },

            # Per-package breakdown
            'per_package': per_package,

            # Cost
            'cost_usd': 0.0,
            'elapsed_s': round(elapsed, 1),
        }

        return self.report

    def save_report(self, output_dir):
        ts = self.report.get('timestamp', datetime.now().strftime('%Y%m%d_%H%M%S'))
        path = Path(str(output_dir)) / f'validation_report_{ts}.json'
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(self.report, f, indent=2, ensure_ascii=False)
        print(f'Report saved: {path}')
        return path


# ── run_p5: wrapper State → State ──────────────────────────

def run_p5(state: dict) -> dict:
    """ValidatorAgent (P5): packages → validation_report.

    Input: refined_packages, mission_json, iteration
    Output: validation_report (continuous metrics, no PASS/FAIL)
    """
    print(f'\n{"="*60}')
    print(f'  P5 — ValidatorAgent (3 layers, deterministic)')
    print(f'{"="*60}')
    t0 = time.time()

    # Materialise packages on disk for MontiCore
    iteration = state.get('iteration', 0)
    refined_dir = VALIDATOR_DIR / f'iter_{iteration}_{state["run_timestamp"]}'
    write_packages_to_dir(state['refined_packages'], refined_dir, f'P5 iter {iteration}')

    # MontiCore JAR — with polling if needed
    jar_expected = BASE_DIR / 'tools' / 'MCSysMLv2.jar'
    skip_mc = state.get('skip_monticore', False)
    if skip_mc:
        jar_path = None
        print(f'  ℹ️ MontiCore skipped in the final iteration (Layers 2+3 only)')
    else:
        jar_path = _ensure_monticore(jar_expected)

    if jar_path:
        jar_size = jar_path.stat().st_size
        print(f'  MontiCore JAR: {jar_path} ({jar_size:,} bytes)')
    else:
        print(f'  ⚠️ Pipeline will continue without MontiCore (SCR = N/A)')

    validator = ValidatorAgent(
        refined_packages=state['refined_packages'],
        mission_data=state['mission_json'],
        client=CLIENT,
        jar_path=jar_path,
        refined_dir=refined_dir,
    )

    report = validator.run(iteration=iteration)
    report_path = validator.save_report(VALIDATOR_DIR)

    elapsed = time.time() - t0
    log_cost(state, f'P5_ValidatorAgent_iter{iteration}', 0.0, elapsed, 0, 0)

    print(f'  P5 finished in {elapsed:.1f}s')

    return {
        'validation_report': report,
    }


# ── Verification ───────────────────────────────────────────
print(f'P5 defined: run_p5(state) → validation_report (SLOC metrics)')
print(f'  3 layers: Syntactic (MontiCore), Structural (S01–S16), Coherence (C01–C10), Limitations (L01–L04)')

## Cell 9 — P6 FixerAgent and run_p6()

In [14]:
# ============================================================
# Master Orchestrator
# Cell 9 — P6 (FixerAgent) — 3-Tier Cascade
# ============================================================

import copy
import textwrap
from collections import Counter

# ── P6 constants ───────────────────────────────────────────

P6_MAX_TOKENS = 16000

PACKAGE_TO_JSON_SECTIONS = {
    'StakeholderPackage':            ['stakeholders'],
    'StakeholderNeedsPackage':       ['stakeholder_needs'],
    'CapabilitiesPackage':           ['capabilities', 'stakeholder_needs'],
    'MissionPackage':                ['mission_goals', 'mission_context'],
    'MissionSpecificationPackage':   ['mission_goals', 'stakeholder_needs'],
    'MissionRequirementsPackage':    ['mission_requirements', 'capabilities'],
    'FunctionSpecificationPackage':  ['functional_requirements', 'mission_requirements'],
    'FunctionalRequirementsPackage': ['functional_requirements', 'mission_requirements'],
    'TechnicalRequirementsPackage':  ['technical_requirements', 'functional_requirements'],
    'MissionPhasesPackage':          ['mission_phases'],
    'OperationsPackage':             ['operations', 'functions'],
    'FunctionsPackage':              ['functions', 'operations'],
    'LogicalComponentsPackage':      ['logical_components'],
    'TechnicalComponentsPackage':    ['technical_components'],
    'TechnicalPortsPackage':         ['ports', 'technical_components'],
    'SystemPackage':                 ['logical_components', 'technical_components', 'ports'],
    'SystemSpecificationPackage':    ['logical_components', 'technical_components'],
    'AnalysisPackage':               ['analysis_cases', 'technical_components'],
    'ContextPackage':                ['mission_context', 'external_systems'],
    'ProgramPackage':                ['mission_goals'],
}


# ── IssueParser  ──────────────────────────────────────────

class IssueParser:
    def __init__(self, report):
        self.report = report
        self.auto_fixable = report.get('auto_fixable', [])
        self.human_review = report.get('human_review', [])
        self.all_issues = report.get('issues', [])

    def get_by_strategy(self, strategy):
        return [i for i in self.auto_fixable if i.get('fix_strategy') == strategy]

    def summary(self):
        surgical = self.get_by_strategy(AUTO_SURGICAL)
        targeted = self.get_by_strategy(AUTO_TARGETED)
        llm = self.get_by_strategy(AUTO_LLM)
        print(f'  Issue Parser: {len(surgical)} surgical, {len(targeted)} targeted, '
              f'{len(llm)} LLM, {len(self.human_review)} human')


# ── JSONContextExtractor ───────────────────────────────

class JSONContextExtractor:
    def __init__(self, mission_data):
        self.mission_data = mission_data

    def extract_for_package(self, package_name, max_chars=30000):
        sections = PACKAGE_TO_JSON_SECTIONS.get(package_name, [])
        if not sections: return '{}'
        relevant = {}
        for key in sections:
            if not key: continue
            if key in self.mission_data:
                relevant[key] = self.mission_data[key]
            else:
                for top_key, top_val in self.mission_data.items():
                    if isinstance(top_val, dict) and key in top_val:
                        relevant[key] = top_val[key]
        result = json.dumps(relevant, indent=2, ensure_ascii=False)
        return result[:max_chars] if len(result) > max_chars else result


# ── SurgicalFixHandler  ──────────────────────────────

class SurgicalFixHandler:

    def __init__(self):
        self.fixes_applied = []
        self.escalated = []

    def fix(self, issue, sysml_content):
        """Dispatch por issue code."""
        code = issue.get('code', '')

        # Handlers
        if code == 'S06': return self._fix_missing_import(issue, sysml_content)
        if code == 'S04': return self._fix_duplicate_definition(issue, sysml_content)
        if code == 'S01': return self._fix_markdown_fences(issue, sysml_content)
        if code == 'S15': return self._fix_duplicate_import(issue, sysml_content)
        if code == 'S09' and 'requires value conversion' in issue.get('message', ''):
            return self._fix_unit_conversion(issue, sysml_content)
        if code == 'S10':
            return self._fix_perform_casing(issue, sysml_content)
        if code == 'S11':
            return self._fix_missing_and(issue, sysml_content)
        if code == 'S15': return self._fix_duplicate_import(issue, sysml_content)
        if code == 'S16': return self._fix_transition_syntax(issue, sysml_content)
        if issue.get('current_value') and issue.get('expected_value'):
            return self._fix_by_replacement(issue, sysml_content)
        if issue.get('regex_pattern') and ' → ' in issue.get('regex_pattern', ''):
            return self._fix_by_regex(issue, sysml_content)

        # Fallback: escalar
        self.escalated.append(issue)
        return sysml_content, False

    # ── S01: Markdown fences ──

    def _fix_markdown_fences(self, issue, content):
        """Remove markdown fences residuais."""
        new_content = re.sub(r'```\w*\n?', '', content)
        if new_content != content:
            self.fixes_applied.append({'issue_code': 'S01', 'package': issue['package'],
                'action': 'Markdown fences removed', 'success': True})
            return new_content, True
        self.escalated.append(issue)
        return content, False

    # ── S04: Duplicate definitions  ──

    def _fix_duplicate_definition(self, issue, content):
        msg = issue.get('message', '')
        match = re.search(r'\"(.+?)\"', msg)
        if not match: self.escalated.append(issue); return content, False
        dup_name = match.group(1)
        lines = content.split('\n')
        def_pattern = re.compile(
            rf'\b(?:state|part|requirement|action|item|port|attribute|connection|concern)'
            rf'\s+def\s+{re.escape(dup_name)}\b')
        occurrences = [idx for idx, line in enumerate(lines) if def_pattern.search(line)]
        if len(occurrences) < 2: self.escalated.append(issue); return content, False
        last_idx = occurrences[-1]
        block_end = self._find_block_end(lines, last_idx)
        del lines[last_idx:block_end + 1]
        content = re.sub(r'\n{3,}', '\n\n', '\n'.join(lines))
        self.fixes_applied.append({'issue_code': 'S04', 'package': issue['package'],
            'action': f'Removed duplicate "{dup_name}" ({block_end - last_idx + 1} lines)', 'success': True})
        return content, True

    def _find_block_end(self, lines, start_idx):
        depth = 0
        for idx in range(start_idx, len(lines)):
            depth += lines[idx].count('{') - lines[idx].count('}')
            if depth <= 0 and idx > start_idx: return idx
        return start_idx

    # ── S06: Missing import  ──

    def _fix_missing_import(self, issue, content):
        msg = issue.get('message', '')
        match = (re.search(r'Import de (\w+) ausente', msg)
                 or re.search(r'Import não resolvido: "(\w+)"', msg)
                 or re.search(r'Missing import[:\s]+(\w+)', msg, re.IGNORECASE))
        if not match: self.escalated.append(issue); return content, False
        import_name = match.group(1)
        import_line = f'\tprivate import {import_name}::*;'
        if f'import {import_name}::' in content:
            self.fixes_applied.append({'issue_code': issue['code'], 'package': issue['package'],
                'action': f'Import {import_name} already present — no-op', 'success': True})
            return content, True
        lines = content.split('\n')
        last_import_idx = -1
        for idx, line in enumerate(lines):
            if re.match(r'\s*(private\s+)?import\s+', line): last_import_idx = idx
        if last_import_idx == -1:
            for idx, line in enumerate(lines):
                if re.match(r'\s*package\s+', line): last_import_idx = idx; break
        if last_import_idx >= 0:
            lines.insert(last_import_idx + 1, import_line)
            self.fixes_applied.append({'issue_code': issue['code'], 'package': issue['package'],
                'action': f'Inserted: {import_line.strip()}', 'success': True})
            return '\n'.join(lines), True
        self.escalated.append(issue); return content, False

    # ── Generic ──

    def _fix_by_replacement(self, issue, content):
        """Simple replacement: current_value → expected_value."""
        current = issue['current_value']
        expected = issue['expected_value']
        if current in content:
            new_content = content.replace(current, expected)
            count = content.count(current)
            self.fixes_applied.append({'issue_code': issue['code'], 'package': issue['package'],
                'action': f'Replace: "{current}" → "{expected}" ({count}x)', 'success': True})
            return new_content, True
        self.escalated.append(issue)
        return content, False

    def _fix_by_regex(self, issue, content):
        """Applies regex correction using the regex_pattern field (format: 'search → replace')."""
        pattern_str = issue['regex_pattern']
        if ' → ' not in pattern_str:
            self.escalated.append(issue)
            return content, False
        search_pat, replace_pat = pattern_str.split(' → ', 1)
        try:
            matches_before = len(re.findall(search_pat, content))
            if matches_before == 0:
                self.escalated.append(issue)
                return content, False
            new_content = re.sub(search_pat, replace_pat, content)
            self.fixes_applied.append({'issue_code': issue['code'], 'package': issue['package'],
                'action': f'Regex: {matches_before} occurrences corrected', 'success': True})
            return new_content, True
        except re.error as e:
            self.escalated.append(issue)
            return content, False

    # ── Specific ──

    def _fix_unit_conversion(self, issue, content):
        """S09: Converts invalid units that require value transformation.

        Mission-agnostic: uses UNIT_CONVERSION_FACTORS for any
        unit Eclipse does not resolve (MHz, GHz, ft, in, lb, mi).
        """
        current = issue.get('current_value', '')
        unit_match = re.search(r'\[(\w+)\]', current)
        if not unit_match:
            self.escalated.append(issue)
            return content, False

        invalid_unit = unit_match.group(1)
        if invalid_unit not in UNIT_CONVERSION_FACTORS:
            self.escalated.append(issue)
            return content, False

        target_unit, factor = UNIT_CONVERSION_FACTORS[invalid_unit]

        def _convert(m):
            val = float(m.group(1))
            converted = val * factor
            if factor >= 1000:
                exp = int(math.log10(factor))
                return f'{val}E{exp} [{target_unit}]'
            if converted == int(converted):
                return f'{int(converted)} [{target_unit}]'
            return f'{converted:.4g} [{target_unit}]'

        pattern = rf'(\d+\.?\d*)\s*\[{re.escape(invalid_unit)}\]'
        count = len(re.findall(pattern, content))
        if count == 0:
            self.escalated.append(issue)
            return content, False

        new_content = re.sub(pattern, _convert, content)
        self.fixes_applied.append({'issue_code': 'S09', 'package': issue['package'],
            'action': f'Unit conversion: {count}x [{invalid_unit}] → [{target_unit}] (×{factor})',
            'success': True})
        return new_content, True

    def _fix_perform_casing(self, issue, content):
        """S10: Converts perform Namespace::PascalCase → camelCase."""
        def _to_camel(m):
            prefix = m.group(1)
            first_char = m.group(2)
            rest = m.group(3)
            return f'{prefix}{first_char.lower()}{rest}'

        pattern = r'(perform\s+\w+::)([A-Z])(\w+)'
        count = len(re.findall(pattern, content))
        new_content = re.sub(pattern, _to_camel, content)
        if new_content != content:
            self.fixes_applied.append({'issue_code': 'S10', 'package': issue['package'],
                'action': f'Perform: {count} names converted to camelCase', 'success': True})
            return new_content, True

        # Escalation fix: if the current_value (PascalCase) no longer exists,
        # the fix was already applied by a previous call on the same package
        current = issue.get('current_value', '')
        if current and current not in content:
            expected = issue.get('expected_value', '')
            if expected and expected in content:
                self.fixes_applied.append({'issue_code': 'S10', 'package': issue['package'],
                    'action': f'Perform: "{current}" already fixed to "{expected}" (previous fix)', 'success': True})
                return content, True

        self.escalated.append(issue)
        return content, False

    def _fix_missing_and(self, issue, content):
        """S11: Inserts 'and' between consecutive comparisons in assert constraint."""
        pattern = re.compile(
            r'(\S+\s*(?:>=|<=|>|<|==)\s*\S+)\s*\n(\s+)(\S+\s*(?:>=|<=|>|<|==))')
        original = content
        while pattern.search(content):
            content = pattern.sub(r'\1 and\n\2\3', content)
        if content != original:
            self.fixes_applied.append({'issue_code': 'S11', 'package': issue['package'],
                'action': 'Constraint: "and" operators inserted', 'success': True})
            return content, True
        self.escalated.append(issue)
        return content, False

    # ── S15: Duplicate imports ──

    def _fix_duplicate_import(self, issue, content):
        """S15: Removes duplicate imports, keeping the first occurrence."""
        import_value = issue.get('current_value', '')
        if not import_value:
            self.escalated.append(issue)
            return content, False
        lines = content.split('\n')
        seen = False
        new_lines = []
        removed = 0
        for line in lines:
            if import_value in line.strip():
                if not seen:
                    seen = True
                    new_lines.append(line)
                else:
                    removed += 1
                    continue
            else:
                new_lines.append(line)
        if removed > 0:
            self.fixes_applied.append({'issue_code': 'S15', 'package': issue['package'],
                'action': f'Removed {removed} duplicate import(s): {import_value}', 'success': True})
            return '\n'.join(new_lines), True
        self.escalated.append(issue)
        return content, False

    # ── S16: Transition syntax ──

    def _fix_transition_syntax(self, issue, content):
        """S16: Converts transition A -> B { trigger accept X; } into first A accept X then B."""
        original = content

        # Pattern 1: transition A -> B { trigger accept Notification; doc /* ... */ }
        def _fix_with_trigger(m):
            source = m.group(1)
            target = m.group(2)
            notification = m.group(3)
            body = m.group(4).strip()
            if body:
                return f'transition first {source} accept {notification} then {target} {{\n{body}\n}}'
            return f'transition first {source} accept {notification} then {target};'

        content = re.sub(
            r'transition\s+(\w+)\s*->\s*(\w+)\s*\{\s*trigger\s+accept\s+(\w+)\s*;\s*(.*?)\}',
            _fix_with_trigger, content, flags=re.DOTALL)

        # Pattern 2: transition A -> B; (no trigger)
        content = re.sub(
            r'transition\s+(\w+)\s*->\s*(\w+)\s*;',
            r'transition first \1 then \2;', content)

        if content != original:
            count = len(re.findall(r'transition\s+\w+\s*->', original))
            self.fixes_applied.append({'issue_code': 'S16', 'package': issue['package'],
                'action': f'Transition syntax: {count}x "->" converted to "first/then"', 'success': True})
            return content, True

        self.escalated.append(issue)
        return content, False

    # ── process_all ──

    def process_all(self, issues, packages):
        """Applies surgical fixes to all affected packages."""
        fixed = dict(packages)
        by_pkg = defaultdict(list)
        for i in issues: by_pkg[i['package']].append(i)

        print(f'\n--- Tier 1: SurgicalFixHandler ({len(issues)} issues) ---')
        for pkg_name, pkg_issues in sorted(by_pkg.items()):
            content = fixed.get(pkg_name, '')
            if not content:
                for issue in pkg_issues:
                    self.escalated.append(issue)
                continue
            for issue in pkg_issues:
                content, success = self.fix(issue, content)
                icon = '✅' if success else '⚠️'
                action = self.fixes_applied[-1]['action'] if success and self.fixes_applied else '(escalated)'
                print(f'  {icon} [{issue["code"]}] {pkg_name}: {action}')
            fixed[pkg_name] = content

        print(f'  Result: {len(self.fixes_applied)} applied, {len(self.escalated)} escalated → Tier 2')
        return fixed

# ── TargetedFixHandler  ──────────────────────
# Tier 2: surgical LLM by snippet.
# Extracts a snippet (±7 lines) around the error, retrieves an Apollo 11
# reference via RAG, asks the LLM for a focused fix.
# Issues escalated from Tier 1 + AUTO_TARGETED issues from P5.

TARGETED_FIX_SYSTEM_PROMPT = (
    'You are a SysML v2 syntax fixer within the CoSMA framework. '
    'Fix ONLY the specific issue described. Do NOT modify any other content. '
    'Preserve exact indentation and line structure.\n\n'
    'Critical rules:\n'
    '- ISQ features use SUBSETTING (:>), e.g.: attribute speed :> ISQ::speed\n'
    '- ScalarValues types use TYPING (:), e.g.: attribute isActive : ScalarValues::Boolean\n'
    '- Instance names are camelCase: part missionSystem, action performLaunch\n'
    '- Type/def names are PascalCase: part def MissionSystem, action def PerformLaunch\n'
    '- Valid SI/CoSMA units (e.g.): kg, m, s, Hz, N, Pa, J, W, V, kn, nmi, km, h, min, rad, [\'°\']\n'
    '- Invalid units requiring conversion: MHz→Hz, GHz→Hz, deg→\'°\', knots→kn, ft→m\n'
    '- satisfy paths must use dot notation: mission.phases.phase.doAction.operation\n'
    '- #refinement dependency uses: to PackageName::\'REQ-ID\''
)

class TargetedFixHandler:
    CONTEXT_LINES = 7
    MAX_TOKENS = 2000

    def __init__(self, client, model=None):
        self.client = client
        self.model = model or LLM_MODEL
        self.fixes_applied = []
        self.escalated = []
        self.total_tokens_in = 0
        self.total_tokens_out = 0

    # ── Snippet extraction ──

    def _extract_snippet_by_line(self, content, line_num):
        """Extracts a snippet around a specific line."""
        lines = content.split('\n')
        if not line_num or line_num < 1 or line_num > len(lines):
            return None, None, None
        start = max(0, line_num - 1 - self.CONTEXT_LINES)
        end = min(len(lines), line_num + self.CONTEXT_LINES)
        snippet = '\n'.join(lines[start:end])
        return snippet, start, end

    def _extract_snippet_by_pattern(self, content, search_str):
        """Extracts a snippet around a search pattern."""
        lines = content.split('\n')
        target_line = None
        for i, line in enumerate(lines):
            if search_str in line:
                target_line = i
                break
        if target_line is None:
            return None, None, None
        start = max(0, target_line - self.CONTEXT_LINES)
        end = min(len(lines), target_line + 1 + self.CONTEXT_LINES)
        snippet = '\n'.join(lines[start:end])
        return snippet, start, end

    # ── Surgical prompt ──

    def _build_targeted_prompt(self, issue, snippet, rag_context):
        parts = [
            f'## FIX ONE SPECIFIC ISSUE in this SysML v2 code snippet\n',
            f'### Issue',
            f'- Code: {issue.get("code", "?")}',
            f'- Message: {issue.get("message", "")}',
        ]
        if issue.get('current_value'):
            parts.append(f'- Current value: {issue["current_value"]}')
        if issue.get('expected_value'):
            parts.append(f'- Expected value: {issue["expected_value"]}')
        if issue.get('suggested_fix'):
            parts.append(f'- Suggested fix: {issue["suggested_fix"]}')

        parts.append(f'\n### Code snippet (lines around the error)')
        parts.append(f'```sysml\n{snippet}\n```')

        if rag_context:
            parts.append(f'\n{rag_context}')

        parts.append(
            '\n### Rules\n'
            '1. Output ONLY the corrected snippet (same number of lines, same indentation)\n'
            '2. Fix ONLY the specific issue described above\n'
            '3. Do NOT add, remove, or modify any other content\n'
            '4. Wrap output in <fixed_snippet> tags'
        )
        return '\n'.join(parts)

    # ── Reinsertion ──

    def _reinsert_snippet(self, content, fixed_snippet, start_line, end_line):
        """Replaces the original snippet with the corrected one."""
        lines = content.split('\n')
        fixed_lines = fixed_snippet.split('\n')
        lines[start_line:end_line] = fixed_lines
        return '\n'.join(lines)

    # ── Individual fix ──

    def fix_issue(self, issue, content, packages=None):
        """Fixes a specific issue via snippet + LLM."""
        # 1. Extract snippet
        line_num = issue.get('line')
        current_val = issue.get('current_value', '')

        if line_num:
            snippet, start, end = self._extract_snippet_by_line(content, line_num)
        elif current_val:
            snippet, start, end = self._extract_snippet_by_pattern(content, current_val)
        else:
            self.escalated.append(issue)
            return content, False

        if snippet is None:
            self.escalated.append(issue)
            return content, False

        # 2. RAG: retrieve correct example from Apollo 11
        pkg_name = issue.get('package', '')
        retrieved = retrieve_sysml_examples(
            query=f'{pkg_name} {issue.get("code", "")} correct syntax',
            top_k=2,
            pkg_name_filter=pkg_name,
            threshold=99.0,
        )
        if not retrieved:
            layer = PACKAGE_TO_LAYER.get(pkg_name, 'technical')
            retrieved = retrieve_sysml_examples(
                query=f'{pkg_name} SysML v2 correct syntax',
                top_k=1,
                layer_filter=layer,
                threshold=99.0,
            )
        rag_ctx = format_rag_context(
            retrieved,
            label='CORRECT PATTERN — Follow this syntax when fixing'
        )

        # 3. LLM: fix snippet
        prompt = self._build_targeted_prompt(issue, snippet, rag_ctx or '')

        try:
            collected = []
            with self.client.messages.stream(
                model=self.model, max_tokens=self.MAX_TOKENS,
                system=TARGETED_FIX_SYSTEM_PROMPT,
                messages=[{'role': 'user', 'content': prompt}]
            ) as stream:
                for text in stream.text_stream:
                    collected.append(text)
            usage = stream.get_final_message().usage
            self.total_tokens_in += usage.input_tokens
            self.total_tokens_out += usage.output_tokens

            response = ''.join(collected)

            # 4. Parse response
            match = re.search(r'<fixed_snippet>\s*(.+?)\s*</fixed_snippet>',
                              response, re.DOTALL)
            if not match:
                self.escalated.append(issue)
                return content, False

            fixed_snippet = match.group(1).strip()
            fixed_snippet = re.sub(r'^```\w*\n', '', fixed_snippet)
            fixed_snippet = re.sub(r'\n```$', '', fixed_snippet)

            # 5. Reinsert
            new_content = self._reinsert_snippet(content, fixed_snippet, start, end)

            self.fixes_applied.append({
                'issue_code': issue['code'],
                'package': pkg_name,
                'action': f'Targeted: lines {start+1}-{end} fixed via LLM',
                'success': True,
            })
            return new_content, True

        except Exception as e:
            print(f'     ⚠️ Targeted LLM error: {e}')
            self.escalated.append(issue)
            return content, False

    # ── process_all ──

    def process_all(self, issues, packages):
        """Applies targeted fixes to all affected packages."""
        fixed = dict(packages)
        by_pkg = defaultdict(list)
        for i in issues: by_pkg[i['package']].append(i)

        if not issues:
            print(f'\n--- Tier 2: TargetedFixHandler (0 issues) ---')
            print(f'  No issues to fix.')
            return fixed

        print(f'\n--- Tier 2: TargetedFixHandler ({len(issues)} issues) ---')
        for pkg_name, pkg_issues in sorted(by_pkg.items()):
            content = fixed.get(pkg_name, '')
            if not content:
                for issue in pkg_issues:
                    self.escalated.append(issue)
                continue
            for issue in pkg_issues:
                content, success = self.fix_issue(issue, content, packages)
                icon = '✅' if success else '⚠️'
                action = self.fixes_applied[-1]['action'] if success and self.fixes_applied else '(escalated → Tier 3)'
                print(f'  {icon} [{issue["code"]}] {pkg_name}: {action}')
            fixed[pkg_name] = content

        cost = estimate_cost(self.total_tokens_in, self.total_tokens_out)
        print(f'  Result: {len(self.fixes_applied)} applied, {len(self.escalated)} escalated → Tier 3')
        print(f'  Tier 2 cost: ${cost:.4f} ({self.total_tokens_in:,} in / {self.total_tokens_out:,} out)')
        return fixed

# ── LLMFullFixHandler  ─────────────
# Tier 3: full package rewrite. Last resort.
# Activated when Tier 1 (Surgical) and Tier 2 (Targeted) failed.
# Guard _validate_rewrite() rejects rewrites that lose content.

LLM_FIX_SYSTEM_PROMPT = """You are a SysML v2 model fixer within the CoSMA framework.

You receive: (1) validation issues, (2) SysML v2 package file, (3) source-of-truth JSON.
Produce the CORRECTED version of the entire package file.

Rules:
- Fix ALL listed issues in a single pass
- Do NOT modify unrelated content
- Follow CoSMA patterns (#refinement, satisfy, etc.)
- Use source-of-truth JSON as authoritative reference
- Output the complete corrected file wrapped in <corrected_file> tags
- No explanations outside the tags

Critical SysML v2 rules:
- ISQ features use SUBSETTING (:>), e.g.: attribute speed :> ISQ::speed
- ScalarValues types use TYPING (:), e.g.: attribute isActive : ScalarValues::Boolean
- Instance names are camelCase (part missionSystem, action performLaunch)
- Type/def names are PascalCase (part def MissionSystem, action def PerformLaunch)
- Valid SI/CoSMA units (e.g.): kg, m, s, Hz, N, Pa, J, W, V, kn, nmi, km, h, min, rad, ['°']
- Do NOT use: MHz, GHz, deg, knots, ft, in, lb (convert to SI equivalents)
- satisfy paths require dot notation through state machine hierarchy
- #refinement dependency format: to PackageName::'REQ-ID'
- Every state def should subtype Phase (:> Phase) and contain do action"""


class LLMFullFixHandler:
    def __init__(self, client, json_extractor, model=None, max_tokens=P6_MAX_TOKENS):
        self.client = client
        self.json_extractor = json_extractor
        self.model = model or LLM_MODEL
        self.max_tokens = max_tokens
        self.fixes_applied = []
        self.total_tokens_in = 0
        self.total_tokens_out = 0

    def _build_user_prompt(self, package_name, issues, sysml_content):
        """Builds the fix prompt with RAG context."""
        # Formatted issues
        issues_block = []
        for idx, issue in enumerate(issues, 1):
            parts = [f'### Issue {idx}/{len(issues)}',
                     f'- Code: {issue.get("code", "?")}',
                     f'- Severity: {issue.get("severity", "?")}',
                     f'- Package: {issue.get("package", "?")}']
            if issue.get('line'): parts.append(f'- Line: {issue["line"]}')
            parts.append(f'- Message: {issue.get("message", "")}')
            if issue.get('current_value'): parts.append(f'- Current value: {issue["current_value"]}')
            if issue.get('expected_value'): parts.append(f'- Expected value: {issue["expected_value"]}')
            if issue.get('suggested_fix'): parts.append(f'- Suggested fix: {issue["suggested_fix"]}')
            issues_block.append('\n'.join(parts))

        # JSON source-of-truth
        json_context = self.json_extractor.extract_for_package(package_name)

        # RAG: retrieve Apollo 11 reference
        retrieved = retrieve_sysml_examples(
            query=f'{package_name} SysML v2 correct syntax patterns',
            top_k=2,
            pkg_name_filter=package_name,
            threshold=99.0,
        )
        if not retrieved:
            layer = PACKAGE_TO_LAYER.get(package_name, 'technical')
            retrieved = retrieve_sysml_examples(
                query=f'{package_name} SysML v2 correct syntax',
                top_k=1,
                layer_filter=layer,
                threshold=99.0,
            )
        rag_ctx = format_rag_context(
            retrieved,
            label='CORRECT PATTERN REFERENCE — Follow this syntax when fixing'
        )
        if rag_ctx:
            print(f'       [RAG] {len(retrieved)} references retrieved for {package_name}')

        # Build prompt
        prompt_parts = [
            f'## Issues to fix in {package_name} ({len(issues)} issues)\n',
            '\n\n'.join(issues_block),
            f'\n\n## Current file (with errors)\n```sysml\n{sysml_content}\n```',
            f'\n\n## JSON source-of-truth\n```json\n{json_context}\n```',
        ]
        if rag_ctx:
            prompt_parts.append(f'\n\n{rag_ctx}')
        prompt_parts.append(
            '\n\nFix ALL issues. Follow the ground truth reference patterns. '
            'Output the corrected file in <corrected_file> tags.'
        )
        return '\n'.join(prompt_parts)

    def _call_llm(self, system_prompt, user_prompt):
        collected = []
        with self.client.messages.stream(
            model=self.model, max_tokens=self.max_tokens,
            system=system_prompt, messages=[{'role': 'user', 'content': user_prompt}]
        ) as stream:
            for text in stream.text_stream: collected.append(text)
        usage = stream.get_final_message().usage
        self.total_tokens_in += usage.input_tokens
        self.total_tokens_out += usage.output_tokens
        return ''.join(collected)

    def _parse_response(self, response):
        match = re.search(r'<corrected_file>\s*(.+?)\s*</corrected_file>', response, re.DOTALL)
        if match:
            content = match.group(1).strip()
            content = re.sub(r'^```\w*\n', '', content)
            content = re.sub(r'\n```$', '', content)
            return content
        match = re.search(r'```(?:sysml)?\s*\n(.+?)\n```', response, re.DOTALL)
        if match: return match.group(1).strip()
        if response.strip().startswith('//'): return response.strip()
        return None

    def _validate_rewrite(self, original, rewritten, pkg_name):
        """Regression guard: rejects rewrites that lose content or introduce errors."""
        # Check 1: did not lose > 20% of definitions
        orig_defs = len(re.findall(r'\bdef\s+\w+', original))
        new_defs = len(re.findall(r'\bdef\s+\w+', rewritten))
        if orig_defs > 0 and new_defs < orig_defs * 0.8:
            print(f'     ⚠️ {pkg_name}: rewrite rejected (lost definitions: {orig_defs} → {new_defs})')
            return False

        # Check 2: did not become too short
        orig_lines = len(original.split('\n'))
        new_lines = len(rewritten.split('\n'))
        if new_lines < orig_lines * 0.5:
            print(f'     ⚠️ {pkg_name}: rewrite rejected (too short: {orig_lines} → {new_lines} lines)')
            return False

        # Check 3: reasonable size ratio
        ratio = len(rewritten) / max(len(original), 1)
        if ratio < 0.5:
            print(f'     ⚠️ {pkg_name}: rewrite rejected (size {ratio:.0%} of original)')
            return False

        # Check 4: post-structural validation — reject if more errors were introduced
        sv = StructuralValidator()
        orig_errors = sum(1 for i in sv.validate(original, pkg_name) if i.severity == P5_ERROR)
        new_errors = sum(1 for i in sv.validate(rewritten, pkg_name) if i.severity == P5_ERROR)
        if new_errors > orig_errors:
            print(f'     ⚠️ {pkg_name}: rewrite rejected (structural errors: {orig_errors} → {new_errors})')
            return False

        return True

    def fix_package(self, package_name, issues, sysml_content):
        prompt = self._build_user_prompt(package_name, issues, sysml_content)
        print(f'  🤖 LLM Full: {package_name} ({len(issues)} issues)...')
        t = time.time()
        try:
            response = self._call_llm(LLM_FIX_SYSTEM_PROMPT, prompt)
            corrected = self._parse_response(response)
            elapsed = time.time() - t
            if corrected is None:
                print(f'     ❌ Parse failed ({elapsed:.1f}s)')
                self.fixes_applied.append({'package': package_name, 'success': False,
                    'reason': 'parse_failed'})
                return sysml_content, False

            # Regression guard
            if not self._validate_rewrite(sysml_content, corrected, package_name):
                self.fixes_applied.append({'package': package_name, 'success': False,
                    'reason': 'validation_rejected'})
                return sysml_content, False

            print(f'     ✅ Fixed ({elapsed:.1f}s, {len(corrected):,} chars)')
            self.fixes_applied.append({'package': package_name, 'issues_count': len(issues),
                'success': True, 'elapsed_s': round(elapsed, 1)})
            return corrected, True
        except Exception as e:
            print(f'     ❌ Error: {e}')
            self.fixes_applied.append({'package': package_name, 'success': False,
                'reason': str(e)})
            return sysml_content, False

    def process_all(self, issues, packages):
        fixed = dict(packages)
        by_pkg = defaultdict(list)
        for i in issues: by_pkg[i['package']].append(i)

        if not issues:
            print(f'\n--- Tier 3: LLMFullFixHandler (0 issues) ---')
            print(f'  No issues to fix.')
            return fixed

        print(f'\n--- Tier 3: LLMFullFixHandler ({len(issues)} issues, {len(by_pkg)} packages) ---')
        for pkg_name, pkg_issues in sorted(by_pkg.items()):
            content = fixed.get(pkg_name, '')
            corrected, success = self.fix_package(pkg_name, pkg_issues, content)
            if success: fixed[pkg_name] = corrected

        cost = estimate_cost(self.total_tokens_in, self.total_tokens_out)
        successful = sum(1 for f in self.fixes_applied if f.get('success'))
        rejected = sum(1 for f in self.fixes_applied if f.get('reason') == 'validation_rejected')
        print(f'  Tier 3: {successful}/{len(by_pkg)} packages fixed, {rejected} rejected')
        print(f'  Tier 3 cost: ${cost:.4f} ({self.total_tokens_in:,} in / {self.total_tokens_out:,} out)')
        return fixed

# ── HumanReviewHandler  ──────────────────────
# Only documents HUMAN_ASSISTED issues for final human review.

class HumanReviewHandler:
    def __init__(self):
        self.report_items = []

    def process(self, issues):
        if not issues:
            print(f'\n--- Human Review (0 issues) ---')
            return self.report_items

        print(f'\n--- Human Review ({len(issues)} issues → Eclipse IDE) ---')
        for i in issues:
            self.report_items.append({
                'package': i.get('package'),
                'code': i.get('code'),
                'message': i.get('message', '')[:120],
                'suggested_fix': i.get('suggested_fix', ''),
                'status': 'PENDING',
            })
            print(f'  📋 [{i.get("code")}] {i.get("package")}: {i.get("message", "")[:75]}')
        return self.report_items


# ── FixerAgent (3-tier cascade) ─────────────
# Tier 1: SurgicalFixHandler
# Tier 2: TargetedFixHandler
# Tier 3: LLMFullFixHandler
# Human: documentation

class FixerAgent:
    def __init__(self, validation_report, refined_packages, mission_data, client, output_dir):
        self.validation_report = validation_report
        self.original_packages = dict(refined_packages)
        self.packages = dict(refined_packages)
        self.mission_data = mission_data
        self.output_dir = Path(str(output_dir))

        self.parser = IssueParser(validation_report)
        self.surgical_handler = SurgicalFixHandler()
        self.targeted_handler = TargetedFixHandler(client)
        self.llm_full_handler = LLMFullFixHandler(client, JSONContextExtractor(mission_data))
        self.human_handler = HumanReviewHandler()
        self.fix_report = {}

    def run(self):
        t0 = time.time()
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        iteration = self.validation_report.get('iteration', 0)

        print(f'\n{"="*70}')
        print(f'  P6 — FixerAgent v5.1 (3-tier cascade) — iteration {iteration}')
        print(f'{"="*70}')

        # Filter out issues from phantom packages
        real_packages = set(self.packages.keys())
        self.parser.auto_fixable = [i for i in self.parser.auto_fixable
                                     if i.get('package', '') in real_packages]

        # Filter out CoSMA framework packages (groundtruth-sourced, must not be modified by P6)
        cosma_framework = {'CoSMAPackage', 'CoSMAQuantitiesAndUnitsPackage', 'CoSMAViewsPackage'}
        cosma_filtered = [i for i in self.parser.auto_fixable if i.get('package', '') in cosma_framework]
        if cosma_filtered:
            print(f'  ℹ️ {len(cosma_filtered)} issues in CoSMA framework packages suppressed (GT-sourced)')
        self.parser.auto_fixable = [i for i in self.parser.auto_fixable
                                     if i.get('package', '') not in cosma_framework]

        # Split by fix_strategy
        surgical_issues = self.parser.get_by_strategy(AUTO_SURGICAL)
        targeted_issues = self.parser.get_by_strategy(AUTO_TARGETED)
        llm_issues = self.parser.get_by_strategy(AUTO_LLM)

        self.parser.summary()

        # ── Tier 1: Surgical  ──
        self.packages = self.surgical_handler.process_all(surgical_issues, self.packages)
        escalated_1 = self.surgical_handler.escalated

        # ── Tier 2: Targeted (surgical LLM) ──
        all_targeted = targeted_issues + escalated_1
        self.packages = self.targeted_handler.process_all(all_targeted, self.packages)
        escalated_2 = self.targeted_handler.escalated

        # ── Tier 3: LLM Full (last resort) ──
        all_full = llm_issues + escalated_2
        self.packages = self.llm_full_handler.process_all(all_full, self.packages)

        # ── Human (documentation for human review) ──
        human_report = self.human_handler.process(self.parser.human_review)

        # ── Metrics ──
        elapsed = time.time() - t0
        modified = [p for p in self.packages
                    if self.packages[p] != self.original_packages.get(p, '')]

        total_received = len(surgical_issues) + len(targeted_issues) + len(llm_issues)
        total_fixed = (len(self.surgical_handler.fixes_applied)
                       + len(self.targeted_handler.fixes_applied)
                       + sum(1 for f in self.llm_full_handler.fixes_applied if f.get('success')))

        targeted_cost = estimate_cost(self.targeted_handler.total_tokens_in,
                                      self.targeted_handler.total_tokens_out)
        full_cost = estimate_cost(self.llm_full_handler.total_tokens_in,
                                  self.llm_full_handler.total_tokens_out)
        total_cost = targeted_cost + full_cost

        self.fix_report = {
            'timestamp': timestamp,
            'iteration': iteration,
            'elapsed_s': round(elapsed, 1),
            'model': LLM_MODEL,

            # Per tier
            'surgical': {
                'attempted': len(surgical_issues),
                'succeeded': len(self.surgical_handler.fixes_applied),
                'escalated': len(escalated_1),
                'cost_usd': 0.0,
            },
            'targeted': {
                'attempted': len(all_targeted),
                'succeeded': len(self.targeted_handler.fixes_applied),
                'escalated': len(escalated_2),
                'cost_usd': round(targeted_cost, 4),
                'tokens_in': self.targeted_handler.total_tokens_in,
                'tokens_out': self.targeted_handler.total_tokens_out,
            },
            'llm_full': {
                'attempted': len(all_full),
                'succeeded': sum(1 for f in self.llm_full_handler.fixes_applied if f.get('success')),
                'rejected': sum(1 for f in self.llm_full_handler.fixes_applied
                                if f.get('reason') == 'validation_rejected'),
                'cost_usd': round(full_cost, 4),
                'tokens_in': self.llm_full_handler.total_tokens_in,
                'tokens_out': self.llm_full_handler.total_tokens_out,
            },
            'human_review': {
                'documented': len(human_report),
            },

            # Aggregates
            'total_issues_received': total_received,
            'total_issues_fixed': total_fixed,
            'fix_rate': round(total_fixed / max(total_received, 1), 3),
            'packages_modified': sorted(modified),
            'total_cost_usd': round(total_cost, 4),
        }

        # ── Final print ──
        print(f'\n{"="*70}')
        print(f'  P6 finished — iteration {iteration}')
        print(f'{"="*70}')
        print(f'  Tier 1 (Surgical):   {len(self.surgical_handler.fixes_applied)} fixed, '
              f'{len(escalated_1)} escalated | $0.00')
        print(f'  Tier 2 (Targeted):   {len(self.targeted_handler.fixes_applied)} fixed, '
              f'{len(escalated_2)} escalated | ${targeted_cost:.4f}')
        suc_3 = sum(1 for f in self.llm_full_handler.fixes_applied if f.get('success'))
        rej_3 = sum(1 for f in self.llm_full_handler.fixes_applied if f.get('reason') == 'validation_rejected')
        print(f'  Tier 3 (LLM Full):   {suc_3} fixed, '
              f'{rej_3} rejected | ${full_cost:.4f}')
        print(f'  Human review:        {len(human_report)} documented')
        print(f'  Total: {total_fixed}/{total_received} fixed ({self.fix_report["fix_rate"]:.1%}) | '
              f'{len(modified)} packages modified | ${total_cost:.4f} | {elapsed:.1f}s')
        print(f'{"="*70}')

        return self.fix_report

    def save(self):
        ts = self.fix_report.get('timestamp', datetime.now().strftime('%Y%m%d_%H%M%S'))
        fixed_dir = self.output_dir / f'fixed_{ts}'
        fixed_dir.mkdir(parents=True, exist_ok=True)
        for pkg, content in sorted(self.packages.items()):
            (fixed_dir / f'{pkg}.sysml').write_text(content, encoding='utf-8')
        report_path = self.output_dir / f'fix_report_{ts}.json'
        with open(report_path, 'w', encoding='utf-8') as f:
            json.dump(self.fix_report, f, indent=2, ensure_ascii=False)
        print(f'  Saved: {fixed_dir} + {report_path.name}')
        return fixed_dir, report_path


# ── run_p6: wrapper State → State ──────────────────────────

def run_p6(state: dict) -> dict:
    """FixerAgent (P6): validation_report + packages → corrected packages.

    Input: validation_report, refined_packages, mission_json
    Output: refined_packages (updated), fix_report
    """
    print(f'\n{"="*60}')
    print(f'  P6 — FixerAgent (3-Tier Cascade)')
    print(f'{"="*60}')
    t0 = time.time()
    iteration = state.get('iteration', 0)

    fixer = FixerAgent(
        validation_report=state['validation_report'],
        refined_packages=state['refined_packages'],
        mission_data=state['mission_json'],
        client=CLIENT,
        output_dir=FIXER_DIR,
    )

    fix_report = fixer.run()
    fixer.save()

    elapsed = time.time() - t0
    cost = fix_report.get('total_cost_usd', 0)
    tokens_in = (fix_report.get('targeted', {}).get('tokens_in', 0)
                 + fix_report.get('llm_full', {}).get('tokens_in', 0))
    tokens_out = (fix_report.get('targeted', {}).get('tokens_out', 0)
                  + fix_report.get('llm_full', {}).get('tokens_out', 0))
    log_cost(state, f'P6_FixerAgent_iter{iteration}', cost, elapsed, tokens_in, tokens_out)

    return {
        'refined_packages': dict(fixer.packages),
        'fix_report': fix_report,
    }


# ── Verification ───────────────────────────────────────────
print(f'P6 defined: run_p6(state) → refined_packages (updated), fix_report')
print(f'  3 tiers: Surgical → Targeted → LLM Full')
print(f'  + Human review (documentation)')

P6 defined: run_p6(state) → refined_packages (updated), fix_report
  3 tiers: Surgical → Targeted → LLM Full
  + Human review (documentation)


---

## Block E — LangGraph StateGraph


### Cell 10 — StateGraph, Wrappers and Conditional Loop



In [ ]:
# ============================================================
# Master Orchestrator
# Cell 10 — LangGraph StateGraph + Wrappers + Resume + Error-Safe
# ============================================================

import operator
from typing import Annotated
from langgraph.graph import StateGraph, START, END


# ── 1. PipelineState with reducers + resume_from ──────────

class PipelineState(TypedDict):
    # Inputs
    pdf_bytes: bytes
    pdf_filename: str
    # P1
    mission_markdown: str
    layer_markdowns: dict
    # P2
    mission_json: dict
    # P3
    skeletons: dict
    # P4
    refined_packages: dict
    # P5
    validation_report: dict
    # P6
    fix_report: dict
    # Control
    iteration: int
    max_iterations: int
    pipeline_status: str
    resume_from: str              # None | 'p1'|'p2'|'p3'|'p4'|'p5'|'p6'
    # Cumulative lists (reducer = operator.add)
    cost_log: Annotated[list, operator.add]
    errors: Annotated[list, operator.add]
    # Paths
    output_base_dir: str
    run_timestamp: str
    pdf_pages: int
    skip_monticore: bool

# ── 2. log_cost / log_error LangGraph-safe ────────────────

def _make_cost_entry(agent, cost_usd, time_s, tokens_in=0, tokens_out=0):
    entry = {
        'agent': agent,
        'cost_usd': round(cost_usd, 4),
        'time_s': round(time_s, 1),
        'tokens_in': tokens_in,
        'tokens_out': tokens_out,
        'timestamp': datetime.now().isoformat(),
    }
    print(f'  💰 {agent}: ${cost_usd:.4f} | {time_s:.1f}s | {tokens_in:,} in / {tokens_out:,} out')
    return entry

_cost_buffer = []

def log_cost(state, agent, cost_usd, time_s, tokens_in=0, tokens_out=0):
    entry = _make_cost_entry(agent, cost_usd, time_s, tokens_in, tokens_out)
    _cost_buffer.append(entry)

def log_error(state, agent, error):
    print(f'  ❌ {agent}: {error}')


# ── 3. Error-safe wrapper factory ─────────────────────────
# Each node is wrapped in try/except. On error:
# - Saves a snapshot of the state on Drive
# - Returns pipeline_status = 'ERROR_AT_PX'
# - The pipeline proceeds to delivery (does not crash)

LAST_RUN_TIMESTAMP = None  # Updated during execution to ease resume

def _save_error_snapshot(state, agent_name, error_msg):
    """Saves a state snapshot to Drive to enable resume."""
    ts = state.get('run_timestamp', 'unknown')
    snapshot_dir = ORCHESTRATOR_DIR / f'error_snapshot_{ts}'
    snapshot_dir.mkdir(parents=True, exist_ok=True)

    # Save packages if any (most valuable output)
    for key in ('refined_packages', 'skeletons'):
        pkgs = state.get(key, {})
        if pkgs:
            pkg_dir = snapshot_dir / key
            pkg_dir.mkdir(exist_ok=True)
            for name, content in pkgs.items():
                (pkg_dir / f'{name}.sysml').write_text(content, encoding='utf-8')

    # Save mission_json if any
    mj = state.get('mission_json', {})
    if mj:
        with open(snapshot_dir / 'mission_json.json', 'w', encoding='utf-8') as f:
            json.dump(mj, f, indent=2, ensure_ascii=False)

    # Save layer_markdowns if any
    lm = state.get('layer_markdowns', {})
    if lm:
        for layer_name, md_content in lm.items():
            (snapshot_dir / f'layer_{layer_name}.md').write_text(md_content, encoding='utf-8')

    # Save error
    error_info = {
        'agent': agent_name,
        'error': str(error_msg),
        'timestamp': datetime.now().isoformat(),
        'run_timestamp': ts,
    }
    with open(snapshot_dir / 'error_info.json', 'w', encoding='utf-8') as f:
        json.dump(error_info, f, indent=2, ensure_ascii=False)

    print(f'  📁 Snapshot saved: {snapshot_dir}')
    return snapshot_dir


def _make_error_safe_wrapper(run_fn, agent_name, agent_label):
    """Creates an error-safe wrapper for a pipeline node."""
    def wrapper(state: dict) -> dict:
        _cost_buffer.clear()
        try:
            result = run_fn(state)
            return {**result, 'cost_log': list(_cost_buffer)}
        except Exception as e:
            print(f'\n{"="*70}')
            print(f'  ❌ ERROR in {agent_label}: {e}')
            print(f'{"="*70}')
            _save_error_snapshot(state, agent_name, e)
            ts = state.get('run_timestamp', 'unknown')
            print(f'\n  ✅ Previous outputs preserved.')
            print(f'  To resume: set RESUME_FROM = \'{agent_name}\' '
                  f'at the top of the execution cell and re-run.')
            print(f'{"="*70}\n')
            return {
                'pipeline_status': f'ERROR_AT_{agent_name.upper()}',
                'cost_log': list(_cost_buffer),
                'errors': [{'agent': agent_name, 'error': str(e)}],
            }
    return wrapper


# ── 4. Node wrappers (error-safe) ─────────────────────────

node_p1 = _make_error_safe_wrapper(run_p1, 'p1', 'P1 PreprocessingAgent')
node_p2 = _make_error_safe_wrapper(run_p2, 'p2', 'P2 MissionAgent')
node_p3 = _make_error_safe_wrapper(run_p3, 'p3', 'P3 TemplateGeneratorClass')
node_p4 = _make_error_safe_wrapper(run_p4, 'p4', 'P4 RefinementAgent')
node_p5 = _make_error_safe_wrapper(run_p5, 'p5', 'P5 ValidatorAgent')
node_p6 = _make_error_safe_wrapper(run_p6, 'p6', 'P6 FixerAgent')


def node_increment(state: dict) -> dict:
    """Increments the iteration counter after P6."""
    new_iter = state.get('iteration', 0) + 1
    max_iter = state.get('max_iterations', MAX_ITERATIONS)
    is_final = new_iter >= max_iter
    if is_final:
        print(f'\n  🔄 Loop P5↔P6: iteration {new_iter}/{max_iter} (FINAL — P5 without MontiCore)')
    else:
        print(f'\n  🔄 Loop P5↔P6: iteration {new_iter}/{max_iter}')
    return {'iteration': new_iter, 'skip_monticore': is_final}


# ── 5. Router node ─────────────────────────────────

def node_router(state: dict) -> dict:
    """Determines the pipeline entry point based on resume_from."""
    resume = state.get('resume_from') or 'p1'
    if resume == 'p1':
        print(f'  ▶️ Full pipeline: P1 → P2 → P3 → P4 → P5 → ...')
    else:
        print(f'\n{"="*70}')
        print(f'  🔄 RESUMING pipeline from {resume.upper()}')
        print(f'  run_timestamp: {state.get("run_timestamp", "?")}')
        print(f'{"="*70}\n')
    return {}

def route_from_router(state: dict) -> str:
    """Conditional edge: router → correct agent."""
    resume = state.get('resume_from') or 'p1'
    status = state.get('pipeline_status', '')

    # If an error already happened, go straight to delivery
    if status.startswith('ERROR_AT_'):
        return 'delivery'

    valid = {'p1', 'p2', 'p3', 'p4', 'p5', 'p6'}
    if resume in valid:
        return resume
    return 'p1'


# ── 6. Post-agent error check ─────────────────────────────
# After each agent, check whether an error occurred to divert to delivery.

def should_continue_or_error(next_node):
    """Generates a conditional-edge function that checks for error before proceeding."""
    def check(state: dict) -> str:
        status = state.get('pipeline_status', '')
        if status.startswith('ERROR_AT_'):
            return 'delivery'
        return next_node
    return check


# ── 7. P5 → fix or delivery  ───────────────────────────────

def should_fix_or_deliver(state: dict) -> str:
    """Decides whether the pipeline should fix (P6) or deliver."""
    status = state.get('pipeline_status', '')
    if status.startswith('ERROR_AT_'):
        return 'delivery'

    iteration = state.get('iteration', 0)
    max_iter = state.get('max_iterations', MAX_ITERATIONS)
    report = state.get('validation_report', {})
    auto_fixable_count = report.get('auto_fixable_count', 0)

    if auto_fixable_count == 0:
        print(f'  ✅ auto_fixable_count=0 — converged, forwarding to delivery')
        return 'delivery'

    if iteration >= max_iter:
        print(f'  ⚠️ Max iterations ({max_iter}) reached — forwarding to delivery')
        return 'delivery'

    print(f'  🔧 {auto_fixable_count} auto-fixable issues — forwarding to P6')
    return 'fix'


# ── 8. Delivery node ──────────────────────────────────────

def node_delivery(state: dict) -> dict:
    """Final node: determines pipeline status and generates the report."""
    global LAST_RUN_TIMESTAMP
    LAST_RUN_TIMESTAMP = state.get('run_timestamp')

    iteration = state.get('iteration', 0)
    report = state.get('validation_report', {})
    metrics = report.get('metrics', {})
    status = state.get('pipeline_status', '')

    # Determine final status
    if status.startswith('ERROR_AT_'):
        pipeline_status = status  # Keep ERROR_AT_PX
    else:
        auto_fixable_count = report.get('auto_fixable_count', 0)
        if auto_fixable_count == 0:
            pipeline_status = 'CONVERGED'
        elif iteration >= state.get('max_iterations', MAX_ITERATIONS):
            pipeline_status = 'MAX_ITERATIONS'
        else:
            pipeline_status = 'INCOMPLETE'

    # Save final packages (if any)
    final_dir = ORCHESTRATOR_DIR / f'final_{state["run_timestamp"]}'
    if state.get('refined_packages'):
        write_packages_to_dir(state['refined_packages'], final_dir, 'FINAL')

    # Total cost
    total_cost = sum(e.get('cost_usd', 0) for e in state.get('cost_log', []))
    total_time = sum(e.get('time_s', 0) for e in state.get('cost_log', []))

    # Final metrics
    scr = metrics.get('scr', {}).get('value')
    gc = metrics.get('gc', {}).get('value')
    sc = metrics.get('sc', {}).get('value')

    print(f'\n{"="*70}')
    is_error = pipeline_status.startswith('ERROR_AT_')
    icon = '❌' if is_error else '✅'
    print(f'  {icon} PIPELINE {"STOPPED" if is_error else "COMPLETED"} — {pipeline_status}')
    print(f'{"="*70}')
    print(f'  Status:       {pipeline_status}')
    print(f'  Iterations:   {iteration}')

    if is_error:
        failed_agent = pipeline_status.replace('ERROR_AT_', '').lower()
        print(f'\n  Previous outputs preserved on Drive.')
        print(f'  To resume: set RESUME_FROM = \'{failed_agent}\' '
              f'at the top of the execution cell and re-run.')
    else:
        if scr is not None:
            print(f'  SCR:          {scr:.1%}')
        else:
            print(f'  SCR:          N/A (MontiCore unavailable)')
        print(f'  GC:           {gc:.1%}' if gc is not None else f'  GC:           N/A')
        print(f'  SC:           {sc:.1%}' if sc is not None else f'  SC:           N/A')
        print(f'  Auto-fixable: {report.get("auto_fixable_count", 0)} remaining')
        print(f'  Human-review: {report.get("human_review_count", 0)} for Eclipse')

    print(f'  Total cost:   ${total_cost:.4f}')
    print(f'  Total time:   {total_time:.1f}s ({total_time/60:.1f} min)')
    print(f'  Packages:     {len(state.get("refined_packages", {}))}')
    if not is_error:
        print(f'  Output:       {final_dir}')
    print(f'{"="*70}')

    return {'pipeline_status': pipeline_status}


# ── 9. build_resume_state  ──────────────────────────

def build_resume_state(resume_from, run_timestamp=None, pdf_bytes=None, pdf_filename=None):
    """Reconstructs state from outputs saved on Drive.

    Args:
        resume_from: 'p2'|'p3'|'p4'|'p5'|'p6' — agent to re-run
        run_timestamp: timestamp of the previous run (default: LAST_RUN_TIMESTAMP)
        pdf_bytes: PDF bytes (default: reload from snapshot)
        pdf_filename: PDF filename (default: reload from snapshot)

    Returns:
        dict: state ready for pipeline.invoke()
    """
    ts = run_timestamp or LAST_RUN_TIMESTAMP
    if not ts:
        raise ValueError('run_timestamp not provided and LAST_RUN_TIMESTAMP not available. '
                         'Provide the timestamp explicitly.')

    print(f'\n{"="*70}')
    print(f'  🔄 Building state to resume from {resume_from.upper()}')
    print(f'  run_timestamp: {ts}')
    print(f'{"="*70}\n')

    state = {
        'pdf_bytes': pdf_bytes or b'',
        'pdf_filename': pdf_filename or '',
        'mission_markdown': '',
        'layer_markdowns': {},
        'mission_json': {},
        'skeletons': {},
        'refined_packages': {},
        'validation_report': {},
        'fix_report': {},
        'iteration': 0,
        'max_iterations': MAX_ITERATIONS,
        'pipeline_status': 'RUNNING',
        'resume_from': resume_from,
        'cost_log': [],
        'errors': [],
        'output_base_dir': str(BASE_DIR),
        'run_timestamp': ts,
        'pdf_pages': 0,
        'skip_monticore': False,
    }

    agents_order = ['p1', 'p2', 'p3', 'p4', 'p5']
    resume_idx = agents_order.index(resume_from) if resume_from in agents_order else 0

    # ── Load P1 outputs (if resuming from p2+) ──
    if resume_idx >= 1:
        p1_dir = PREPROCESSING_DIR
        md_files = sorted(p1_dir.rglob(f'*{ts}*.md')) or sorted(p1_dir.rglob('*.md'))
        if md_files:
            state['mission_markdown'] = md_files[-1].read_text(encoding='utf-8')
            print(f'  ✅ P1 loaded: {md_files[-1].name} ({len(state["mission_markdown"]):,} chars)')
        # Layer markdowns
        layer_dir = p1_dir
        layer_files = sorted(layer_dir.rglob(f'layer_*_{ts}.md')) or sorted(layer_dir.rglob('layer_*.md'))
        if layer_files:
            for lf in layer_files:
                layer_name = lf.stem.replace(f'_{ts}', '').replace('layer_', '')
                state['layer_markdowns'][layer_name] = lf.read_text(encoding='utf-8')
            print(f'  ✅ P1 layers: {len(state["layer_markdowns"])} layers loaded')

    # ── Load P2 outputs (if resuming from p3+) ──
    if resume_idx >= 2:
        p2_dirs = sorted(MISSION_AGENT_DIR.glob(f'run_{ts}*'))
        if not p2_dirs:
            p2_dirs = sorted(MISSION_AGENT_DIR.glob('run_*'))
        if p2_dirs:
            p2_dir = p2_dirs[-1]
            json_files = list(p2_dir.glob('mission_data*.json')) or list(p2_dir.glob('*.json'))
            if json_files:
                with open(json_files[-1], 'r', encoding='utf-8') as f:
                    state['mission_json'] = json.load(f)
                print(f'  ✅ P2 loaded: {json_files[-1].name}')
            else:
                # Fallback: error snapshot
                snap_json = ORCHESTRATOR_DIR / f'error_snapshot_{ts}' / 'mission_json.json'
                if snap_json.exists():
                    with open(snap_json, 'r', encoding='utf-8') as f:
                        state['mission_json'] = json.load(f)
                    print(f'  ✅ P2 loaded from snapshot: {snap_json.name}')

    # ── Load P3 outputs (if resuming from p4+) ──
    if resume_idx >= 3:
        p3_dirs = sorted(TEMPLATE_DIR.glob(f'skeletons_{ts}*'))
        if not p3_dirs:
            p3_dirs = sorted(TEMPLATE_DIR.glob('skeletons_*'))
        if p3_dirs:
            p3_dir = p3_dirs[-1]
            sysml_files = list(p3_dir.rglob('*.sysml'))
            for sf in sysml_files:
                pkg_name = sf.stem
                state['skeletons'][pkg_name] = sf.read_text(encoding='utf-8')
            print(f'  ✅ P3 loaded: {len(state["skeletons"])} skeletons from {p3_dir.name}')
        else:
            snap_dir = ORCHESTRATOR_DIR / f'error_snapshot_{ts}' / 'skeletons'
            if snap_dir.exists():
                for sf in snap_dir.rglob('*.sysml'):
                    state['skeletons'][sf.stem] = sf.read_text(encoding='utf-8')
                print(f'  ✅ P3 loaded from snapshot: {len(state["skeletons"])} skeletons')

    # ── Load P4 outputs (if resuming from p5+) ──
    if resume_idx >= 4:
        p4_dirs = sorted(REFINEMENT_DIR.glob(f'refined_{ts}*'))
        if not p4_dirs:
            p4_dirs = sorted(REFINEMENT_DIR.glob('refined_*'))
        if p4_dirs:
            p4_dir = p4_dirs[-1]
            sysml_files = list(p4_dir.rglob('*.sysml'))
            for sf in sysml_files:
                pkg_name = sf.stem
                state['refined_packages'][pkg_name] = sf.read_text(encoding='utf-8')
            print(f'  ✅ P4 loaded: {len(state["refined_packages"])} packages from {p4_dir.name}')
        else:
            snap_dir = ORCHESTRATOR_DIR / f'error_snapshot_{ts}' / 'refined_packages'
            if snap_dir.exists():
                for sf in snap_dir.rglob('*.sysml'):
                    state['refined_packages'][sf.stem] = sf.read_text(encoding='utf-8')
                print(f'  ✅ P4 loaded from snapshot: {len(state["refined_packages"])} packages')

    # Validate that required data was loaded
    required = {
        'p2': [],
        'p3': ['mission_json'],
        'p4': ['mission_json', 'skeletons'],
        'p5': ['mission_json', 'refined_packages'],
        'p6': ['mission_json', 'refined_packages', 'validation_report'],
    }
    missing = []
    for field in required.get(resume_from, []):
        val = state.get(field)
        if not val or (isinstance(val, dict) and len(val) == 0):
            missing.append(field)

    if missing:
        print(f'\n  ⚠️ ATTENTION: required fields not found: {missing}')
        print(f'  The pipeline may fail. Check the outputs on Drive.')
    else:
        print(f'\n  ✅ All required fields loaded to resume from {resume_from.upper()}')

    print(f'{"="*70}\n')
    return state


# ── 10. Build StateGraph ───────────────────────────────────

graph = StateGraph(PipelineState)

# Nodes
graph.add_node('router',             node_router)
graph.add_node('p1_preprocessing',   node_p1)
graph.add_node('p2_mission_agent',   node_p2)
graph.add_node('p3_template_gen',    node_p3)
graph.add_node('p4_refinement',      node_p4)
graph.add_node('p5_validator',       node_p5)
graph.add_node('p6_fixer',           node_p6)
graph.add_node('increment',          node_increment)
graph.add_node('delivery',           node_delivery)

# START → router
graph.add_edge(START, 'router')

# Router → correct agent (conditional)
graph.add_conditional_edges(
    'router',
    route_from_router,
    {
        'p1': 'p1_preprocessing',
        'p2': 'p2_mission_agent',
        'p3': 'p3_template_gen',
        'p4': 'p4_refinement',
        'p5': 'p5_validator',
        'p6': 'p6_fixer',
        'delivery': 'delivery',
    }
)

# Linear edges with error-check: P1 → P2 → P3 → P4 → P5
graph.add_conditional_edges('p1_preprocessing', should_continue_or_error('p2_mission_agent'),
    {'p2_mission_agent': 'p2_mission_agent', 'delivery': 'delivery'})
graph.add_conditional_edges('p2_mission_agent', should_continue_or_error('p3_template_gen'),
    {'p3_template_gen': 'p3_template_gen', 'delivery': 'delivery'})
graph.add_conditional_edges('p3_template_gen', should_continue_or_error('p4_refinement'),
    {'p4_refinement': 'p4_refinement', 'delivery': 'delivery'})
graph.add_conditional_edges('p4_refinement', should_continue_or_error('p5_validator'),
    {'p5_validator': 'p5_validator', 'delivery': 'delivery'})

# P5 → delivery OR P6 (loop with error-check)
graph.add_conditional_edges(
    'p5_validator',
    should_fix_or_deliver,
    {'delivery': 'delivery', 'fix': 'p6_fixer'}
)

# P6 → increment → P5 (with error-check on P6)
graph.add_conditional_edges('p6_fixer', should_continue_or_error('increment'),
    {'increment': 'increment', 'delivery': 'delivery'})
graph.add_edge('increment', 'p5_validator')

# Terminal
graph.add_edge('delivery', END)

# Compile
pipeline = graph.compile()


# ── 11. Verification ───────────────────────────────────────
print(f'StateGraph compiled successfully. ✅')
print(f'  Nodes: router → p1 → p2 → p3 → p4 → p5 → [P6↔P5 loop] → delivery')
print(f'  Modular resume: RESUME_FROM = "p1"|"p2"|"p3"|"p4"|"p5"|"p6"')
print(f'  Error-safe: each agent saves a snapshot on Drive on failure')
print(f'  Termination condition:')
print(f'    (a) CONVERGED: auto_fixable_count == 0')
print(f'    (b) MAX_ITERATIONS: {MAX_ITERATIONS} cycles')
print(f'\nReady to run. ✅')

---

## Block F — End-to-end execution and final report

### Cell 11 — Military mission (PDF) upload, execution and final report

In [ ]:
# ============================================================
# Master Orchestrator
# Cell 11 — PDF Upload + End-to-End Execution + Resume
# ============================================================
# RESUME_FROM allows resuming from any agent.
# On error, set RESUME_FROM and re-run this cell.
# ============================================================

from google.colab import files as colab_files
from collections import defaultdict

# ── CONFIGURATION ──────────────────────────────────────────
# For a full run: RESUME_FROM = None
# To resume after an error: RESUME_FROM = 'p2', 'p3', 'p4', 'p5', or 'p6'
# The system automatically loads previous outputs from Drive.

RESUME_FROM = None


# ── 1. Build state ────────────────────────────────────────

if RESUME_FROM:
    # ── RESUME MODE: load previous outputs from Drive ──
    initial_state = build_resume_state(
        resume_from=RESUME_FROM,
        # If you want the most recent version folder:
        # write run_timestamp= LAST_RUN_TIMESTAMP or RUN_TIMESTAMP
        run_timestamp= LAST_RUN_TIMESTAMP or RUN_TIMESTAMP,
    )
    print(f'Mode: RESUME from {RESUME_FROM.upper()}')

else:
    # ── FULL MODE: upload PDF and run P1→P6 ──
    print('Waiting for upload of the military-mission PDF...')
    print('(Upload the PDF of the military mission to be modelled)')
    print()

    uploaded = colab_files.upload()
    pdf_filename = list(uploaded.keys())[0]
    pdf_bytes = uploaded[pdf_filename]
    print(f'\nPDF received: {pdf_filename} ({len(pdf_bytes):,} bytes)')

    initial_state = {
        'pdf_bytes': pdf_bytes,
        'pdf_filename': pdf_filename,
        'mission_markdown': '',
        'layer_markdowns': {},
        'mission_json': {},
        'skeletons': {},
        'refined_packages': {},
        'validation_report': {},
        'fix_report': {},
        'iteration': 0,
        'max_iterations': MAX_ITERATIONS,
        'pipeline_status': 'RUNNING',
        'resume_from': None,
        'cost_log': [],
        'errors': [],
        'output_base_dir': str(BASE_DIR),
        'run_timestamp': RUN_TIMESTAMP,
        'pdf_pages': 0,
        'skip_monticore': False,
    }

print(f'\nInitial state created:')
print(f'  run_timestamp: {initial_state.get("run_timestamp")}')
print(f'  max_iterations: {MAX_ITERATIONS}')
print(f'  resume_from: {RESUME_FROM or "(full run)"}')


# ── 2. Run pipeline ───────────────────────────────────────

mode_label = f'RESUMING from {RESUME_FROM.upper()}' if RESUME_FROM else 'P1→P2→P3→P4→P5→[P6↔P5]→delivery'
print(f'\n{"="*70}')
print(f'  RUNNING PIPELINE — {mode_label}')
print(f'{"="*70}\n')

t_total = time.time()

final_state = pipeline.invoke(initial_state)

elapsed_total = time.time() - t_total


# ── 3. Final report ───────────────────────────────────────

pipeline_status = final_state.get('pipeline_status', '?')
is_error = pipeline_status.startswith('ERROR_AT_')

print(f'\n\n{"="*70}')
print(f'  PIPELINE FINAL REPORT')
print(f'{"="*70}')
print(f'  Status:           {pipeline_status}')
print(f'  P5↔P6 iterations: {final_state.get("iteration", 0)}')
print(f'  Total time:        {elapsed_total:.1f}s ({elapsed_total/60:.1f} min)')

if is_error:
    failed_agent = pipeline_status.replace('ERROR_AT_', '')
    print(f'\n  ❌ Pipeline stopped at {failed_agent.upper()}.')
    print(f'  To resume: set RESUME_FROM = \'{failed_agent}\' and re-run this cell.')

# Detailed cost
cost_log = final_state.get('cost_log', [])
print(f'\n  COST PER AGENT:')
print(f'  {"Agent":<35s} {"Cost":>10s} {"Time":>10s} {"Tokens In":>12s} {"Tokens Out":>12s}')
print(f'  {"-"*79}')
total_cost = 0
for entry in cost_log:
    agent = entry.get('agent', '?')
    cost = entry.get('cost_usd', 0)
    time_s = entry.get('time_s', 0)
    t_in = entry.get('tokens_in', 0)
    t_out = entry.get('tokens_out', 0)
    total_cost += cost
    print(f'  {agent:<35s} ${cost:>9.4f} {time_s:>9.1f}s {t_in:>11,} {t_out:>11,}')
print(f'  {"-"*79}')
print(f'  {"TOTAL":<35s} ${total_cost:>9.4f} {elapsed_total:>9.1f}s')

# SLOC metrics
report = final_state.get('validation_report', {})
metrics = report.get('metrics', {})
if metrics:
    sloc_m = metrics.get('sloc', {})
    if sloc_m:
        print(f'\n  SLOC METRICS:')
        print(f'    SLOC generated:  {sloc_m.get("total_sloc", 0):,}')
        print(f'    SLOC with issue: {sloc_m.get("sloc_with_issues", 0)}')
        print(f'    SLOC rate (P5):  {sloc_m.get("sloc_rate", 0):.1%}')
        print(f'    Clean packages:  {sloc_m.get("packages_clean", 0)}/{sloc_m.get("packages_total", 0)}')
    print(f'    Auto-fixable remaining: {report.get("auto_fixable_count", 0)}')
    print(f'    Human-review: {report.get("human_review_count", 0)}')

# Final packages
refined = final_state.get('refined_packages', {})
print(f'\n  FINAL PACKAGES: {len(refined)}')
total_chars = sum(len(v) for v in refined.values())
print(f'  Total chars: {total_chars:,}')

# Errors
errors = final_state.get('errors', [])
if errors:
    print(f'\n  ERRORS ({len(errors)}):')
    for err in errors:
        print(f'    ❌ {err}')

print(f'\n  Output dir: {ORCHESTRATOR_DIR}')
print(f'{"="*70}')


# ── 4. SLOC metrics ───────────────────────────────────────

report = final_state.get('validation_report', {})
metrics = report.get('metrics', {})
fix_rep = final_state.get('fix_report', {})
all_issues = report.get('issues', [])
auto_fixable = report.get('auto_fixable', [])
human_review = report.get('human_review', [])

sloc_m = metrics.get('sloc', {})
if sloc_m:
    print(f'\n\n{"="*70}')
    print(f'  4. SLOC METRICS')
    print(f'{"="*70}')
    print(f'  ┌──────────────────────────────────────────────────────────┐')
    print(f'  │ SLOC generated:   {sloc_m.get("total_sloc",0):>5d}  '
          f'(of {sloc_m.get("total_loc",0)} LOC — {sloc_m.get("total_blank",0)} blank, '
          f'{sloc_m.get("total_comment",0)} //)   │')
    print(f'  │ SLOC with issue:  {sloc_m.get("sloc_with_issues",0):>5d}  '
          f'(unique lines with P5 issue)             │')
    print(f'  │ Occurrences:      {sloc_m.get("total_occurrences",0):>5d}  '
          f'(total errors/warnings detected)         │')
    print(f'  │ SLOC rate (P5):   {sloc_m.get("sloc_rate",0):6.1%}  '
          f'({sloc_m.get("sloc_clean",0)}/{sloc_m.get("total_sloc",0)} SLOC without issues)    │')
    print(f'  │ Clean packages:   {sloc_m.get("packages_clean",0):>3d}/{sloc_m.get("packages_total",0):<3d}  '
          f'({sloc_m.get("packages_clean",0)/max(sloc_m.get("packages_total",1),1):.1%})'
          f'                           │')
    print(f'  └──────────────────────────────────────────────────────────┘')
    pdf_pages = final_state.get('pdf_pages', 0)
    if pdf_pages > 0:
        print(f'  Generation ratio: {sloc_m.get("total_sloc",0)/pdf_pages:.1f} SLOC/PDF page '
              f'({pdf_pages} pages)')


# ── 5. SLOC per Package ────────────────────────────────────

per_package = report.get('per_package', {})
if per_package:
    print(f'\n\n{"="*70}')
    print(f'  5. SLOC PER PACKAGE')
    print(f'{"="*70}')
    print(f'  {"Package":<40s} {"SLOC":>5s} {"//":>4s} {"Issues":>6s} '
          f'{"Rate":>7s}')
    print(f'  {"-"*65}')

    for pkg_name in sorted(per_package.keys()):
        pkg = per_package[pkg_name]
        sloc = pkg.get('sloc', 0)
        comm = pkg.get('comment', 0)
        issues = pkg.get('sloc_with_issues', 0)
        rate = pkg.get('sloc_rate', 1.0)

        print(f'  {pkg_name:<40s} {sloc:>5d} {comm:>4d} {issues:>6d} '
              f'{rate:>6.1%}')

    print(f'  {"-"*65}')
    t_sloc = sum(p.get('sloc', 0) for p in per_package.values())
    t_comm = sum(p.get('comment', 0) for p in per_package.values())
    t_issues = sum(p.get('sloc_with_issues', 0) for p in per_package.values())
    t_clean = sum(1 for p in per_package.values()
                  if p.get('sloc_with_issues', 0) == 0)
    print(f'  {"TOTAL":<40s} {t_sloc:>5d} {t_comm:>4d} {t_issues:>6d}')
    print(f'  {t_clean}/{len(per_package)} packages without issues')

# ── 5b. SLOC per CoSMA Layer ──────────────────────────────

layer_stats = report.get('layer_stats', {})
if layer_stats:
    print(f'\n\n{"="*70}')
    print(f'  5b. SLOC PER CoSMA LAYER')
    print(f'{"="*70}')
    print(f'  {"Layer":<18s} {"Pkgs":>5s} {"SLOC":>6s} {"Issues":>7s} '
          f'{"Clean":>8s} {"Rate":>7s}')
    print(f'  {"-"*55}')
    layer_order = ['Purpose', 'Operation', 'Function', 'Logical', 'Technical',
                   'Requirements', 'Analysis', 'Execution', 'Program', 'CoSMA', 'root', 'Other']
    for layer in layer_order:
        if layer not in layer_stats:
            continue
        ls = layer_stats[layer]
        rate = (ls['sloc'] - ls['issues']) / max(ls['sloc'], 1)
        print(f'  {layer:<18s} {ls["pkgs"]:>5d} {ls["sloc"]:>6d} {ls["issues"]:>7d} '
              f'{ls["clean"]:>3d}/{ls["pkgs"]:<3d}  {rate:>6.1%}')
    print(f'  {"-"*55}')


# ── 6. P6 corrections, 3-tier cascade ───────────────────

if fix_rep:
    print(f'\n\n{"="*70}')
    print(f'  6. P6 CORRECTIONS (3-tier cascade)')
    print(f'{"="*70}')

    surg = fix_rep.get('surgical', {})
    targ = fix_rep.get('targeted', {})
    full = fix_rep.get('llm_full', {})

    print(f'  Tier 1 (Surgical):   {surg.get("succeeded", 0)} fixed, '
          f'{surg.get("escalated", 0)} escalated | '
          f'${surg.get("cost_usd", 0):.2f}')
    print(f'  Tier 2 (Targeted):   {targ.get("succeeded", 0)} fixed, '
          f'{targ.get("escalated", 0)} escalated | '
          f'${targ.get("cost_usd", 0):.4f}')
    print(f'  Tier 3 (LLM Full):   {full.get("succeeded", 0)} fixed, '
          f'{full.get("rejected", 0)} rejected | '
          f'${full.get("cost_usd", 0):.4f}')

    total_received = fix_rep.get('total_issues_received', 0)
    total_fixed = fix_rep.get('total_issues_fixed', 0)
    fix_rate = fix_rep.get('fix_rate', 0)
    print(f'\n  Fix rate: {total_fixed}/{total_received} ({fix_rate:.1%})')
    print(f'  Packages modified: {", ".join(fix_rep.get("packages_modified", []))}')
    print(f'  P6 total cost: ${fix_rep.get("total_cost_usd", 0):.4f}')


# ── 7. Resolved issues — breakdown ────────────────────────

print(f'\n\n{"="*70}')
print(f'  7. RESOLVED ISSUES ({len(auto_fixable)})')
print(f'{"="*70}')

if not auto_fixable:
    print(f'  No issue required automatic correction.')
else:
    strategy_order = [AUTO_SURGICAL, AUTO_TARGETED, AUTO_LLM]
    strategy_labels = {
        AUTO_SURGICAL:  '🔧 AUTO_SURGICAL (deterministic regex, $0.00)',
        AUTO_TARGETED:  '🎯 AUTO_TARGETED (surgical LLM by snippet)',
        AUTO_LLM:       '🤖 AUTO_LLM (full rewrite)',
    }

    by_strategy = defaultdict(list)
    for iss in auto_fixable:
        by_strategy[iss.get('fix_strategy', 'UNKNOWN')].append(iss)

    for strategy in strategy_order:
        issues = by_strategy.get(strategy, [])
        if not issues:
            continue
        label = strategy_labels.get(strategy, strategy)
        print(f'\n  ── {label} ({len(issues)}) ──')

        by_pkg = defaultdict(list)
        for iss in issues:
            by_pkg[iss.get('package', 'Unknown')].append(iss)

        for pkg_name in sorted(by_pkg.keys()):
            pkg_issues = by_pkg[pkg_name]
            for iss in pkg_issues:
                code = iss.get('code', '?')
                msg = iss.get('message', '')
                curr = iss.get('current_value', '')
                exp = iss.get('expected_value', '')

                print(f'    [{code}] {pkg_name}: {msg[:80]}')
                if curr and exp:
                    print(f'         {curr} → {exp}')


# ── 8. Issues for Human Review  ───────────

print(f'\n\n{"="*70}')
print(f'  8. ISSUES FOR HUMAN REVIEW ({len(human_review)})')
print(f'{"="*70}')

if not human_review:
    print(f'  ✅ No issue requires human review.')
else:
    print(f'  📋 These issues require human validation')
    print(f'     (SysML v2 Pilot Implementation, OMG, Xtext-based)')
    print(f'     Open the .sysml packages and resolve them issue by issue.')

    # Group by code for an organised view
    by_code = defaultdict(list)
    for iss in human_review:
        by_code[iss.get('code', 'UNK')].append(iss)

    code_labels = {
        'S13': 'Satisfy Path Resolution',
        'C02': 'Port Def Consistency',
        'C03': 'Satisfy Target Exists',
        'C05': 'Basetype Consistency',
        'S08': 'ISQ Feature (no known alias)',
        'S09': 'Unit Symbol (unknown)',
        'MC00': 'MontiCore Unavailable',
        'MC02': 'MontiCore Symbol Resolution',
        'MC05': 'MontiCore Timeout',
        'MC06': 'MontiCore Exception',
    }

    for code in sorted(by_code.keys()):
        issues = by_code[code]
        label = code_labels.get(code, code)
        print(f'\n  ── {code}: {label} ({len(issues)}) ──')

        by_pkg = defaultdict(list)
        for iss in issues:
            by_pkg[iss.get('package', 'Unknown')].append(iss)

        for pkg_name in sorted(by_pkg.keys()):
            pkg_issues = by_pkg[pkg_name]
            print(f'\n    📦 {pkg_name}')
            for iss in pkg_issues:
                sev = iss.get('severity', '?')
                msg = iss.get('message', '')
                curr = iss.get('current_value', '')
                exp = iss.get('expected_value', '')
                fix = iss.get('suggested_fix', '')
                spec = iss.get('spec_reference', '')
                line = iss.get('line')

                sev_icon = {'ERROR': '❌', 'WARNING': '⚠️', 'INFO': 'ℹ️'}.get(sev, '?')
                line_str = f' (L{line})' if line else ''

                print(f'      {sev_icon}{line_str} {msg}')
                if curr:
                    print(f'         Current: {curr}')
                if exp:
                    print(f'         Expected: {exp}')
                if fix:
                    print(f'         ➜ Suggestion: {fix}')
                if spec:
                    print(f'         📖 Ref: {spec}')


# ── 9. Cost per Agent ─────────────────────────────────────

print(f'\n\n{"="*70}')
print(f'  9. COST PER AGENT')
print(f'{"="*70}')
print(f'  {"Agent":<35s} {"Cost":>10s} {"Time":>10s} {"Tokens In":>12s} {"Tokens Out":>12s}')
print(f'  {"-"*79}')
total_cost = 0
for entry in cost_log:
    agent = entry.get('agent', '?')
    cost = entry.get('cost_usd', 0)
    time_s = entry.get('time_s', 0)
    t_in = entry.get('tokens_in', 0)
    t_out = entry.get('tokens_out', 0)
    total_cost += cost
    print(f'  {agent:<35s} ${cost:>9.4f} {time_s:>9.1f}s {t_in:>11,} {t_out:>11,}')
print(f'  {"-"*79}')
print(f'  {"TOTAL":<35s} ${total_cost:>9.4f} {elapsed_total:>9.1f}s')

print(f'\n  Output dir: {ORCHESTRATOR_DIR}')
print(f'{"="*70}')


# ── 10. Save enriched JSON report ─────────────────────────

final_report = {
    'pipeline_status': pipeline_status,
    'iterations': final_state.get('iteration', 0),
    'total_time_s': round(elapsed_total, 1),
    'total_cost_usd': round(total_cost, 4),
    'packages_count': len(refined),
    'total_chars': total_chars,
    'resume_from': RESUME_FROM,
    'metrics': metrics,
    'per_package': per_package,
    'fix_report_summary': {
        'surgical': fix_rep.get('surgical', {}),
        'targeted': fix_rep.get('targeted', {}),
        'llm_full': fix_rep.get('llm_full', {}),
        'fix_rate': fix_rep.get('fix_rate', 0),
        'total_cost_usd': fix_rep.get('total_cost_usd', 0),
    },
    'auto_fixable': auto_fixable,
    'human_review': human_review,
    'cost_log': cost_log,
    'errors': errors,
    'all_issues': all_issues,
    'layer_stats': report.get('layer_stats', {}),
    'pdf_pages': final_state.get('pdf_pages', 0),
    'run_timestamp': initial_state.get('run_timestamp', RUN_TIMESTAMP),
}

report_path = ORCHESTRATOR_DIR / f'pipeline_report_{initial_state.get("run_timestamp", RUN_TIMESTAMP)}.json'
with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(final_report, f, indent=2, ensure_ascii=False)
print(f'\nJSON report saved: {report_path}')


# ── 11. Delivery Report Markdown ──────────────────────────

def _generate_delivery_report_md(final_state, final_report, elapsed_total):
    """Generates the Delivery Report Markdown (deterministic, $0.00)."""

    report = final_state.get('validation_report', {})
    fix_rep = final_state.get('fix_report', {})
    metrics = report.get('metrics', {})
    per_package = report.get('per_package', {})
    auto_fixable = report.get('auto_fixable', [])
    human_review = report.get('human_review', [])
    refined = final_state.get('refined_packages', {})
    cost_log = final_state.get('cost_log', [])
    total_cost = final_report.get('total_cost_usd', 0)
    pipeline_status = final_report.get('pipeline_status', '?')
    run_ts = final_report.get('run_timestamp', '?')

    lines = []
    _a = lines.append

    # ── 1. Header ──
    _a(f'# CoSMA Pipeline — Delivery Report')
    _a(f'')
    _a(f'**Run timestamp:** {run_ts}')
    _a(f'**Source document:** {final_state.get("pdf_filename", "?")}')
    _a(f'**Model:** {LLM_MODEL}')
    _a(f'**Date:** {datetime.now().strftime("%Y-%m-%d %H:%M")}')
    _a(f'')

    # ── 2. Executive Summary ──
    _a(f'## 1. Executive Summary')
    _a(f'')

    sloc_md = metrics.get('sloc', {})

    _a(f'| Metric | Value |')
    _a(f'|--------|-------|')
    _a(f'| Pipeline status | **{pipeline_status}** |')
    _a(f'| Packages delivered | {len(refined)} |')
    _a(f'| Total SysML v2 chars | {sum(len(v) for v in refined.values()):,} |')
    _a(f'| SLOC (effective notation) | {sloc_md.get("total_sloc", 0):,} |')
    _a(f'| Total LOC | {sloc_md.get("total_loc", 0):,} |')
    _a(f'| P5↔P6 iterations | {final_report.get("iterations", 0)} |')
    _a(f'| SLOC with issue (P5) | {sloc_md.get("sloc_with_issues", 0)} |')
    _a(f'| SLOC rate (P5) | {sloc_md.get("sloc_rate", 0):.1%} |')
    _a(f'| Clean packages | {sloc_md.get("packages_clean", 0)}/{sloc_md.get("packages_total", 0)} |')
    _a(f'| Auto-fixable remaining | {report.get("auto_fixable_count", 0)} |')
    _a(f'| Human-review for Eclipse | {report.get("human_review_count", 0)} |')
    _a(f'| Total cost | ${total_cost:.4f} |')
    _a(f'| Total time | {elapsed_total:.0f}s ({elapsed_total/60:.1f} min) |')
    pdf_pg = final_state.get('pdf_pages', 0)
    if pdf_pg > 0:
        _a(f'| SLOC/PDF page | {sloc_md.get("total_sloc",0)/pdf_pg:.1f} |')
    _a(f'')

    # ── 3. Per-Package Status ──
    _a(f'## 2. Delivered Packages')
    _a(f'')
    _a(f'| # | Package | SLOC | // | Issues | SLOC Rate | Status |')
    _a(f'|---|---------|------|----|--------|-----------|--------| ')

    for idx, pkg_name in enumerate(sorted(refined.keys()), 1):
        pkg = per_package.get(pkg_name, {})
        sloc_v = pkg.get('sloc', 0)
        comm_v = pkg.get('comment', 0)
        issues_v = pkg.get('sloc_with_issues', 0)
        rate_v = pkg.get('sloc_rate', 1.0)

        status = f'{issues_v} issues' if issues_v > 0 else 'PASS'
        _a(f'| {idx} | {pkg_name} | {sloc_v} | {comm_v} | {issues_v} | '
           f'{rate_v:.1%} | {status} |')
    _a(f'')

    # ── 4. P6 Corrections ──
    _a(f'## 3. Corrections Applied (P6 — 3-Level Cascade)')
    _a(f'')

    if fix_rep:
        surg = fix_rep.get('surgical', {})
        targ = fix_rep.get('targeted', {})
        full = fix_rep.get('llm_full', {})

        _a(f'| Level | Succeeded | Escalated/Rejected | Cost |')
        _a(f'|-------|-----------|-------------------|------|')
        _a(f'| Tier 1 — Surgical (regex) | {surg.get("succeeded", 0)} | '
           f'{surg.get("escalated", 0)} escalated | ${surg.get("cost_usd", 0):.2f} |')
        _a(f'| Tier 2 — Targeted (LLM snippet) | {targ.get("succeeded", 0)} | '
           f'{targ.get("escalated", 0)} escalated | ${targ.get("cost_usd", 0):.4f} |')
        _a(f'| Tier 3 — LLM Full (rewrite) | {full.get("succeeded", 0)} | '
           f'{full.get("rejected", 0)} rejected | ${full.get("cost_usd", 0):.4f} |')
        _a(f'| **Total** | **{fix_rep.get("total_issues_fixed", 0)}** / '
           f'{fix_rep.get("total_issues_received", 0)} | '
           f'Fix rate: {fix_rep.get("fix_rate", 0):.1%} | '
           f'**${fix_rep.get("total_cost_usd", 0):.4f}** |')
        _a(f'')
    else:
        _a(f'No P6 corrections were applied.')
        _a(f'')

    # ── 5. Resolved Issues ──
    _a(f'## 4. Resolved Issues — Auto-Corrected ({len(auto_fixable)})')
    _a(f'')

    if not auto_fixable:
        _a(f'No issues required auto-correction.')
    else:
        strategy_order = [AUTO_SURGICAL, AUTO_TARGETED, AUTO_LLM]
        strategy_md_labels = {
            AUTO_SURGICAL: 'Surgical (deterministic regex)',
            AUTO_TARGETED: 'Targeted (surgical LLM)',
            AUTO_LLM:      'LLM Full (rewrite)',
        }

        for strategy in strategy_order:
            s_issues = [i for i in auto_fixable if i.get('fix_strategy') == strategy]
            if not s_issues:
                continue
            label = strategy_md_labels.get(strategy, strategy)
            _a(f'### 4.{strategy_order.index(strategy)+1} {label} ({len(s_issues)})')
            _a(f'')
            _a(f'| Package | Code | Description |')
            _a(f'|---------|------|-------------|')
            for iss in s_issues:
                pkg = iss.get('package', '?')
                code = iss.get('code', '?')
                msg = iss.get('message', '').replace('|', '\\|')[:100]
                _a(f'| {pkg} | {code} | {msg} |')
            _a(f'')
    _a(f'')

    # ── 6. Human Review ──
    _a(f'## 5. Issues for Human Review — Eclipse IDE ({len(human_review)})')
    _a(f'')

    if not human_review:
        _a(f'All issues were resolved automatically. '
           f'No human intervention required.')
    else:
        _a(f'These issues require validation in **Eclipse IDE** '
           f'(SysML v2 Pilot Implementation, OMG, Xtext-based).')
        _a(f'Open the `.sysml` packages in Eclipse and resolve each issue.')
        _a(f'')

        by_code = defaultdict(list)
        for iss in human_review:
            by_code[iss.get('code', 'UNK')].append(iss)

        code_md_labels = {
            'S13': 'Satisfy Path Resolution',
            'C02': 'Port Def Consistency',
            'C03': 'Satisfy Target Exists',
            'C05': 'Basetype Consistency',
            'S08': 'ISQ Feature Name (unknown)',
            'S09': 'Unit Symbol (unknown)',
            'MC00': 'MontiCore Unavailable',
            'MC02': 'MontiCore Symbol Resolution',
        }

        for code in sorted(by_code.keys()):
            issues = by_code[code]
            label = code_md_labels.get(code, code)
            _a(f'### 5.{list(sorted(by_code.keys())).index(code)+1} {code}: {label} ({len(issues)})')
            _a(f'')

            by_pkg = defaultdict(list)
            for iss in issues:
                by_pkg[iss.get('package', 'Unknown')].append(iss)

            for pkg_name in sorted(by_pkg.keys()):
                pkg_issues = by_pkg[pkg_name]
                _a(f'**{pkg_name}:**')
                _a(f'')
                for iss in pkg_issues:
                    sev = iss.get('severity', '?')
                    msg = iss.get('message', '')
                    curr = iss.get('current_value', '')
                    exp = iss.get('expected_value', '')
                    fix = iss.get('suggested_fix', '')
                    spec = iss.get('spec_reference', '')
                    line = iss.get('line')

                    sev_icon = {'ERROR': '❌', 'WARNING': '⚠️',
                                'INFO': 'ℹ️'}.get(sev, '?')
                    line_ref = f', line {line}' if line else ''

                    _a(f'- {sev_icon} **{msg}**{line_ref}')
                    if curr:
                        _a(f'  - Current: `{curr}`')
                    if exp:
                        _a(f'  - Expected: `{exp}`')
                    if fix:
                        _a(f'  - Suggested fix: {fix}')
                    if spec:
                        _a(f'  - Spec reference: {spec}')
                    _a(f'')
            _a(f'')

    # ── 7. Cost Breakdown ──
    _a(f'## 6. Cost Breakdown')
    _a(f'')
    _a(f'| Agent | Cost (USD) | Time (s) | Tokens In | Tokens Out |')
    _a(f'|-------|-----------|----------|-----------|------------|')
    for entry in cost_log:
        agent = entry.get('agent', '?')
        cost = entry.get('cost_usd', 0)
        t_s = entry.get('time_s', 0)
        t_in = entry.get('tokens_in', 0)
        t_out = entry.get('tokens_out', 0)
        _a(f'| {agent} | ${cost:.4f} | {t_s:.1f} | {t_in:,} | {t_out:,} |')
    _a(f'| **TOTAL** | **${total_cost:.4f}** | **{elapsed_total:.1f}** | | |')
    _a(f'')

    # ── 8. Summary Statistics ──
    _a(f'## 7. Validation Summary (SLOC)')
    _a(f'')
    _a(f'| Metric | Value |')
    _a(f'|--------|-------|')
    _a(f'| SLOC generated | {sloc_md.get("total_sloc", 0):,} |')
    _a(f'| SLOC with issue (P5) | {sloc_md.get("sloc_with_issues", 0)} |')
    _a(f'| Total occurrences | {sloc_md.get("total_occurrences", 0)} |')
    _a(f'| SLOC rate (P5) | {sloc_md.get("sloc_rate", 0):.1%} |')
    _a(f'| Packages limpos | {sloc_md.get("packages_clean",0)}/{sloc_md.get("packages_total",0)} |')
    _a(f'| Auto-corrected issues | {len(auto_fixable)} |')
    _a(f'| Human review required | {len(human_review)} |')
    _a(f'| Fix rate (P6) | {fix_rep.get("fix_rate", "N/A")} |')
    _a(f'')

    # ── Footer ──
    _a(f'---')
    _a(f'')
    _a(f'*Generated by CoSMA Pipeline v5.1 — '
       f'{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}*')
    _a(f'')

    return '\n'.join(lines)


# Generate and save
if not pipeline_status.startswith('ERROR_AT_'):
    delivery_md = _generate_delivery_report_md(
        final_state, final_report, elapsed_total)

    ts_label = initial_state.get('run_timestamp', RUN_TIMESTAMP)
    delivery_md_path = ORCHESTRATOR_DIR / f'delivery_report_{ts_label}.md'
    delivery_md_path.write_text(delivery_md, encoding='utf-8')
    print(f'Delivery report saved: {delivery_md_path}')

    final_dir = ORCHESTRATOR_DIR / f'final_{ts_label}'
    if final_dir.exists():
        (final_dir / 'delivery_report.md').write_text(
            delivery_md, encoding='utf-8')
        print(f'Delivery report copied to: {final_dir / "delivery_report.md"}')
else:
    print(f'\n⚠️ Delivery report not generated (pipeline stopped with error).')
    print(f'Resume the pipeline and re-run to generate the report.')

print(f'\n✅ Multi-agent system pipeline — COMPLETED')